# Channel operator V2 — portable trained notebook

This is a self-contained export for importing into the normal Kaggle notebook editor. It embeds the actual 70,000-step model, full optimizer/RNG checkpoint, source and synthetic CFD data. It has no private Kaggle input dependencies and no API credentials.

Default cells perform inference and reproduce validation; they do not train. The strict momentum-residual gate is still unmet. Trained scope: four smooth inlet cosine modes, L1<=0.8, Re 10–100, channel 5x1.


In [ ]:
import base64, os, sys, subprocess
from pathlib import Path
os.environ["MPLCONFIGDIR"]="/tmp/channel-mpl"
assets={'channel_operator/__init__.py': 'IiIiQ29uZGl0aW9uYWwgcGh5c2ljcy1pbmZvcm1lZCBvcGVyYXRvciBmb3IgYSBzdGVhZHkgMkQgY2hhbm5lbC4iIiIK', 'channel_operator/data.py': 'IiIiR2VuZXJhdGUgaW5kZXBlbmRlbnQgY29udmVyZ2VkIEZFTSBsYWJlbHMgYW5kIGEgZml4ZWQgaGVsZC1vdXQgc3BsaXQuIiIiCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSAucHJvZmlsZXMgaW1wb3J0IHNhbXBsZV9wcm9maWxlcywgZnJvbV9jb2VmZmljaWVudHMKZnJvbSAucmVmZXJlbmNlIGltcG9ydCBzb2x2ZV9jaGFubmVsCgoKZGVmIGdlbmVyYXRlKG91dHB1dCwgdHJhaW5fY2FzZXM9NDgsIHRlc3RfY2FzZXM9OCwgc2VlZD0xNzMxKToKICAgIG91dHB1dCA9IFBhdGgob3V0cHV0KQogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKQogICAgdHJhaW5fcHJvZmlsZXMsIHRyYWluX2NvZWZmID0gc2FtcGxlX3Byb2ZpbGVzKHJuZywgdHJhaW5fY2FzZXMpCiAgICB0cmFpbl9yZSA9IG5wLmV4cChybmcudW5pZm9ybShucC5sb2coMTApLCBucC5sb2coMTAwKSwgdHJhaW5fY2FzZXMpKQogICAgdHJhaW5fcmVbOjJdID0gWzEwLCAxMDBdCiAgICAjIERpc2pvaW50IGRldGVybWluaXN0aWMgdGVzdCBjb25kaXRpb25zLCBuZXZlciB1c2VkIGZvciBtb2RlbCBzZWxlY3Rpb24uCiAgICB0ZXN0X3Byb2ZpbGVzLCB0ZXN0X2NvZWZmID0gc2FtcGxlX3Byb2ZpbGVzKG5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKzk5OSksIHRlc3RfY2FzZXMpCiAgICB0ZXN0X3JlID0gbnAubGluc3BhY2UoMTIsIDk4LCB0ZXN0X2Nhc2VzKQogICAgeCwgeSA9IG5wLm1lc2hncmlkKG5wLmxpbnNwYWNlKC4wMTUsIDQuOTg1LCA0OSksIG5wLmxpbnNwYWNlKC4wMSwgLjk5LCAyMSksIGluZGV4aW5nPSJpaiIpCiAgICB4eSA9IG5wLmNvbHVtbl9zdGFjayhbeC5yYXZlbCgpLCB5LnJhdmVsKCldKQogICAgbWV0YWRhdGEgPSB7InNlZWQiOiBzZWVkLCAidHJhaW5fY2FzZXMiOiB0cmFpbl9jYXNlcywgInRlc3RfY2FzZXMiOiB0ZXN0X2Nhc2VzLCAidGVzdF91c2VkX2Zvcl90cmFpbmluZyI6IEZhbHNlLCAiY2FzZXMiOiBbXX0KICAgIGZvciBuYW1lLCBwcm9maWxlcywgY29lZmYsIHJlcyBpbiBbKCJ0cmFpbiIsIHRyYWluX3Byb2ZpbGVzLCB0cmFpbl9jb2VmZiwgdHJhaW5fcmUpLCAoInRlc3QiLCB0ZXN0X3Byb2ZpbGVzLCB0ZXN0X2NvZWZmLCB0ZXN0X3JlKV06CiAgICAgICAgZmllbGRzLCBtZXNoX2RpZmZlcmVuY2VzID0gW10sIFtdCiAgICAgICAgZm9yIGksIChwcm9maWxlLCByZSkgaW4gZW51bWVyYXRlKHppcChwcm9maWxlcywgcmVzKSk6CiAgICAgICAgICAgIGV2YWx1YXRlLCBpbmZvID0gc29sdmVfY2hhbm5lbChwcm9maWxlLCByZSwgbng9NDAgaWYgbmFtZT09InRyYWluIiBlbHNlIDY0LCBueT0xNiBpZiBuYW1lPT0idHJhaW4iIGVsc2UgMjQpCiAgICAgICAgICAgIHZhbHVlcyA9IGV2YWx1YXRlKHh5KQogICAgICAgICAgICBmaWVsZHMuYXBwZW5kKHZhbHVlcykKICAgICAgICAgICAgaWYgbmFtZSA9PSAidGVzdCI6CiAgICAgICAgICAgICAgICBjb2Fyc2UsIF8gPSBzb2x2ZV9jaGFubmVsKHByb2ZpbGUsIHJlLCBueD00MCwgbnk9MTYpCiAgICAgICAgICAgICAgICBvdGhlciA9IGNvYXJzZSh4eSkKICAgICAgICAgICAgICAgIG1lc2hfZGlmZmVyZW5jZXMuYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAidmVsb2NpdHlfcmVsYXRpdmVfbDIiOiBmbG9hdChucC5saW5hbGcubm9ybShvdGhlcls6LCA6Ml0tdmFsdWVzWzosIDoyXSkvbnAubGluYWxnLm5vcm0odmFsdWVzWzosIDoyXSkpLAogICAgICAgICAgICAgICAgICAgICJwcmVzc3VyZV9yZWxhdGl2ZV9sMiI6IGZsb2F0KG5wLmxpbmFsZy5ub3JtKG90aGVyWzosIDJdLXZhbHVlc1s6LCAyXSkvbnAubGluYWxnLm5vcm0odmFsdWVzWzosIDJdKSl9KQogICAgICAgICAgICBtZXRhZGF0YVsiY2FzZXMiXS5hcHBlbmQoZGljdChzcGxpdD1uYW1lLCBpbmRleD1pLCByZXlub2xkcz1mbG9hdChyZSksICoqaW5mbykpCiAgICAgICAgICAgIHByaW50KGYiRkVNIHtuYW1lfSB7aSsxfS97bGVuKHJlcyl9IFJlPXtyZTouMmZ9IGl0ZXJhdGlvbnM9e2luZm9bJ2l0ZXJhdGlvbnMnXX0iLCBmbHVzaD1UcnVlKQogICAgICAgIG5wLnNhdmV6X2NvbXByZXNzZWQob3V0cHV0L2Yie25hbWV9Lm5weiIsIHByb2ZpbGVzPXByb2ZpbGVzLCBjb2VmZmljaWVudHM9Y29lZmYsIHJleW5vbGRzPXJlcywgeHk9eHksIGZpZWxkcz1ucC5hcnJheShmaWVsZHMpKQogICAgICAgIGlmIG5hbWUgPT0gInRlc3QiOgogICAgICAgICAgICBtZXRhZGF0YVsidGVzdF9tZXNoX2NvbXBhcmlzb24iXSA9IG1lc2hfZGlmZmVyZW5jZXMKICAgICMgRXhhY3Qgc29sdXRpb24gdmVyaWZpZXMgRkVNIGZvcm11bGF0aW9uIGFuZCBwcmVzc3VyZSByZWZlcmVuY2UgaW5kZXBlbmRlbnRseS4KICAgIHJlID0gNDAuCiAgICBleGFjdCA9IG5wLmNvbHVtbl9zdGFjayhbNip4eVs6LCAxXSooMS14eVs6LCAxXSksIG5wLnplcm9zKGxlbih4eSkpLCAxMi9yZSooNS14eVs6LCAwXSldKQogICAgZXZhbHVhdGUsIGluZm8gPSBzb2x2ZV9jaGFubmVsKGZyb21fY29lZmZpY2llbnRzKG5wLnplcm9zKDQpKSwgcmUsIG54PTIwLCBueT04KQogICAgbWV0YWRhdGFbInBvaXNldWlsbGVfbWF4X2Fic19lcnJvciJdID0gZmxvYXQobnAubWF4KG5wLmFicyhldmFsdWF0ZSh4eSktZXhhY3QpKSkKICAgIChvdXRwdXQvInJlZmVyZW5jZV9tZXRhZGF0YS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKG1ldGFkYXRhLCBpbmRlbnQ9MikpCiAgICByZXR1cm4gbWV0YWRhdGEK', 'channel_operator/evaluate.py': 'IiIiSGVsZC1vdXQgRkVNIGNvbXBhcmlzb24gcGx1cyBmcmVzaCBQREUsIGJvdW5kYXJ5IGFuZCBmbHV4IGRpYWdub3N0aWNzLiIiIgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgbWF0cGxvdGxpYgptYXRwbG90bGliLnVzZSgiQWdnIikKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdApmcm9tIC5tb2RlbCBpbXBvcnQgbG9hZF9tb2RlbCwgcmVzaWR1YWxzLCBvdXRsZXRfcmVzaWR1YWxzLCBDaGFubmVsRGVlcE9OZXQKZnJvbSAucHJvZmlsZXMgaW1wb3J0IGZyb21fY29lZmZpY2llbnRzLCBZCgoKZGVmIGV2YWx1YXRlKG91dHB1dCk6CiAgICBvdXRwdXQgPSBQYXRoKG91dHB1dCkKICAgIGRldmljZSA9ICJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIKICAgIG1vZGVsLCBja3B0ID0gbG9hZF9tb2RlbChvdXRwdXQvIm1vZGVsLnB0IiwgZGV2aWNlKQogICAgdGVuc29yID0gbGFtYmRhIGE6IHRvcmNoLmFzX3RlbnNvcihhLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9ZGV2aWNlKQogICAgd2l0aCBucC5sb2FkKG91dHB1dC8idGVzdC5ucHoiKSBhcyBkOgogICAgICAgIHByb2ZpbGVzLCByZXMsIHh5LCB0cnV0aCA9IFtkW2tdIGZvciBrIGluIFsicHJvZmlsZXMiLCAicmV5bm9sZHMiLCAieHkiLCAiZmllbGRzIl1dCiAgICBwLCByID0gdGVuc29yKHByb2ZpbGVzKSwgdGVuc29yKHJlcykKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIHByZWRpY3Rpb24gPSBtb2RlbChwLCByLCB0ZW5zb3IoeHkpW05vbmVdLmV4cGFuZChsZW4ocCksIC0xLCAtMSkpLmNwdSgpLm51bXB5KCkKICAgICAgICBiYXNlbGluZSA9IENoYW5uZWxEZWVwT05ldCgqKmNrcHRbIm1vZGVsX2NvbmZpZyJdKS50byhkZXZpY2UpCiAgICAgICAgYmFzZWxpbmUudHJ1bmtbLTFdLndlaWdodC56ZXJvXygpCiAgICAgICAgYmFzZWxpbmUudHJ1bmtbLTFdLmJpYXMuemVyb18oKQogICAgICAgIGluaXRpYWwgPSBiYXNlbGluZShwLCByLCB0ZW5zb3IoeHkpW05vbmVdLmV4cGFuZChsZW4ocCksIC0xLCAtMSkpLmNwdSgpLm51bXB5KCkKICAgIGNhc2VzID0gW10KICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyg4NDg1MykKICAgIGZvciBpIGluIHJhbmdlKGxlbihwKSk6CiAgICAgICAgcHAsIHJyID0gcFtpOmkrMV0sIHJbaTppKzFdCiAgICAgICAgY29vcmRzID0gcm5nLnVuaWZvcm0oWy4wMDUsIC4wMDVdLCBbNC45OTUsIC45OTVdLCAoMSwgMjA0OCwgMikpCiAgICAgICAgbSwgbXgsIG15ID0gcmVzaWR1YWxzKG1vZGVsLCBwcCwgcnIsIHRlbnNvcihjb29yZHMpKQogICAgICAgIGd4LCBneSA9IG5wLm1lc2hncmlkKG5wLmxpbnNwYWNlKDAsIDUsIDMxKSwgbnAubGluc3BhY2UoMCwgMSwgMjU3KSwgaW5kZXhpbmc9ImlqIikKICAgICAgICBncmlkID0gbnAuc3RhY2soW2d4LCBneV0sIC0xKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICB1dnAgPSBtb2RlbChwcCwgcnIsIHRlbnNvcihncmlkLnJlc2hhcGUoMSwgLTEsIDIpKSkuY3B1KCkubnVtcHkoKS5yZXNoYXBlKDMxLCAyNTcsIDMpCiAgICAgICAgZmx1eCA9IG5wLnRyYXBlem9pZCh1dnBbLi4uLCAwXSwgZ3ksIGF4aXM9LTEpCiAgICAgICAgb3V0bGV0X3h5ID0gbnAuc3RhY2soW25wLmZ1bGwoMjU3LCA1LiksIG5wLmxpbnNwYWNlKDAsIDEsIDI1NyldLCAtMSkKICAgICAgICBvbiwgb3QgPSBvdXRsZXRfcmVzaWR1YWxzKG1vZGVsLCBwcCwgcnIsIHRlbnNvcihvdXRsZXRfeHlbTm9uZV0pKQogICAgICAgIGVycm9yID0gcHJlZGljdGlvbltpXS10cnV0aFtpXQogICAgICAgIHJlY29yZCA9IHsiY2FzZSI6IGksICJyZXlub2xkcyI6IGZsb2F0KHJlc1tpXSksCiAgICAgICAgICAgICAgICAgICJ2ZWxvY2l0eV9yZWxhdGl2ZV9sMiI6IGZsb2F0KG5wLmxpbmFsZy5ub3JtKGVycm9yWzosIDoyXSkvbnAubGluYWxnLm5vcm0odHJ1dGhbaSwgOiwgOjJdKSksCiAgICAgICAgICAgICAgICAgICJwcmVzc3VyZV9yZWxhdGl2ZV9sMiI6IGZsb2F0KG5wLmxpbmFsZy5ub3JtKGVycm9yWzosIDJdKS9ucC5saW5hbGcubm9ybSh0cnV0aFtpLCA6LCAyXSkpLAogICAgICAgICAgICAgICAgICAidW50cmFpbmVkX2Jhc2VsaW5lX3ZlbG9jaXR5X3JlbGF0aXZlX2wyIjogZmxvYXQobnAubGluYWxnLm5vcm0oaW5pdGlhbFtpLCA6LCA6Ml0tdHJ1dGhbaSwgOiwgOjJdKS9ucC5saW5hbGcubm9ybSh0cnV0aFtpLCA6LCA6Ml0pKSwKICAgICAgICAgICAgICAgICAgInVudHJhaW5lZF9iYXNlbGluZV9wcmVzc3VyZV9yZWxhdGl2ZV9sMiI6IGZsb2F0KG5wLmxpbmFsZy5ub3JtKGluaXRpYWxbaSwgOiwgMl0tdHJ1dGhbaSwgOiwgMl0pL25wLmxpbmFsZy5ub3JtKHRydXRoW2ksIDosIDJdKSksCiAgICAgICAgICAgICAgICAgICJ2X2Fic29sdXRlX3Jtc2UiOiBmbG9hdChucC5zcXJ0KG5wLm1lYW4oZXJyb3JbOiwgMV0qKjIpKSksCiAgICAgICAgICAgICAgICAgICJkaXZlcmdlbmNlX3JtcyI6IGZsb2F0KG0uc3F1YXJlKCkubWVhbigpLnNxcnQoKS5kZXRhY2goKSksCiAgICAgICAgICAgICAgICAgICJtb21lbnR1bV9ybXMiOiBmbG9hdCgobXguc3F1YXJlKCkubWVhbigpK215LnNxdWFyZSgpLm1lYW4oKSkuc3FydCgpLmRldGFjaCgpKSwKICAgICAgICAgICAgICAgICAgIm1heF9mbHV4X2Vycm9yIjogZmxvYXQobnAubWF4KG5wLmFicyhmbHV4LTEpKSksCiAgICAgICAgICAgICAgICAgICJzY2FsZWRfb3V0bGV0X3JtcyI6IGZsb2F0KChvbi5zcXVhcmUoKS5tZWFuKCkrb3Quc3F1YXJlKCkubWVhbigpKS5zcXJ0KCkuZGV0YWNoKCkpfQogICAgICAgIGNhc2VzLmFwcGVuZChyZWNvcmQpCiAgICBtZXRhID0ganNvbi5sb2Fkcygob3V0cHV0LyJyZWZlcmVuY2VfbWV0YWRhdGEuanNvbiIpLnJlYWRfdGV4dCgpKQogICAgIyBEZWxpYmVyYXRlbHkgZXhwbGljaXQgYWNjZXB0YW5jZSBnYXRlczsgYSBjb21wbGV0ZWQgdHJhaW5pbmcgaXMgbm90IGEgcGFzcy4KICAgIGNyaXRlcmlhID0geyJ2ZWxvY2l0eV9yZWxhdGl2ZV9sMl9tYXgiOiAuMDUsICJwcmVzc3VyZV9yZWxhdGl2ZV9sMl9tYXgiOiAuMTAsCiAgICAgICAgICAgICAgICAibWF4X2ZsdXhfZXJyb3IiOiAuMDIsICJkaXZlcmdlbmNlX3Jtc19tYXgiOiAuMDMsCiAgICAgICAgICAgICAgICAibW9tZW50dW1fcm1zX21heCI6IC4wNSwgInNjYWxlZF9vdXRsZXRfcm1zX21heCI6IC4xMCwKICAgICAgICAgICAgICAgICJyZWZlcmVuY2VfbWVzaF92ZWxvY2l0eV9tYXgiOiAuMDEsICJyZWZlcmVuY2VfbWVzaF9wcmVzc3VyZV9tYXgiOiAuMDJ9CiAgICBnYXRlcyA9IHsidmVsb2NpdHkiOiBtYXgoY1sidmVsb2NpdHlfcmVsYXRpdmVfbDIiXSBmb3IgYyBpbiBjYXNlcyk8LjA1LAogICAgICAgICAgICAgInByZXNzdXJlIjogbWF4KGNbInByZXNzdXJlX3JlbGF0aXZlX2wyIl0gZm9yIGMgaW4gY2FzZXMpPC4xMCwKICAgICAgICAgICAgICJmbHV4IjogbWF4KGNbIm1heF9mbHV4X2Vycm9yIl0gZm9yIGMgaW4gY2FzZXMpPC4wMiwKICAgICAgICAgICAgICJkaXZlcmdlbmNlIjogbWF4KGNbImRpdmVyZ2VuY2Vfcm1zIl0gZm9yIGMgaW4gY2FzZXMpPC4wMywKICAgICAgICAgICAgICJtb21lbnR1bSI6IG1heChjWyJtb21lbnR1bV9ybXMiXSBmb3IgYyBpbiBjYXNlcyk8LjA1LAogICAgICAgICAgICAgIm91dGxldCI6IG1heChjWyJzY2FsZWRfb3V0bGV0X3JtcyJdIGZvciBjIGluIGNhc2VzKTwuMTAsCiAgICAgICAgICAgICAicmVmZXJlbmNlX21lc2giOiBhbGwoY1sidmVsb2NpdHlfcmVsYXRpdmVfbDIiXTwuMDEgYW5kIGNbInByZXNzdXJlX3JlbGF0aXZlX2wyIl08LjAyIGZvciBjIGluIG1ldGFbInRlc3RfbWVzaF9jb21wYXJpc29uIl0pLAogICAgICAgICAgICAgInJlZmVyZW5jZV9wb2lzZXVpbGxlIjogbWV0YVsicG9pc2V1aWxsZV9tYXhfYWJzX2Vycm9yIl08MWUtN30KICAgIHJlcG9ydCA9IHsiY2hlY2twb2ludF9zdGVwIjogY2twdFsic3RlcCJdLCAidHJhaW5pbmdfbW9kZSI6IGNrcHRbIm1vZGUiXSwgInRlc3RfY2FzZXMiOiBjYXNlcywKICAgICAgICAgICAgICAiYWNjZXB0YW5jZV9jcml0ZXJpYSI6IGNyaXRlcmlhLCAiZ2F0ZXMiOiBnYXRlcywgInBhc3NlZCI6IGFsbChnYXRlcy52YWx1ZXMoKSksCiAgICAgICAgICAgICAgInJlZmVyZW5jZV9tZXNoX2NvbXBhcmlzb24iOiBtZXRhWyJ0ZXN0X21lc2hfY29tcGFyaXNvbiJdLAogICAgICAgICAgICAgICJzY29wZSI6ICJIZWxkLW91dCBzbW9vdGggcG9zaXRpdmUgbm8tc2xpcCBpbmxldHMgd2l0aGluIHRoZSBmb3VyLW1vZGUgTDE8PTAuNiBmYW1pbHksIFJlIGluIFsxMCwxMDBdOyBubyBleHRyYXBvbGF0aW9uIGd1YXJhbnRlZS4ifQogICAgKG91dHB1dC8ibWV0cmljcy5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCwgaW5kZW50PTIpKQogICAgbnAuc2F2ZXpfY29tcHJlc3NlZChvdXRwdXQvImhlbGRvdXRfcHJlZGljdGlvbnMubnB6IiwgeHk9eHksIHByb2ZpbGVzPXByb2ZpbGVzLCByZXlub2xkcz1yZXMsIHJlZmVyZW5jZT10cnV0aCwgcHJlZGljdGlvbj1wcmVkaWN0aW9uKQogICAgaSA9IGludChucC5hcmdtYXgoW2NbInZlbG9jaXR5X3JlbGF0aXZlX2wyIl0gZm9yIGMgaW4gY2FzZXNdKSkKICAgIGZpZywgYXhlcyA9IHBsdC5zdWJwbG90cygzLCAzLCBmaWdzaXplPSgxMywgOCksIGNvbnN0cmFpbmVkX2xheW91dD1UcnVlKQogICAgZm9yIGosIG5hbWUgaW4gZW51bWVyYXRlKFsidSIsICJ2IiwgInAiXSk6CiAgICAgICAgZm9yIGssICh0aXRsZSwgZmllbGQpIGluIGVudW1lcmF0ZShbKCJGRU0iLCB0cnV0aCksICgib3BlcmF0b3IiLCBwcmVkaWN0aW9uKSwgKCJlcnJvciIsIHByZWRpY3Rpb24tdHJ1dGgpXSk6CiAgICAgICAgICAgIGF4ID0gYXhlc1tqLCBrXQogICAgICAgICAgICB6ID0gZmllbGRbaSwgOiwgal0ucmVzaGFwZSg0OSwgMjEpLlQKICAgICAgICAgICAgbGltaXRzID0ge30KICAgICAgICAgICAgaWYgayA8IDI6CiAgICAgICAgICAgICAgICBsaW1pdHMgPSBkaWN0KHZtaW49bWluKHRydXRoW2ksIDosIGpdLm1pbigpLCBwcmVkaWN0aW9uW2ksIDosIGpdLm1pbigpKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdm1heD1tYXgodHJ1dGhbaSwgOiwgal0ubWF4KCksIHByZWRpY3Rpb25baSwgOiwgal0ubWF4KCkpKQogICAgICAgICAgICBpbSA9IGF4Lmltc2hvdyh6LCBvcmlnaW49Imxvd2VyIiwgZXh0ZW50PVsuMDE1LCA0Ljk4NSwgLjAxLCAuOTldLCBhc3BlY3Q9ImF1dG8iLCBjbWFwPSJjb29sd2FybSIsICoqbGltaXRzKQogICAgICAgICAgICBheC5zZXRfdGl0bGUoZiJ7bmFtZX06IHt0aXRsZX0iKQogICAgICAgICAgICBmaWcuY29sb3JiYXIoaW0sIGF4PWF4KQogICAgZmlnLnN1cHRpdGxlKGYiV29yc3QgaGVsZC1vdXQgdmVsb2NpdHkgY2FzZTogUmU9e3Jlc1tpXTouMWZ9IikKICAgIGZpZy5zYXZlZmlnKG91dHB1dC8iaGVsZG91dF9maWVsZHMucG5nIiwgZHBpPTE2MCkKICAgIHBsdC5jbG9zZShmaWcpCiAgICBwcmludChqc29uLmR1bXBzKHJlcG9ydCwgaW5kZW50PTIpLCBmbHVzaD1UcnVlKQogICAgcmV0dXJuIHJlcG9ydAo=', 'channel_operator/model.py': 'aW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCmZyb20gLnByb2ZpbGVzIGltcG9ydCBTRU5TT1JTLCBZLCBTUExJTkVfTUFQCgoKZGVmIG1scChkaW0sIHdpZHRoLCBvdXRwdXQsIGxheWVycz0zKToKICAgIGJsb2NrcyA9IFtubi5MaW5lYXIoZGltLCB3aWR0aCksIG5uLlRhbmgoKV0KICAgIGZvciBfIGluIHJhbmdlKGxheWVycy0xKToKICAgICAgICBibG9ja3MgKz0gW25uLkxpbmVhcih3aWR0aCwgd2lkdGgpLCBubi5UYW5oKCldCiAgICBibG9ja3MgKz0gW25uLkxpbmVhcih3aWR0aCwgb3V0cHV0KV0KICAgIHJldHVybiBubi5TZXF1ZW50aWFsKCpibG9ja3MpCgoKY2xhc3MgQ2hhbm5lbERlZXBPTmV0KG5uLk1vZHVsZSk6CiAgICAiIiJCcmFuY2g6IGlubGV0IHNhbXBsZXMgKyBsb2coUmUpLiBUcnVuazogbm9uZGltZW5zaW9uYWwgeCx5LgoKICAgIEhhcmQgaW5sZXQvbm8tc2xpcCBjb25zdHJhaW50czsgUG9pc2V1aWxsZSBpcyBhIGJhc2VsaW5lLCBub3QgdHJhaW5pbmcgZGF0YS4KICAgIElucHV0IHNoYXBlczogcHJvZmlsZXMgW0IsMzNdLCBSZSBbQl0sIGNvb3JkaW5hdGVzIFtCLE4sMl0uCiAgICAiIiIKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBsZW5ndGg9NS4sIHdpZHRoPTY0LCBsYXRlbnQ9MzIsIGVudHJhbmNlX2ZlYXR1cmVzPUZhbHNlKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmNvbmZpZyA9IGRpY3QobGVuZ3RoPWxlbmd0aCwgd2lkdGg9d2lkdGgsIGxhdGVudD1sYXRlbnQpCiAgICAgICAgc2VsZi5lbnRyYW5jZV9mZWF0dXJlcyA9IGVudHJhbmNlX2ZlYXR1cmVzCiAgICAgICAgaWYgZW50cmFuY2VfZmVhdHVyZXM6CiAgICAgICAgICAgIHNlbGYuY29uZmlnWyJlbnRyYW5jZV9mZWF0dXJlcyJdID0gVHJ1ZQogICAgICAgIHNlbGYubGVuZ3RoLCBzZWxmLmxhdGVudCA9IGxlbmd0aCwgbGF0ZW50CiAgICAgICAgc2VsZi5icmFuY2ggPSBtbHAoU0VOU09SUysxLCB3aWR0aCwgMypsYXRlbnQpCiAgICAgICAgc2VsZi50cnVuayA9IG1scCgxNCBpZiBlbnRyYW5jZV9mZWF0dXJlcyBlbHNlIDEwLCB3aWR0aCwgMypsYXRlbnQpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYudHJ1bmtbLTFdLndlaWdodCwgc3RkPTFlLTMpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi50cnVua1stMV0uYmlhcykKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigic3BsaW5lX21hcCIsIHRvcmNoLnRlbnNvcihTUExJTkVfTUFQLmNvcHkoKSwgZHR5cGU9dG9yY2guZmxvYXQzMikpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImJhc2UiLCB0b3JjaC50ZW5zb3IoNipZKigxLVkpLCBkdHlwZT10b3JjaC5mbG9hdDMyKSkKCiAgICBkZWYgaW5sZXQoc2VsZiwgcHJvZmlsZXMsIHkpOgogICAgICAgICMgQWxsIHNwbGluZSBwb2x5bm9taWFsIGNvZWZmaWNpZW50cyBkZXBlbmQgbGluZWFybHkgb24gYnJhbmNoIHNlbnNvcnMuCiAgICAgICAgY29lZmYgPSB0b3JjaC5laW5zdW0oImtqcyxicy0+YmtqIiwgc2VsZi5zcGxpbmVfbWFwLCBwcm9maWxlcykKICAgICAgICBpbmRleCA9ICh5LmRldGFjaCgpKihTRU5TT1JTLTEpKS5sb25nKCkuY2xhbXAoMCwgU0VOU09SUy0yKQogICAgICAgIHQgPSB5IC0gaW5kZXggLyAoU0VOU09SUy0xKQogICAgICAgIGMgPSBjb2VmZi5nYXRoZXIoMiwgaW5kZXhbOiwgTm9uZSwgOl0uZXhwYW5kKC0xLCA0LCAtMSkpCiAgICAgICAgcmV0dXJuICgoY1s6LCAwXSp0K2NbOiwgMV0pKnQrY1s6LCAyXSkqdCtjWzosIDNdCgogICAgZGVmIGlubGV0X2ludGVncmFsKHNlbGYsIHByb2ZpbGVzLCB5KToKICAgICAgICBjb2VmZiA9IHRvcmNoLmVpbnN1bSgia2pzLGJzLT5ia2oiLCBzZWxmLnNwbGluZV9tYXAsIHByb2ZpbGVzKQogICAgICAgIGggPSAxLyhTRU5TT1JTLTEpCiAgICAgICAgaW50ZWdyYWxzID0gY29lZmZbOiwgMF0qaCoqNC80ICsgY29lZmZbOiwgMV0qaCoqMy8zICsgY29lZmZbOiwgMl0qaCpoLzIgKyBjb2VmZls6LCAzXSpoCiAgICAgICAgcHJlZml4ID0gdG9yY2guY2F0KFtpbnRlZ3JhbHMubmV3X3plcm9zKGxlbihwcm9maWxlcyksIDEpLCBpbnRlZ3JhbHMuY3Vtc3VtKC0xKV0sIC0xKQogICAgICAgIGluZGV4ID0gKHkuZGV0YWNoKCkqKFNFTlNPUlMtMSkpLmxvbmcoKS5jbGFtcCgwLCBTRU5TT1JTLTIpCiAgICAgICAgdCA9IHktaW5kZXgqaAogICAgICAgIGMgPSBjb2VmZi5nYXRoZXIoMiwgaW5kZXhbOiwgTm9uZSwgOl0uZXhwYW5kKC0xLCA0LCAtMSkpCiAgICAgICAgcmV0dXJuIHByZWZpeC5nYXRoZXIoMSwgaW5kZXgpICsgY1s6LCAwXSp0Kio0LzQrY1s6LCAxXSp0KiozLzMrY1s6LCAyXSp0KnQvMitjWzosIDNdKnQKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBwcm9maWxlcywgcmV5bm9sZHMsIHh5KToKICAgICAgICB4LCB5ID0geHlbLi4uLCAwXSwgeHlbLi4uLCAxXQogICAgICAgIGNvbmRpdGlvbiA9IHRvcmNoLmNhdChbcHJvZmlsZXMtc2VsZi5iYXNlLCAodG9yY2gubG9nKHJleW5vbGRzLzEwKS90b3JjaC5sb2cocmV5bm9sZHMubmV3X3RlbnNvcigxMC4pKSlbOiwgTm9uZV1dLCAtMSkKICAgICAgICBiID0gc2VsZi5icmFuY2goY29uZGl0aW9uKS5yZXNoYXBlKC0xLCAzLCBzZWxmLmxhdGVudCkKICAgICAgICBjb29yZHMgPSB0b3JjaC5zdGFjaygoMip4L3NlbGYubGVuZ3RoLTEsIDIqeS0xKSwgLTEpCiAgICAgICAgbW9kZXMgPSB0b3JjaC5hcmFuZ2UoMSwgNSwgZGV2aWNlPXh5LmRldmljZSwgZHR5cGU9eHkuZHR5cGUpCiAgICAgICAgcGhhc2UgPSB0b3JjaC5waSp5Wy4uLiwgTm9uZV0qbW9kZXMKICAgICAgICBmZWF0dXJlcyA9IHRvcmNoLmNhdChbY29vcmRzLCB0b3JjaC5zaW4ocGhhc2UpLCB0b3JjaC5jb3MocGhhc2UpXSwgLTEpCiAgICAgICAgaWYgc2VsZi5lbnRyYW5jZV9mZWF0dXJlczoKICAgICAgICAgICAgIyBSZXNvbHZlIHNob3J0IGVudHJhbmNlIHNjYWxlcyB3aXRob3V0IGFza2luZyB0YW5oKHgvTCkgdG8gaW52ZW50IHRoZW0uCiAgICAgICAgICAgIGxvY2FsID0gdG9yY2guc3RhY2soW3RvcmNoLmxvZzFwKDIwKngpL3RvcmNoLmxvZyh4Lm5ld190ZW5zb3IoMTAxLikpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaC5leHAoLXgpLCB0b3JjaC5leHAoLTQqeCksIHRvcmNoLmV4cCgtMTYqeCldLCAtMSkKICAgICAgICAgICAgZmVhdHVyZXMgPSB0b3JjaC5jYXQoW2ZlYXR1cmVzLCBsb2NhbF0sIC0xKQogICAgICAgIHQgPSBzZWxmLnRydW5rKGZlYXR1cmVzKS5yZXNoYXBlKCp4LnNoYXBlLCAzLCBzZWxmLmxhdGVudCkKICAgICAgICByYXcgPSB0b3JjaC5laW5zdW0oImJvbCxibm9sLT5ibm8iLCBiLCB0KSAvIHNlbGYubGF0ZW50KiouNQogICAgICAgIGF4aWFsX21hc2sgPSAxLXRvcmNoLmV4cCgteCkgaWYgc2VsZi5lbnRyYW5jZV9mZWF0dXJlcyBlbHNlIHgvc2VsZi5sZW5ndGgKICAgICAgICBtYXNrID0gYXhpYWxfbWFzayo0KnkqKDEteSkKICAgICAgICAjIERpdmVyZ2VuY2UtZnJlZSBlbnRyYW5jZS1kZWNheSBiYXNlbGluZTsgdGhlIG5ldHdvcmsgbGVhcm5zIGNvcnJlY3Rpb25zLgogICAgICAgIGVsbCA9IC4wNSpyZXlub2xkc1s6LCBOb25lXSsuMQogICAgICAgIGRlY2F5ID0gdG9yY2guZXhwKC0oeC9lbGwpKioyKQogICAgICAgIGRlbHRhID0gc2VsZi5pbmxldChwcm9maWxlcywgeSktNip5KigxLXkpCiAgICAgICAgcHNpX2RlbHRhID0gc2VsZi5pbmxldF9pbnRlZ3JhbChwcm9maWxlcywgeSktKDMqeSp5LTIqeSoqMykKICAgICAgICB1ID0gNip5KigxLXkpK2RlbHRhKmRlY2F5ICsgbWFzaypyYXdbLi4uLCAwXQogICAgICAgIHYgPSBwc2lfZGVsdGEqKDIqeC9lbGwqKjIpKmRlY2F5ICsgbWFzaypyYXdbLi4uLCAxXQogICAgICAgIHAgPSAoMTIqKHNlbGYubGVuZ3RoLXgpICsgMTIqcmF3Wy4uLiwgMl0pIC8gcmV5bm9sZHNbOiwgTm9uZV0KICAgICAgICByZXR1cm4gdG9yY2guc3RhY2soKHUsIHYsIHApLCAtMSkKCgpkZWYgZ3JhZGllbnQodmFsdWUsIHh5KToKICAgIHJldHVybiB0b3JjaC5hdXRvZ3JhZC5ncmFkKHZhbHVlLnN1bSgpLCB4eSwgY3JlYXRlX2dyYXBoPVRydWUpWzBdCgoKZGVmIHJlc2lkdWFscyhtb2RlbCwgcHJvZmlsZXMsIHJleW5vbGRzLCB4eSk6CiAgICB4eSA9IHh5LmRldGFjaCgpLnJlcXVpcmVzX2dyYWRfKFRydWUpCiAgICBmaWVsZHMgPSBtb2RlbChwcm9maWxlcywgcmV5bm9sZHMsIHh5KQogICAgdSwgdiwgcCA9IGZpZWxkcy51bmJpbmQoLTEpCiAgICBndSwgZ3YsIGdwID0gW2dyYWRpZW50KHosIHh5KSBmb3IgeiBpbiAodSwgdiwgcCldCiAgICBsYXB1ID0gZ3JhZGllbnQoZ3VbLi4uLCAwXSwgeHkpWy4uLiwgMF0gKyBncmFkaWVudChndVsuLi4sIDFdLCB4eSlbLi4uLCAxXQogICAgbGFwdiA9IGdyYWRpZW50KGd2Wy4uLiwgMF0sIHh5KVsuLi4sIDBdICsgZ3JhZGllbnQoZ3ZbLi4uLCAxXSwgeHkpWy4uLiwgMV0KICAgIG51ID0gMS9yZXlub2xkc1s6LCBOb25lXQogICAgbWFzcyA9IGd1Wy4uLiwgMF0rZ3ZbLi4uLCAxXQogICAgbXggPSB1Kmd1Wy4uLiwgMF0rdipndVsuLi4sIDFdK2dwWy4uLiwgMF0tbnUqbGFwdQogICAgbXkgPSB1Kmd2Wy4uLiwgMF0rdipndlsuLi4sIDFdK2dwWy4uLiwgMV0tbnUqbGFwdgogICAgcmV0dXJuIG1hc3MsIG14LCBteQoKCmRlZiBvdXRsZXRfcmVzaWR1YWxzKG1vZGVsLCBwcm9maWxlcywgcmV5bm9sZHMsIHh5KToKICAgIHh5ID0geHkuZGV0YWNoKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgIGZpZWxkcyA9IG1vZGVsKHByb2ZpbGVzLCByZXlub2xkcywgeHkpCiAgICAjIFN0YW5kYXJkIHVuc3ltbWV0cmljLWdyYWRpZW50IGRvLW5vdGhpbmcgYm91bmRhcnkgZnJvbSB0aGUgRkVNIHdlYWsgZm9ybS4KICAgIHV4ID0gZ3JhZGllbnQoZmllbGRzWy4uLiwgMF0sIHh5KVsuLi4sIDBdCiAgICB2eCA9IGdyYWRpZW50KGZpZWxkc1suLi4sIDFdLCB4eSlbLi4uLCAwXQogICAgcmV0dXJuIHV4LXJleW5vbGRzWzosIE5vbmVdKmZpZWxkc1suLi4sIDJdLCB2eAoKCmRlZiBsb2FkX21vZGVsKHBhdGgsIGRldmljZT0iY3B1Iik6CiAgICBjaGVja3BvaW50ID0gdG9yY2gubG9hZChwYXRoLCBtYXBfbG9jYXRpb249ZGV2aWNlLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICBtb2RlbCA9IENoYW5uZWxEZWVwT05ldCgqKmNoZWNrcG9pbnRbIm1vZGVsX2NvbmZpZyJdKS50byhkZXZpY2UpCiAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3QoY2hlY2twb2ludFsic3RhdGVfZGljdCJdKQogICAgbW9kZWwuZXZhbCgpCiAgICByZXR1cm4gbW9kZWwsIGNoZWNrcG9pbnQK', 'channel_operator/predict.py': 'aW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gLm1vZGVsIGltcG9ydCBsb2FkX21vZGVsLCByZXNpZHVhbHMKZnJvbSAucHJvZmlsZXMgaW1wb3J0IGZyb21fY29lZmZpY2llbnRzLCBsb2FkX2NzdgoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWNoZWNrcG9pbnQiLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1yZSIsIHR5cGU9ZmxvYXQsIHJlcXVpcmVkPVRydWUpCiAgICBncm91cCA9IHBhcnNlci5hZGRfbXV0dWFsbHlfZXhjbHVzaXZlX2dyb3VwKHJlcXVpcmVkPVRydWUpCiAgICBncm91cC5hZGRfYXJndW1lbnQoIi0tY29lZmZpY2llbnRzIiwgdHlwZT1mbG9hdCwgbmFyZ3M9NCkKICAgIGdyb3VwLmFkZF9hcmd1bWVudCgiLS1pbmxldC1jc3YiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQiLCBkZWZhdWx0PSJwcmVkaWN0aW9uLm5weiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW54IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MjAxKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1ueSIsIHR5cGU9aW50LCBkZWZhdWx0PTgxKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIGlmIG5vdCAxMCA8PSBhcmdzLnJlIDw9IDEwMDoKICAgICAgICBwYXJzZXIuZXJyb3IoIlZhbGlkYXRlZCBSZXlub2xkcyBpbnRlcnZhbCBpcyBbMTAsMTAwXSIpCiAgICBwcm9maWxlID0gbG9hZF9jc3YoYXJncy5pbmxldF9jc3YpIGlmIGFyZ3MuaW5sZXRfY3N2IGVsc2UgZnJvbV9jb2VmZmljaWVudHMoYXJncy5jb2VmZmljaWVudHMpCiAgICBpZiBhcmdzLmNvZWZmaWNpZW50cyBhbmQgc3VtKGFicyh4KSBmb3IgeCBpbiBhcmdzLmNvZWZmaWNpZW50cykgPiAuNjoKICAgICAgICBwcmludCgiV0FSTklORzogcHJvZmlsZSBpcyBvdXRzaWRlIHRoZSB0cmFpbmluZyBjb2VmZmljaWVudCBkb21haW4iKQogICAgaWYgYXJncy5pbmxldF9jc3Y6CiAgICAgICAgcHJpbnQoIldBUk5JTkc6IGN1c3RvbSBwcm9maWxlcyBtYXkgYmUgb3V0c2lkZSB0aGUgdHJhaW5pbmcgZmFtaWx5OyB2YWxpZGF0ZSBhZ2FpbnN0IENGRCIpCiAgICBtb2RlbCwgY2hlY2twb2ludCA9IGxvYWRfbW9kZWwoYXJncy5jaGVja3BvaW50KQogICAgbWV0cmljc19wYXRoID0gUGF0aChhcmdzLmNoZWNrcG9pbnQpLnBhcmVudC8ibWV0cmljcy5qc29uIgogICAgdmFsaWRhdGlvbiA9IGpzb24ubG9hZHMobWV0cmljc19wYXRoLnJlYWRfdGV4dCgpKSBpZiBtZXRyaWNzX3BhdGguZXhpc3RzKCkgZWxzZSBOb25lCiAgICB2YWxpZGF0aW9uX2N1cnJlbnQgPSBib29sKHZhbGlkYXRpb24gYW5kIHZhbGlkYXRpb24uZ2V0KCJjaGVja3BvaW50X3N0ZXAiKSA9PSBjaGVja3BvaW50WyJzdGVwIl0pCiAgICBpZiBub3QgdmFsaWRhdGlvbl9jdXJyZW50IG9yIG5vdCB2YWxpZGF0aW9uLmdldCgicGFzc2VkIik6CiAgICAgICAgcHJpbnQoIldBUk5JTkc6IHRoaXMgY2hlY2twb2ludCBoYXMgbm90IHBhc3NlZCBhbGwgZG9jdW1lbnRlZCB2YWxpZGF0aW9uIGdhdGVzIikKICAgIHgsIHkgPSBucC5tZXNoZ3JpZChucC5saW5zcGFjZSgwLCA1LCBhcmdzLm54KSwgbnAubGluc3BhY2UoMCwgMSwgYXJncy5ueSksIGluZGV4aW5nPSJpaiIpCiAgICB4eSA9IG5wLnN0YWNrKFt4LCB5XSwgLTEpCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICB2YWx1ZXMgPSBtb2RlbCh0b3JjaC50ZW5zb3IocHJvZmlsZVtOb25lXSwgZHR5cGU9dG9yY2guZmxvYXQzMiksIHRvcmNoLnRlbnNvcihbYXJncy5yZV0pLAogICAgICAgICAgICAgICAgICAgICAgIHRvcmNoLnRlbnNvcih4eS5yZXNoYXBlKDEsIC0xLCAyKSwgZHR5cGU9dG9yY2guZmxvYXQzMikpLm51bXB5KCkucmVzaGFwZShhcmdzLm54LCBhcmdzLm55LCAzKQogICAgbnAuc2F2ZXpfY29tcHJlc3NlZChhcmdzLm91dHB1dCwgeD14LCB5PXksIHU9dmFsdWVzWy4uLiwgMF0sIHY9dmFsdWVzWy4uLiwgMV0sIHA9dmFsdWVzWy4uLiwgMl0sIGlubGV0PXByb2ZpbGUsIHJleW5vbGRzPWFyZ3MucmUpCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoMjI0KQogICAgcG9pbnRzID0gdG9yY2gudGVuc29yKHJuZy51bmlmb3JtKFsuMDA1LCAuMDA1XSwgWzQuOTk1LCAuOTk1XSwgKDEsIDEwMjQsIDIpKSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgIG1hc3MsIG14LCBteSA9IHJlc2lkdWFscyhtb2RlbCwgdG9yY2gudGVuc29yKHByb2ZpbGVbTm9uZV0sIGR0eXBlPXRvcmNoLmZsb2F0MzIpLCB0b3JjaC50ZW5zb3IoW2FyZ3MucmVdKSwgcG9pbnRzKQogICAgZmx1eCA9IG5wLnRyYXBlem9pZCh2YWx1ZXNbLi4uLCAwXSwgeSwgYXhpcz0tMSkKICAgIGRpYWdub3N0aWNzID0geyJvdXRwdXQiOiBhcmdzLm91dHB1dCwgInJleW5vbGRzIjogYXJncy5yZSwgInRyYWluZWRfc3RlcCI6IGNoZWNrcG9pbnRbInN0ZXAiXSwgInJldHJhaW5pbmciOiBGYWxzZSwKICAgICAgICAgICAgICAgICAgICJjaGVja3BvaW50X3Bhc3NlZF92YWxpZGF0aW9uIjogYm9vbCh2YWxpZGF0aW9uX2N1cnJlbnQgYW5kIHZhbGlkYXRpb24uZ2V0KCJwYXNzZWQiKSksCiAgICAgICAgICAgICAgICAgICAibWF4X2ZsdXhfZXJyb3IiOiBmbG9hdChucC5tYXgobnAuYWJzKGZsdXgtMSkpKSwKICAgICAgICAgICAgICAgICAgICJkaXZlcmdlbmNlX3JtcyI6IGZsb2F0KG1hc3Muc3F1YXJlKCkubWVhbigpLnNxcnQoKS5kZXRhY2goKSksCiAgICAgICAgICAgICAgICAgICAibW9tZW50dW1fcm1zIjogZmxvYXQoKG14LnNxdWFyZSgpLm1lYW4oKStteS5zcXVhcmUoKS5tZWFuKCkpLnNxcnQoKS5kZXRhY2goKSksCiAgICAgICAgICAgICAgICAgICAibm90ZSI6ICJSZXNpZHVhbHMgYXJlIGRpYWdub3N0aWNzLCBub3QgYSBzdWJzdGl0dXRlIGZvciBpbmRlcGVuZGVudCBDRkQgZXJyb3IgZXN0aW1hdGVzLiJ9CiAgICBQYXRoKGFyZ3Mub3V0cHV0KS53aXRoX3N1ZmZpeCgiLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGlhZ25vc3RpY3MsIGluZGVudD0yKSkKICAgIHByaW50KGpzb24uZHVtcHMoZGlhZ25vc3RpY3MpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK', 'channel_operator/profiles.py': 'IiIiTm8tc2xpcCBpbmxldCBwcm9maWxlcywgcmVwcmVzZW50ZWQgYnkgbm9ybWFsaXplZCBjdWJpYy1zcGxpbmUgc2Vuc29ycy4iIiIKaW1wb3J0IG51bXB5IGFzIG5wCmZyb20gc2NpcHkuaW50ZXJwb2xhdGUgaW1wb3J0IEN1YmljU3BsaW5lCgpTRU5TT1JTID0gMzMKWSA9IG5wLmxpbnNwYWNlKDAuLCAxLiwgU0VOU09SUykKIyBOb3QtYS1rbm90IGludGVycG9sYXRpb24gcmVwcm9kdWNlcyBxdWFkcmF0aWNzIChpbmNsdWRpbmcgUG9pc2V1aWxsZSkgZXhhY3RseS4KX2Jhc2lzID0gQ3ViaWNTcGxpbmUoWSwgbnAuZXllKFNFTlNPUlMpLCBheGlzPTApClNQTElORV9NQVAgPSBfYmFzaXMuYwpJTlRFR1JBTF9XRUlHSFRTID0gX2Jhc2lzLmludGVncmF0ZSgwLiwgMS4pCgoKZGVmIG5vcm1hbGl6ZSh2YWx1ZXMpOgogICAgdmFsdWVzID0gbnAuYXNhcnJheSh2YWx1ZXMsIGR0eXBlPWZsb2F0KQogICAgaWYgdmFsdWVzLnNoYXBlWy0xXSAhPSBTRU5TT1JTIG9yIG5vdCBucC5pc2Zpbml0ZSh2YWx1ZXMpLmFsbCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCB7U0VOU09SU30gZmluaXRlIGlubGV0IHNhbXBsZXMiKQogICAgaWYgbnAubWF4KG5wLmFicyh2YWx1ZXNbLi4uLCBbMCwgLTFdXSkpID4gMWUtODoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbmxldCBtdXN0IGJlIHplcm8gYXQgYm90aCBuby1zbGlwIHdhbGxzIikKICAgIGRlbnNlID0gQ3ViaWNTcGxpbmUoWSwgdmFsdWVzLCBheGlzPS0xKShucC5saW5zcGFjZSgwLiwgMS4sIDI1NykpCiAgICBpZiBkZW5zZS5taW4oKSA8IC0xZS04OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlN1cHBvcnRlZCBpbmxldHMgbXVzdCBiZSBub25uZWdhdGl2ZSwgaW5jbHVkaW5nIHNwbGluZSBpbnRlcnBvbGF0aW9uIikKICAgIG1lYW4gPSB2YWx1ZXMgQCBJTlRFR1JBTF9XRUlHSFRTCiAgICBpZiBucC5hbnkobWVhbiA8PSAwKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbmxldCBtdXN0IGhhdmUgYSBwb3NpdGl2ZSBtZWFuIHZlbG9jaXR5IikKICAgIHJldHVybiB2YWx1ZXMgLyBtZWFuWy4uLiwgTm9uZV0KCgpkZWYgZnJvbV9jb2VmZmljaWVudHMoY29lZmZpY2llbnRzKToKICAgIGEgPSBucC5hc2FycmF5KGNvZWZmaWNpZW50cykKICAgIG1vZGVzID0gbnAuYXJhbmdlKDEsIGEuc2hhcGVbLTFdICsgMSkKICAgIG1vZHVsYXRpb24gPSAxICsgYSBAIG5wLmNvcyhucC5waSAqIG1vZGVzWzosIE5vbmVdICogWSkKICAgIHJldHVybiBub3JtYWxpemUoNiAqIFkgKiAoMS1ZKSAqIG1vZHVsYXRpb24pCgoKZGVmIHNhbXBsZV9wcm9maWxlcyhybmcsIGNvdW50LCBtb2Rlcz00LCByYWRpdXM9LjYpOgogICAgYSA9IHJuZy5ub3JtYWwoc2l6ZT0oY291bnQsIG1vZGVzKSkKICAgIGEgKj0gcm5nLnVuaWZvcm0oMC4sIHJhZGl1cywgKGNvdW50LCAxKSkgLyBucC5hYnMoYSkuc3VtKGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKICAgIHJldHVybiBmcm9tX2NvZWZmaWNpZW50cyhhKSwgYQoKCmRlZiBsb2FkX2NzdihwYXRoKToKICAgICIiIkNTViB3aXRoIGhlYWRlciB5LHU7IHkgaXMgbm9uZGltZW5zaW9uYWwgYW5kIHNwYW5zIFswLDFdLiIiIgogICAgZGF0YSA9IG5wLmxvYWR0eHQocGF0aCwgZGVsaW1pdGVyPSIsIiwgc2tpcHJvd3M9MSkKICAgIGlmIGRhdGEubmRpbSAhPSAyIG9yIGRhdGEuc2hhcGVbMV0gIT0gMiBvciBsZW4oZGF0YSkgPCA0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNTViBtdXN0IGNvbnRhaW4gYXQgbGVhc3QgZm91ciB5LHUgcm93cyIpCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoZGF0YSkuYWxsKCkgb3IgbnAuYW55KG5wLmRpZmYoZGF0YVs6LCAwXSkgPD0gMCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQ1NWIGNvb3JkaW5hdGVzIG11c3QgYmUgZmluaXRlIGFuZCBzdHJpY3RseSBpbmNyZWFzaW5nIikKICAgIGlmIG5vdCBucC5hbGxjbG9zZShkYXRhW1swLCAtMV0sIDBdLCBbMCwgMV0sIGF0b2w9MWUtMTApOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNTViB5IG11c3Qgc3BhbiBbMCwgMV0iKQogICAgcmV0dXJuIG5vcm1hbGl6ZShDdWJpY1NwbGluZShkYXRhWzosIDBdLCBkYXRhWzosIDFdKShZKSkK', 'channel_operator/reference.py': 'IiIiSW5kZXBlbmRlbnQgVGF5bG9yLUhvb2QgUDIvUDEgc3RlYWR5IE5hdmllci1TdG9rZXMgZmluaXRlLWVsZW1lbnQgc29sdmVyLgoKTm8gUElOTiBwcmVkaWN0aW9ucyBhcmUgdXNlZCBieSB0aGlzIHNvbHZlci4gTmF0dXJhbCBvdXRsZXQgdXNlcyB2aXNjb3NpdHkqZHUvZG4tcG49MC4KIiIiCmltcG9ydCBudW1weSBhcyBucApmcm9tIHNjaXB5LmludGVycG9sYXRlIGltcG9ydCBDdWJpY1NwbGluZQpmcm9tIHNjaXB5LnNwYXJzZSBpbXBvcnQgYm1hdCwgY3NyX21hdHJpeApmcm9tIHNrZmVtIGltcG9ydCBNZXNoVHJpLCBCYXNpcywgRWxlbWVudFRyaVAyLCBFbGVtZW50VHJpUDEsIEVsZW1lbnRWZWN0b3IsIEJpbGluZWFyRm9ybSwgYXNtLCBjb25kZW5zZSwgc29sdmUKZnJvbSBza2ZlbS5oZWxwZXJzIGltcG9ydCBkZG90LCBkb3QsIGRpdgpmcm9tIC5wcm9maWxlcyBpbXBvcnQgWQoKCkBCaWxpbmVhckZvcm0KZGVmIGRpZmZ1c2lvbih1LCB2LCB3KToKICAgIHJldHVybiBkZG90KHUuZ3JhZCwgdi5ncmFkKQoKCkBCaWxpbmVhckZvcm0KZGVmIGRpdmVyZ2VuY2UodSwgcSwgdyk6CiAgICByZXR1cm4gLXEqZGl2KHUpCgoKQEJpbGluZWFyRm9ybQpkZWYgY29udmVjdGlvbih1LCB2LCB3KToKICAgIHJldHVybiBkb3QobnAuZWluc3VtKCJqLi4uLGlqLi4uLT5pLi4uIiwgdy53aW5kLCB1LmdyYWQpLCB2KQoKCmRlZiBzb2x2ZV9jaGFubmVsKHByb2ZpbGUsIHJleW5vbGRzLCBsZW5ndGg9NS4sIG54PTQwLCBueT0xNiwgdG9sZXJhbmNlPTFlLTcsIG1heF9pdGVyYXRpb25zPTE1MCwgaW5sZXRfZnVuY3Rpb249Tm9uZSk6CiAgICBtZXNoID0gTWVzaFRyaS5pbml0X3RlbnNvcihucC5saW5zcGFjZSgwLCBsZW5ndGgsIG54KzEpLCBucC5saW5zcGFjZSgwLCAxLCBueSsxKSkKICAgIHViID0gQmFzaXMobWVzaCwgRWxlbWVudFZlY3RvcihFbGVtZW50VHJpUDIoKSksIGludG9yZGVyPTUpCiAgICBwYiA9IEJhc2lzKG1lc2gsIEVsZW1lbnRUcmlQMSgpLCBpbnRvcmRlcj01KQogICAgYSwgYiA9IGFzbShkaWZmdXNpb24sIHViKS9yZXlub2xkcywgYXNtKGRpdmVyZ2VuY2UsIHViLCBwYikKICAgIHdhbGxzID0gdWIuZ2V0X2RvZnMobGFtYmRhIHg6IG5wLmlzY2xvc2UoeFsxXSwgMCkgfCBucC5pc2Nsb3NlKHhbMV0sIDEpKQogICAgaW5sZXQgPSB1Yi5nZXRfZG9mcyhsYW1iZGEgeDogbnAuaXNjbG9zZSh4WzBdLCAwKSkKICAgIGZpeGVkID0gbnAudW5pb24xZCh3YWxscy5hbGwoKSwgaW5sZXQuYWxsKCkpCiAgICBiYyA9IG5wLnplcm9zKHViLk4rcGIuTikKICAgIGlubGV0X3UgPSBpbmxldC5hbGwoWyJ1XjEiXSkKICAgIGlubGV0X2Z1bmN0aW9uID0gQ3ViaWNTcGxpbmUoWSwgcHJvZmlsZSkgaWYgaW5sZXRfZnVuY3Rpb24gaXMgTm9uZSBlbHNlIGlubGV0X2Z1bmN0aW9uCiAgICBiY1tpbmxldF91XSA9IGlubGV0X2Z1bmN0aW9uKHViLmRvZmxvY3NbMSwgaW5sZXRfdV0pCiAgICB6ZXJvID0gY3NyX21hdHJpeCgocGIuTiwgcGIuTikpCiAgICBzdG9rZXMgPSBibWF0KFtbYSwgYi5UXSwgW2IsIHplcm9dXSwgZm9ybWF0PSJjc3IiKQogICAgc29sdXRpb24gPSBzb2x2ZSgqY29uZGVuc2Uoc3Rva2VzLCBucC56ZXJvcyhsZW4oYmMpKSwgeD1iYywgRD1maXhlZCkpCiAgICBmcmVlID0gbnAuc2V0ZGlmZjFkKG5wLmFyYW5nZShsZW4oYmMpKSwgZml4ZWQpCiAgICBjb252ZXJnZWQgPSBGYWxzZQogICAgZm9yIGl0ZXJhdGlvbiBpbiByYW5nZShtYXhfaXRlcmF0aW9ucyk6CiAgICAgICAgYyA9IGFzbShjb252ZWN0aW9uLCB1Yiwgd2luZD11Yi5pbnRlcnBvbGF0ZShzb2x1dGlvbls6dWIuTl0pKQogICAgICAgIG1hdHJpeCA9IGJtYXQoW1thK2MsIGIuVF0sIFtiLCB6ZXJvXV0sIGZvcm1hdD0iY3NyIikKICAgICAgICB1cGRhdGVkID0gc29sdmUoKmNvbmRlbnNlKG1hdHJpeCwgbnAuemVyb3MobGVuKGJjKSksIHg9YmMsIEQ9Zml4ZWQpKQogICAgICAgIGVycm9yID0gbnAubGluYWxnLm5vcm0odXBkYXRlZFs6dWIuTl0tc29sdXRpb25bOnViLk5dKSAvIG1heChucC5saW5hbGcubm9ybSh1cGRhdGVkWzp1Yi5OXSksIDFlLTEyKQogICAgICAgIHNvbHV0aW9uID0gLjcqdXBkYXRlZCArIC4zKnNvbHV0aW9uCiAgICAgICAgc29sdXRpb25bZml4ZWRdID0gYmNbZml4ZWRdCiAgICAgICAgaWYgZXJyb3IgPCB0b2xlcmFuY2U6CiAgICAgICAgICAgIGNvbnZlcmdlZCA9IFRydWUKICAgICAgICAgICAgYnJlYWsKICAgICMgQ2hlY2sgdGhlIG5vbmxpbmVhciBhbGdlYnJhaWMgZXF1YXRpb25zLCBub3Qgb25seSB0aGUgaXRlcmF0ZSBpbmNyZW1lbnQuCiAgICBjID0gYXNtKGNvbnZlY3Rpb24sIHViLCB3aW5kPXViLmludGVycG9sYXRlKHNvbHV0aW9uWzp1Yi5OXSkpCiAgICBtYXRyaXggPSBibWF0KFtbYStjLCBiLlRdLCBbYiwgemVyb11dLCBmb3JtYXQ9ImNzciIpCiAgICBub25saW5lYXJfcmVzaWR1YWwgPSBucC5saW5hbGcubm9ybSgobWF0cml4QHNvbHV0aW9uKVtmcmVlXSkKICAgIGlmIG5vdCBjb252ZXJnZWQgb3Igbm90IG5wLmlzZmluaXRlKHNvbHV0aW9uKS5hbGwoKSBvciBub25saW5lYXJfcmVzaWR1YWwgPiAxZS02OgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkZFTSBmYWlsZWQgYXQgUmU9e3JleW5vbGRzfTogZGVsdGE9e2Vycm9yOi4zZ30sIHJlc2lkdWFsPXtub25saW5lYXJfcmVzaWR1YWw6LjNnfSIpCgogICAgZGVmIGV2YWx1YXRlKHh5KToKICAgICAgICB4eSA9IG5wLmFzYXJyYXkoeHkpCiAgICAgICAgdmVsb2NpdHkgPSB1Yi5pbnRlcnBvbGF0b3Ioc29sdXRpb25bOnViLk5dKSh4eS5UKS5UCiAgICAgICAgcHJlc3N1cmUgPSBwYi5pbnRlcnBvbGF0b3Ioc29sdXRpb25bdWIuTjpdKSh4eS5UKQogICAgICAgIHJldHVybiBucC5jb2x1bW5fc3RhY2soW3ZlbG9jaXR5LCBwcmVzc3VyZV0pCgogICAgcmV0dXJuIGV2YWx1YXRlLCBkaWN0KGl0ZXJhdGlvbnM9aXRlcmF0aW9uKzEsIGluY3JlbWVudD1mbG9hdChlcnJvciksIG5vbmxpbmVhcl9yZXNpZHVhbD1mbG9hdChub25saW5lYXJfcmVzaWR1YWwpLCBueD1ueCwgbnk9bnkpCg==', 'channel_operator/train.py': 'IiIiVHJhaW4gb25lIG9wZXJhdG9yIGZvciB0aGUgZW50aXJlIGlubGV0L1JlIGZhbWlseTsgbmV2ZXIgcmVmaXQgYXQgaW5mZXJlbmNlLiIiIgppbXBvcnQganNvbgppbXBvcnQgbWF0aAppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaApmcm9tIC5tb2RlbCBpbXBvcnQgQ2hhbm5lbERlZXBPTmV0LCByZXNpZHVhbHMsIG91dGxldF9yZXNpZHVhbHMKZnJvbSAucHJvZmlsZXMgaW1wb3J0IHNhbXBsZV9wcm9maWxlcwoKCmRlZiB0cmFpbihvdXRwdXQsIHN0ZXBzPTEyMDAwLCBzZWVkPTQyLCBiYXRjaD04LCBwb2ludHM9OTYsIHdpZHRoPTY0LCBsYXRlbnQ9MzIsIHBoeXNpY3Nfb25seT1GYWxzZSwgcmVzdW1lPU5vbmUsIGVudHJhbmNlX2ZlYXR1cmVzPUZhbHNlLCBkYXRhX3dlaWdodD1Ob25lKToKICAgIG91dHB1dCA9IFBhdGgob3V0cHV0KQogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpCiAgICBucC5yYW5kb20uc2VlZChzZWVkKQogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICB0b3JjaC5zZXRfbnVtX3RocmVhZHMoNCkKICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgbW9kZWwgPSBDaGFubmVsRGVlcE9OZXQod2lkdGg9d2lkdGgsIGxhdGVudD1sYXRlbnQsIGVudHJhbmNlX2ZlYXR1cmVzPWVudHJhbmNlX2ZlYXR1cmVzKS50byhkZXZpY2UpCiAgICBkYXRhX3dlaWdodCA9ICgyMDAgaWYgZW50cmFuY2VfZmVhdHVyZXMgZWxzZSAyMCkgaWYgZGF0YV93ZWlnaHQgaXMgTm9uZSBlbHNlIGRhdGFfd2VpZ2h0CiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9MmUtMykKICAgIGhpc3RvcnksIGZpcnN0X3N0ZXAgPSBbXSwgMAogICAgaWYgcmVzdW1lOgogICAgICAgIGNoZWNrcG9pbnQgPSB0b3JjaC5sb2FkKHJlc3VtZSwgbWFwX2xvY2F0aW9uPWRldmljZSwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgICAgIGlmIGNoZWNrcG9pbnRbIm1vZGVsX2NvbmZpZyJdICE9IG1vZGVsLmNvbmZpZzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUmVzdW1lIG1vZGVsIGFyY2hpdGVjdHVyZSBtaXNtYXRjaCIpCiAgICAgICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KGNoZWNrcG9pbnRbInN0YXRlX2RpY3QiXSkKICAgICAgICBvcHRpbWl6ZXIubG9hZF9zdGF0ZV9kaWN0KGNoZWNrcG9pbnRbIm9wdGltaXplciJdKQogICAgICAgIGZpcnN0X3N0ZXAgPSBjaGVja3BvaW50WyJzdGVwIl0KICAgICAgICBpZiBzdGVwcyA8PSBmaXJzdF9zdGVwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVGFyZ2V0IHN0ZXBzIG11c3QgZXhjZWVkIHJlc3VtZWQgY2hlY2twb2ludCBzdGVwIHtmaXJzdF9zdGVwfSIpCiAgICB0ZW5zb3IgPSBsYW1iZGEgYTogdG9yY2guYXNfdGVuc29yKGEsIGR0eXBlPXRvcmNoLmZsb2F0MzIsIGRldmljZT1kZXZpY2UpCiAgICBkYXRhID0gTm9uZQogICAgaWYgbm90IHBoeXNpY3Nfb25seToKICAgICAgICB3aXRoIG5wLmxvYWQob3V0cHV0LyJ0cmFpbi5ucHoiKSBhcyBkOgogICAgICAgICAgICBkYXRhID0ge2s6IHRlbnNvcihkW2tdKSBmb3IgayBpbiBbInByb2ZpbGVzIiwgInJleW5vbGRzIiwgInh5IiwgImZpZWxkcyJdfQogICAgc3RhcnQgPSB0aW1lLnRpbWUoKQogICAgbG9nID0gb3BlbihvdXRwdXQvInRyYWluaW5nLmpzb25sIiwgImEiLCBidWZmZXJpbmc9MSkKCiAgICBkZWYgc2F2ZShzdGVwKToKICAgICAgICBzdGF0ZSA9IHsic3RhdGVfZGljdCI6IG1vZGVsLnN0YXRlX2RpY3QoKSwgIm1vZGVsX2NvbmZpZyI6IG1vZGVsLmNvbmZpZywgIm9wdGltaXplciI6IG9wdGltaXplci5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICAgICAgInN0ZXAiOiBzdGVwLCAic2VlZCI6IHNlZWQsICJtb2RlIjogInBoeXNpY3Nfb25seSIgaWYgcGh5c2ljc19vbmx5IGVsc2UgImh5YnJpZF9waHlzaWNzX2luZm9ybWVkIiwKICAgICAgICAgICAgICAgICAidHJhaW5pbmdfZG9tYWluIjogeyJsZW5ndGgiOiA1LCAiaGVpZ2h0IjogMSwgInJleW5vbGRzIjogWzEwLCAxMDBdLCAiY29zaW5lX21vZGVzIjogNCwgImNvZWZmaWNpZW50X2wxX3JhZGl1cyI6IC42fSwKICAgICAgICAgICAgICAgICAidG9yY2hfdmVyc2lvbiI6IHRvcmNoLl9fdmVyc2lvbl9ffQogICAgICAgIHN0YXRlWyJsb3NzX2NvbmZpZyJdID0gZGljdChtYXNzPTEwLCBtb21lbnR1bT0xLCBvdXRsZXQ9LjEsIGZsdXg9MTAsIGRhdGE9ZGF0YV93ZWlnaHQpCiAgICAgICAgdGVtcG9yYXJ5ID0gb3V0cHV0LyJjaGVja3BvaW50LnRtcCIKICAgICAgICB0b3JjaC5zYXZlKHN0YXRlLCB0ZW1wb3JhcnkpCiAgICAgICAgdGVtcG9yYXJ5LnJlcGxhY2Uob3V0cHV0LyJtb2RlbC5wdCIpCgogICAgZm9yIHN0ZXAgaW4gcmFuZ2UoZmlyc3Rfc3RlcCsxLCBzdGVwcysxKToKICAgICAgICBtb2RlbC50cmFpbigpCiAgICAgICAgb3B0aW1pemVyLnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKQogICAgICAgIHByb2dyZXNzID0gc3RlcC9zdGVwcwogICAgICAgIGxyID0gMmUtNCArIC41KigyZS0zLTJlLTQpKigxK21hdGguY29zKG1hdGgucGkqcHJvZ3Jlc3MpKQogICAgICAgIGZvciBncm91cCBpbiBvcHRpbWl6ZXIucGFyYW1fZ3JvdXBzOgogICAgICAgICAgICBncm91cFsibHIiXSA9IGxyCiAgICAgICAgIyBFdmVyeSBzdGVwIHNhbXBsZXMgTkVXIGlubGV0L1JlIGNvbmRpdGlvbnMsIG5vdCBqdXN0IHRoZSBGRU0gY2FzZXMuCiAgICAgICAgcHJvZmlsZXMsIF8gPSBzYW1wbGVfcHJvZmlsZXMocm5nLCBiYXRjaCkKICAgICAgICBwcm9maWxlcyA9IHRlbnNvcihwcm9maWxlcykKICAgICAgICByZSA9IHRlbnNvcihucC5leHAocm5nLnVuaWZvcm0obnAubG9nKDEwKSwgbnAubG9nKDEwMCksIGJhdGNoKSkpCiAgICAgICAgeHkgPSB0b3JjaC5yYW5kKGJhdGNoLCBwb2ludHMsIDIsIGRldmljZT1kZXZpY2UpCiAgICAgICAgIyBIYWxmIG9mIHggbG9jYXRpb25zIGZvY3VzIG9uIHRoZSBkZXZlbG9waW5nIGVudHJhbmNlIHJlZ2lvbi4KICAgICAgICB4eVsuLi4sIDBdID0gNSp4eVsuLi4sIDBdKioyCiAgICAgICAgbWFzcywgbXgsIG15ID0gcmVzaWR1YWxzKG1vZGVsLCBwcm9maWxlcywgcmUsIHh5KQogICAgICAgIG1hc3NfbG9zcyA9IG1hc3Muc3F1YXJlKCkubWVhbigpCiAgICAgICAgbW9tZW50dW0gPSBteC5zcXVhcmUoKS5tZWFuKCkrbXkuc3F1YXJlKCkubWVhbigpCiAgICAgICAgb3V0bGV0ID0gdG9yY2gucmFuZChiYXRjaCwgMzIsIDIsIGRldmljZT1kZXZpY2UpCiAgICAgICAgb3V0bGV0Wy4uLiwgMF0gPSA1CiAgICAgICAgb24sIG90ID0gb3V0bGV0X3Jlc2lkdWFscyhtb2RlbCwgcHJvZmlsZXMsIHJlLCBvdXRsZXQpCiAgICAgICAgYmNfbG9zcyA9IG9uLnNxdWFyZSgpLm1lYW4oKStvdC5zcXVhcmUoKS5tZWFuKCkKICAgICAgICAjIEludGVncmF0ZWQgbWFzcyBiYWxhbmNlIG9uIHJhbmRvbWx5IHNlbGVjdGVkIGNyb3NzLXNlY3Rpb25zLgogICAgICAgIGd4ID0gdG9yY2gucmFuZChiYXRjaCwgMywgMSwgZGV2aWNlPWRldmljZSkqNQogICAgICAgIGd5ID0gdG9yY2gubGluc3BhY2UoMCwgMSwgNDksIGRldmljZT1kZXZpY2UpCiAgICAgICAgZ3JpZCA9IHRvcmNoLnN0YWNrKChneC5leHBhbmQoLTEsIC0xLCBsZW4oZ3kpKSwgZ3lbTm9uZSwgTm9uZSwgOl0uZXhwYW5kKGJhdGNoLCAzLCAtMSkpLCAtMSkucmVzaGFwZShiYXRjaCwgLTEsIDIpCiAgICAgICAgdSA9IG1vZGVsKHByb2ZpbGVzLCByZSwgZ3JpZClbLi4uLCAwXS5yZXNoYXBlKGJhdGNoLCAzLCAtMSkKICAgICAgICBmbHV4X2xvc3MgPSAodG9yY2gudHJhcGV6b2lkKHUsIGd5LCBkaW09LTEpLTEpLnNxdWFyZSgpLm1lYW4oKQogICAgICAgIHN1cGVydmlzZWQgPSB0ZW5zb3IoMC4pCiAgICAgICAgaWYgZGF0YSBpcyBub3QgTm9uZToKICAgICAgICAgICAgY2kgPSB0b3JjaC5yYW5kaW50KGxlbihkYXRhWyJwcm9maWxlcyJdKSwgKGJhdGNoLCksIGRldmljZT1kZXZpY2UpCiAgICAgICAgICAgIHBpID0gdG9yY2gucmFuZGludChsZW4oZGF0YVsieHkiXSksICgxOTIsKSwgZGV2aWNlPWRldmljZSkKICAgICAgICAgICAgcHJlZGljdGlvbiA9IG1vZGVsKGRhdGFbInByb2ZpbGVzIl1bY2ldLCBkYXRhWyJyZXlub2xkcyJdW2NpXSwgZGF0YVsieHkiXVtwaV1bTm9uZV0uZXhwYW5kKGJhdGNoLCAtMSwgLTEpKQogICAgICAgICAgICB0YXJnZXQgPSBkYXRhWyJmaWVsZHMiXVtjaVs6LCBOb25lXSwgcGlbTm9uZSwgOl1dCiAgICAgICAgICAgIHNjYWxlID0gdG9yY2gub25lcyhiYXRjaCwgMSwgMywgZGV2aWNlPWRldmljZSkKICAgICAgICAgICAgc2NhbGVbLi4uLCAyXSA9IGRhdGFbInJleW5vbGRzIl1bY2ksIE5vbmVdLzYwCiAgICAgICAgICAgIHN1cGVydmlzZWQgPSAoKHByZWRpY3Rpb24tdGFyZ2V0KSpzY2FsZSkuc3F1YXJlKCkubWVhbigpCiAgICAgICAgIyBGRU0gd2FybS1zdGFydCByZW1haW5zIHBhcnQgb2YgdGhlIGxvc3MsIHBoeXNpY3MgaXMgYWN0aXZlIGZyb20gc3RlcCAxLgogICAgICAgIGxvc3MgPSAxMCptYXNzX2xvc3MgKyBtb21lbnR1bSArIC4xKmJjX2xvc3MgKyAxMCpmbHV4X2xvc3MgKyBkYXRhX3dlaWdodCpzdXBlcnZpc2VkCiAgICAgICAgaWYgbm90IHRvcmNoLmlzZmluaXRlKGxvc3MpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJOb25maW5pdGUgbG9zcyBhdCBzdGVwIHtzdGVwfSIpCiAgICAgICAgbG9zcy5iYWNrd2FyZCgpCiAgICAgICAgdG9yY2gubm4udXRpbHMuY2xpcF9ncmFkX25vcm1fKG1vZGVsLnBhcmFtZXRlcnMoKSwgNS4pCiAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgIGlmIHN0ZXAgPT0gMSBvciBzdGVwICUgMTAwID09IDAgb3Igc3RlcCA9PSBzdGVwczoKICAgICAgICAgICAgcmVjb3JkID0geyJzdGVwIjogc3RlcCwgImxvc3MiOiBmbG9hdChsb3NzLmRldGFjaCgpKSwgImRpdmVyZ2VuY2VfbXNlIjogZmxvYXQobWFzc19sb3NzLmRldGFjaCgpKSwKICAgICAgICAgICAgICAgICAgICAgICJtb21lbnR1bV9tc2UiOiBmbG9hdChtb21lbnR1bS5kZXRhY2goKSksICJvdXRsZXRfbXNlIjogZmxvYXQoYmNfbG9zcy5kZXRhY2goKSksCiAgICAgICAgICAgICAgICAgICAgICAiZmx1eF9tc2UiOiBmbG9hdChmbHV4X2xvc3MuZGV0YWNoKCkpLCAic3VwZXJ2aXNlZF9tc2UiOiBmbG9hdChzdXBlcnZpc2VkLmRldGFjaCgpKSwKICAgICAgICAgICAgICAgICAgICAgICJzZWNvbmRzIjogdGltZS50aW1lKCktc3RhcnQsICJsciI6IGxyfQogICAgICAgICAgICBsb2cud3JpdGUoanNvbi5kdW1wcyhyZWNvcmQpKyJcbiIpCiAgICAgICAgICAgIGhpc3RvcnkuYXBwZW5kKHJlY29yZCkKICAgICAgICAgICAgcHJpbnQoanNvbi5kdW1wcyhyZWNvcmQpLCBmbHVzaD1UcnVlKQogICAgICAgIGlmIHN0ZXAgJSAxMDAwID09IDAgb3Igc3RlcCA9PSBzdGVwczoKICAgICAgICAgICAgc2F2ZShzdGVwKQogICAgbG9nLmNsb3NlKCkKICAgIChvdXRwdXQvInRyYWluaW5nX3N1bW1hcnkuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyh7ImRldmljZSI6IHN0cihkZXZpY2UpLCAiZ3B1IjogdG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoKSBpZiBkZXZpY2UudHlwZT09ImN1ZGEiIGVsc2UgTm9uZSwKICAgICAgICAic3RlcHMiOiBzdGVwcywgInNlY29uZHMiOiB0aW1lLnRpbWUoKS1zdGFydCwgIm1vZGUiOiAicGh5c2ljc19vbmx5IiBpZiBwaHlzaWNzX29ubHkgZWxzZSAiaHlicmlkX3BoeXNpY3NfaW5mb3JtZWQiLCAibGFzdCI6IGhpc3RvcnlbLTFdIGlmIGhpc3RvcnkgZWxzZSBOb25lfSwgaW5kZW50PTIpKQogICAgcmV0dXJuIG1vZGVsCg==', 'channel_operator/v2_train.py': 'IiIiUGh5c2ljcy1iYWxhbmNlZCB0cmFpbmluZzsgZXZlcnkgcHJpbnRlZCB0cmFpbmluZyByZWNvcmQgaW5jbHVkZXMgdW5zZWVuIENGRCBlcnJvci4iIiIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSAudjJfcHJvZmlsZXMgaW1wb3J0IHNhbXBsZQpmcm9tIC52Ml9tb2RlbCBpbXBvcnQgU3RyZWFtRGVlcE9OZXQsIHBoeXNpY3MsIG91dGxldAoKCmRlZiBtb25pdG9yaW5nX2Vycm9ycyhtb2RlbCxkYXRhLHRlbnNvcik6CiAgICBlcnJvcnM9W10KICAgIG1vZGVsLmV2YWwoKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgZm9yIGkgaW4gcmFuZ2UobGVuKGRhdGFbJ3Byb2ZpbGVzJ10pKToKICAgICAgICAgICAgcHJlZD1tb2RlbCh0ZW5zb3IoZGF0YVsncHJvZmlsZXMnXVtpOmkrMV0pLHRlbnNvcihkYXRhWydyZXlub2xkcyddW2k6aSsxXSksdGVuc29yKGRhdGFbJ3h5J11bTm9uZV0pKS5jcHUoKS5udW1weSgpWzBdCiAgICAgICAgICAgIHRhcmdldD1kYXRhWydmaWVsZHMnXVtpXTtkaWZmZXJlbmNlPXByZWQtdGFyZ2V0CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoeydjYXNlJzppLCdSZSc6ZmxvYXQoZGF0YVsncmV5bm9sZHMnXVtpXSksCiAgICAgICAgICAgICAgICAndV9yZWxfbDInOmZsb2F0KG5wLmxpbmFsZy5ub3JtKGRpZmZlcmVuY2VbOiwwXSkvbnAubGluYWxnLm5vcm0odGFyZ2V0WzosMF0pKSwKICAgICAgICAgICAgICAgICd2X3JlbF9sMic6ZmxvYXQobnAubGluYWxnLm5vcm0oZGlmZmVyZW5jZVs6LDFdKS9tYXgobnAubGluYWxnLm5vcm0odGFyZ2V0WzosMV0pLDFlLTEyKSksCiAgICAgICAgICAgICAgICAndl9ybXNlJzpmbG9hdChucC5zcXJ0KG5wLm1lYW4oZGlmZmVyZW5jZVs6LDFdKioyKSkpLAogICAgICAgICAgICAgICAgJ3BfcmVsX2wyJzpmbG9hdChucC5saW5hbGcubm9ybShkaWZmZXJlbmNlWzosMl0pL25wLmxpbmFsZy5ub3JtKHRhcmdldFs6LDJdKSksCiAgICAgICAgICAgICAgICAndmVsb2NpdHlfcmVsX2wyJzpmbG9hdChucC5saW5hbGcubm9ybShkaWZmZXJlbmNlWzosOjJdKS9ucC5saW5hbGcubm9ybSh0YXJnZXRbOiw6Ml0pKX0pCiAgICBtb2RlbC50cmFpbigpCiAgICByZXR1cm4geydyb2xlJzondmFsaWRhdGlvbl9vbmx5X25ldmVyX2JhY2twcm9wYWdhdGVkJywKICAgICAgICAgICAgJ3ZlbG9jaXR5X3JlbF9sMl9tYXgnOm1heChjWyd2ZWxvY2l0eV9yZWxfbDInXSBmb3IgYyBpbiBlcnJvcnMpLAogICAgICAgICAgICAncF9yZWxfbDJfbWF4JzptYXgoY1sncF9yZWxfbDInXSBmb3IgYyBpbiBlcnJvcnMpLAogICAgICAgICAgICAndmVsb2NpdHlfcmVsX2wyX21lYW4nOmZsb2F0KG5wLm1lYW4oW2NbJ3ZlbG9jaXR5X3JlbF9sMiddIGZvciBjIGluIGVycm9yc10pKSwKICAgICAgICAgICAgJ3BfcmVsX2wyX21lYW4nOmZsb2F0KG5wLm1lYW4oW2NbJ3BfcmVsX2wyJ10gZm9yIGMgaW4gZXJyb3JzXSkpLAogICAgICAgICAgICAnY2FzZXMnOmVycm9yc30KCgpkZWYgbW9uaXRvcmluZ19waHlzaWNzKG1vZGVsLGRhdGEsdGVuc29yKToKICAgIHJuZz1ucC5yYW5kb20uZGVmYXVsdF9ybmcoODEzNzIpCiAgICByZWNvcmRzPVtdCiAgICBmb3IgaSBpbiByYW5nZShsZW4oZGF0YVsncHJvZmlsZXMnXSkpOgogICAgICAgIHAscj10ZW5zb3IoZGF0YVsncHJvZmlsZXMnXVtpOmkrMV0pLHRlbnNvcihkYXRhWydyZXlub2xkcyddW2k6aSsxXSkKICAgICAgICB4eT10ZW5zb3Iocm5nLnVuaWZvcm0oWy4wMDUsLjAwNV0sWzQuOTk1LC45OTVdLCgxLDI1NiwyKSkpCiAgICAgICAgbXgsbXksbWFzcz1waHlzaWNzKG1vZGVsLHAscix4eSkKICAgICAgICBvbixvdD1vdXRsZXQobW9kZWwscCxyLHRlbnNvcihucC5saW5zcGFjZSguMDAxLC45OTksNjUpW05vbmVdKSkKICAgICAgICByZWNvcmRzLmFwcGVuZCh7J2Nhc2UnOmksJ21vbWVudHVtX3Jtcyc6ZmxvYXQoKG14LnNxdWFyZSgpLm1lYW4oKStteS5zcXVhcmUoKS5tZWFuKCkpLnNxcnQoKS5kZXRhY2goKSksCiAgICAgICAgICAgICAgICAgICAgICAgICdkaXZlcmdlbmNlX3Jtcyc6ZmxvYXQobWFzcy5zcXVhcmUoKS5tZWFuKCkuc3FydCgpLmRldGFjaCgpKSwKICAgICAgICAgICAgICAgICAgICAgICAgJ291dGxldF9ybXMnOmZsb2F0KChvbi5zcXVhcmUoKS5tZWFuKCkrb3Quc3F1YXJlKCkubWVhbigpKS5zcXJ0KCkuZGV0YWNoKCkpfSkKICAgIHJldHVybiB7J21vbWVudHVtX3Jtc19tYXgnOm1heCh4Wydtb21lbnR1bV9ybXMnXSBmb3IgeCBpbiByZWNvcmRzKSwnZGl2ZXJnZW5jZV9ybXNfbWF4JzptYXgoeFsnZGl2ZXJnZW5jZV9ybXMnXSBmb3IgeCBpbiByZWNvcmRzKSwKICAgICAgICAgICAgJ291dGxldF9ybXNfbWF4JzptYXgoeFsnb3V0bGV0X3JtcyddIGZvciB4IGluIHJlY29yZHMpLCdjYXNlcyc6cmVjb3Jkc30KCgpkZWYgdHJhaW4ob3V0cHV0LHN0ZXBzPTIwMDAwLGxvZ19ldmVyeT0yNTAsYmF0Y2g9OCxwb2ludHM9OTYsd2lkdGg9NjQsbGF0ZW50PTMyLGRlcHRoPTMscmVzdW1lPU5vbmUpOgogICAgb3V0cHV0PVBhdGgob3V0cHV0KTtvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBzZWVkPTUxOTc7dG9yY2gubWFudWFsX3NlZWQoc2VlZCk7cm5nPW5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKTt0b3JjaC5zZXRfbnVtX3RocmVhZHMoNCkKICAgIGRldmljZT0nY3VkYScgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICdjcHUnCiAgICB0ZW5zb3I9bGFtYmRhIHg6dG9yY2guYXNfdGVuc29yKHgsZHR5cGU9dG9yY2guZmxvYXQzMixkZXZpY2U9ZGV2aWNlKQogICAgd2l0aCBucC5sb2FkKG91dHB1dC8ndHJhaW4ubnB6JykgYXMgZDogZGF0YT17azpkW2tdLmNvcHkoKSBmb3IgayBpbiBkLmZpbGVzfQogICAgd2l0aCBucC5sb2FkKG91dHB1dC8nbW9uaXRvci5ucHonKSBhcyBkOiBtb25pdG9yPXtrOmRba10uY29weSgpIGZvciBrIGluIGQuZmlsZXN9CiAgICAjIE1vbml0b3JlZCBjb2VmZmljaWVudHMgbmV2ZXIgb2NjdXIgaW4gdGhlIHN1cGVydmlzZWQgdHJhaW5pbmcgc2V0LgogICAgZGlzdGFuY2U9bnAuYWJzKGRhdGFbJ2NvZWZmaWNpZW50cyddWzosTm9uZSw6XS1tb25pdG9yWydjb2VmZmljaWVudHMnXVtOb25lLDosOl0pLm1heCgtMSkKICAgIGlmIGRpc3RhbmNlLm1pbigpPDFlLTg6IHJhaXNlIFZhbHVlRXJyb3IoJ01vbml0b3IgaW5sZXQgbGVha2FnZSBpbnRvIHRyYWluaW5nJykKICAgIHRkPXtrOnRlbnNvcihkYXRhW2tdKSBmb3IgayBpbiBbJ3Byb2ZpbGVzJywncmV5bm9sZHMnLCd4eScsJ2ZpZWxkcyddfQogICAgbW9kZWw9U3RyZWFtRGVlcE9OZXQod2lkdGgsbGF0ZW50LGRlcHRoKS50byhkZXZpY2UpCiAgICBvcHRpbWl6ZXI9dG9yY2gub3B0aW0uQWRhbShtb2RlbC5wYXJhbWV0ZXJzKCksbHI9MWUtMykKICAgIGZpcnN0PTA7ZGF0YV93ZWlnaHQ9MS4KICAgIGlmIHJlc3VtZToKICAgICAgICBzdGF0ZT10b3JjaC5sb2FkKHJlc3VtZSxtYXBfbG9jYXRpb249ZGV2aWNlLHdlaWdodHNfb25seT1GYWxzZSkKICAgICAgICBpZiBzdGF0ZVsnY29uZmlnJ10hPW1vZGVsLmNvbmZpZzpyYWlzZSBWYWx1ZUVycm9yKCdSZXN1bWUgYXJjaGl0ZWN0dXJlIG1pc21hdGNoJykKICAgICAgICBtb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVbJ3N0YXRlX2RpY3QnXSk7b3B0aW1pemVyLmxvYWRfc3RhdGVfZGljdChzdGF0ZVsnb3B0aW1pemVyJ10pCiAgICAgICAgZmlyc3Q9c3RhdGVbJ3N0ZXAnXTtkYXRhX3dlaWdodD1zdGF0ZVsnZGF0YV93ZWlnaHQnXQogICAgICAgIGlmIHN0ZXBzPD1maXJzdDpyYWlzZSBWYWx1ZUVycm9yKCdUYXJnZXQgc3RlcHMgbXVzdCBleGNlZWQgcmVzdW1lZCBzdGVwJykKICAgICAgICBybmcuYml0X2dlbmVyYXRvci5zdGF0ZT1zdGF0ZVsnbnVtcHlfcm5nX3N0YXRlJ107dG9yY2guc2V0X3JuZ19zdGF0ZShzdGF0ZVsndG9yY2hfcm5nX3N0YXRlJ10uY3B1KCkpCiAgICAgICAgaWYgZGV2aWNlPT0nY3VkYScgYW5kIHN0YXRlLmdldCgnY3VkYV9ybmdfc3RhdGVzJykgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRvcmNoLmN1ZGEuc2V0X3JuZ19zdGF0ZV9hbGwoc3RhdGVbJ2N1ZGFfcm5nX3N0YXRlcyddKQogICAgcGFyYW1zPWxpc3QobW9kZWwucGFyYW1ldGVycygpKTtzdGFydD10aW1lLnRpbWUoKTtoaXN0b3J5PVtdCiAgICBsb2c9b3BlbihvdXRwdXQvJ2xlYXJuaW5nX2N1cnZlLmpzb25sJywnYScsYnVmZmVyaW5nPTEpCiAgICBzdGFydF9yZWNvcmQ9eydzdGVwJzpmaXJzdCwnc2Vjb25kcyc6MC4sJ3ZhbGlkYXRpb24nOm1vbml0b3JpbmdfZXJyb3JzKG1vZGVsLG1vbml0b3IsdGVuc29yKSwnZGV2aWNlJzpkZXZpY2UsJ3BoYXNlJzonc3RhcnQnfQogICAgcHJpbnQoanNvbi5kdW1wcyhzdGFydF9yZWNvcmQpLGZsdXNoPVRydWUpO2xvZy53cml0ZShqc29uLmR1bXBzKHN0YXJ0X3JlY29yZCkrJ1xuJykKICAgIGdyYWRfYmFsYW5jZT17fTtwYXNzZWRfY2hlY2tzPTAKICAgIGZvciBzdGVwIGluIHJhbmdlKGZpcnN0KzEsc3RlcHMrMSk6CiAgICAgICAgbW9kZWwudHJhaW4oKTtvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgbHI9bWF4KDFlLTUsMWUtMyouOSoqKHN0ZXAvMTAwMCkpCiAgICAgICAgZm9yIGdyb3VwIGluIG9wdGltaXplci5wYXJhbV9ncm91cHM6Z3JvdXBbJ2xyJ109bHIKICAgICAgICBjaT10b3JjaC5yYW5kaW50KGxlbih0ZFsncHJvZmlsZXMnXSksKGJhdGNoLCksZGV2aWNlPWRldmljZSkKICAgICAgICBmcmVzaCxyZSxjb2VmZj1zYW1wbGUocm5nLGJhdGNoLy8yKQogICAgICAgIHBwPXRvcmNoLmNhdChbdGRbJ3Byb2ZpbGVzJ11bY2lbOmJhdGNoLWJhdGNoLy8yXV0sdGVuc29yKGZyZXNoKV0sMCkKICAgICAgICBycj10b3JjaC5jYXQoW3RkWydyZXlub2xkcyddW2NpWzpiYXRjaC1iYXRjaC8vMl1dLHRlbnNvcihyZSldLDApCiAgICAgICAgeHk9dG9yY2gucmFuZChiYXRjaCxwb2ludHMsMixkZXZpY2U9ZGV2aWNlKQogICAgICAgIGNob29zZT10b3JjaC5yYW5kKGJhdGNoLHBvaW50cyxkZXZpY2U9ZGV2aWNlKTwuNQogICAgICAgIHh5Wy4uLiwwXT01KnRvcmNoLndoZXJlKGNob29zZSx4eVsuLi4sMF0qKjIseHlbLi4uLDBdKQogICAgICAgIG14LG15LG1hc3M9cGh5c2ljcyhtb2RlbCxwcCxycix4eSkKICAgICAgICBtb21lbnR1bT0obXguc3F1YXJlKCkrbXkuc3F1YXJlKCkpLm1lYW4oKQogICAgICAgIG9uLG90PW91dGxldChtb2RlbCxwcCxycix0b3JjaC5yYW5kKGJhdGNoLDMyLGRldmljZT1kZXZpY2UpKQogICAgICAgIG91dGxldF9sb3NzPShvbi5zcXVhcmUoKStvdC5zcXVhcmUoKSkubWVhbigpCiAgICAgICAgcGh5c2ljYWw9bW9tZW50dW0rb3V0bGV0X2xvc3MKICAgICAgICBwaT10b3JjaC5yYW5kaW50KGxlbih0ZFsneHknXSksKDE5MiwpLGRldmljZT1kZXZpY2UpCiAgICAgICAgcHJlZGljdGlvbj1tb2RlbCh0ZFsncHJvZmlsZXMnXVtjaV0sdGRbJ3JleW5vbGRzJ11bY2ldLHRkWyd4eSddW3BpXVtOb25lXS5leHBhbmQoYmF0Y2gsLTEsLTEpKQogICAgICAgIHRhcmdldD10ZFsnZmllbGRzJ11bY2lbOixOb25lXSxwaVtOb25lLDpdXQogICAgICAgIHNjYWxlPXRvcmNoLm9uZXMoYmF0Y2gsMSwzLGRldmljZT1kZXZpY2UpCiAgICAgICAgc2NhbGVbLi4uLDFdPTEwICMgbGF0ZXJhbCB2ZWxvY2l0eSBpcyBzbWFsbCBidXQgcGh5c2ljYWxseSBpbXBvcnRhbnQKICAgICAgICBzY2FsZVsuLi4sMl09dGRbJ3JleW5vbGRzJ11bY2ksTm9uZV0vNjAKICAgICAgICBkYXRhX2xvc3M9KChwcmVkaWN0aW9uLXRhcmdldCkqc2NhbGUpLnNxdWFyZSgpLm1lYW4oKQogICAgICAgIGlmIHN0ZXA9PWZpcnN0KzEgb3Igc3RlcCUyNTA9PTA6CiAgICAgICAgICAgIGRlZiBub3JtKGxvc3MpOgogICAgICAgICAgICAgICAgZ3M9dG9yY2guYXV0b2dyYWQuZ3JhZChsb3NzLHBhcmFtcyxyZXRhaW5fZ3JhcGg9VHJ1ZSxhbGxvd191bnVzZWQ9VHJ1ZSkKICAgICAgICAgICAgICAgIHJldHVybiBzdW0oZy5zcXVhcmUoKS5zdW0oKSBmb3IgZyBpbiBncyBpZiBnIGlzIG5vdCBOb25lKS5zcXJ0KCkuZGV0YWNoKCkKICAgICAgICAgICAgZ3AsZ2Q9bm9ybShwaHlzaWNhbCksbm9ybShkYXRhX2xvc3MpCiAgICAgICAgICAgIHRhcmdldF93ZWlnaHQ9ZmxvYXQoKGdwLyhnZCsxZS0xMikpLmNsYW1wKDFlLTMsMTApKQogICAgICAgICAgICBkYXRhX3dlaWdodD0uOCpkYXRhX3dlaWdodCsuMip0YXJnZXRfd2VpZ2h0CiAgICAgICAgICAgIGdyYWRfYmFsYW5jZT17J3BoeXNpY2FsX2dyYWRpZW50X2wyJzpmbG9hdChncCksJ3Jhd19kYXRhX2dyYWRpZW50X2wyJzpmbG9hdChnZCksJ3dlaWdodGVkX2RhdGFfZ3JhZGllbnRfbDInOmZsb2F0KGdkKSpkYXRhX3dlaWdodH0KICAgICAgICBsb3NzPXBoeXNpY2FsK2RhdGFfd2VpZ2h0KmRhdGFfbG9zcwogICAgICAgIGlmIG5vdCB0b3JjaC5pc2Zpbml0ZShsb3NzKTpyYWlzZSBSdW50aW1lRXJyb3IoZidOb25maW5pdGUgbG9zcyBhdCB7c3RlcH0nKQogICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgIGdyYWRfbm9ybT1mbG9hdCh0b3JjaC5ubi51dGlscy5jbGlwX2dyYWRfbm9ybV8ocGFyYW1zLDEwLikpCiAgICAgICAgb3B0aW1pemVyLnN0ZXAoKQogICAgICAgIGlmIHN0ZXAlbG9nX2V2ZXJ5PT0wIG9yIHN0ZXA9PXN0ZXBzOgogICAgICAgICAgICByZWNvcmQ9eydzdGVwJzpzdGVwLCdzZWNvbmRzJzp0aW1lLnRpbWUoKS1zdGFydCwnbG9zcyc6ZmxvYXQobG9zcy5kZXRhY2goKSksJ21vbWVudHVtX21zZSc6ZmxvYXQobW9tZW50dW0uZGV0YWNoKCkpLAogICAgICAgICAgICAgICAgICAgICdvdXRsZXRfbXNlJzpmbG9hdChvdXRsZXRfbG9zcy5kZXRhY2goKSksJ2RhdGFfbXNlJzpmbG9hdChkYXRhX2xvc3MuZGV0YWNoKCkpLCdkYXRhX3dlaWdodCc6ZGF0YV93ZWlnaHQsCiAgICAgICAgICAgICAgICAgICAgJ2RpdmVyZ2VuY2Vfcm1zJzpmbG9hdChtYXNzLnNxdWFyZSgpLm1lYW4oKS5zcXJ0KCkuZGV0YWNoKCkpLCdncmFkaWVudF9ub3JtX2JlZm9yZV9jbGlwJzpncmFkX25vcm0sCiAgICAgICAgICAgICAgICAgICAgJ2xyJzpsciwnZ3JhZGllbnRfYmFsYW5jZSc6Z3JhZF9iYWxhbmNlLCd2YWxpZGF0aW9uJzptb25pdG9yaW5nX2Vycm9ycyhtb2RlbCxtb25pdG9yLHRlbnNvcil9CiAgICAgICAgICAgIHByaW50KGpzb24uZHVtcHMocmVjb3JkKSxmbHVzaD1UcnVlKTtsb2cud3JpdGUoanNvbi5kdW1wcyhyZWNvcmQpKydcbicpO2hpc3RvcnkuYXBwZW5kKHJlY29yZCkKICAgICAgICAgICAgc3RvcD1GYWxzZQogICAgICAgICAgICBpZiBzdGVwJTEwMDA9PTAgb3Igc3RlcD09c3RlcHM6CiAgICAgICAgICAgICAgICBkaWFnbm9zdGljPW1vbml0b3JpbmdfcGh5c2ljcyhtb2RlbCxtb25pdG9yLHRlbnNvcikKICAgICAgICAgICAgICAgIHJlY29yZFsndmFsaWRhdGlvbl9waHlzaWNzJ109ZGlhZ25vc3RpYwogICAgICAgICAgICAgICAgcGFzc2VkPShyZWNvcmRbJ3ZhbGlkYXRpb24nXVsndmVsb2NpdHlfcmVsX2wyX21heCddPC4wNSBhbmQgcmVjb3JkWyd2YWxpZGF0aW9uJ11bJ3BfcmVsX2wyX21heCddPC4xCiAgICAgICAgICAgICAgICAgICAgICAgIGFuZCBkaWFnbm9zdGljWydtb21lbnR1bV9ybXNfbWF4J108LjA1IGFuZCBkaWFnbm9zdGljWydvdXRsZXRfcm1zX21heCddPC4xKQogICAgICAgICAgICAgICAgcGFzc2VkX2NoZWNrcz1wYXNzZWRfY2hlY2tzKzEgaWYgcGFzc2VkIGVsc2UgMAogICAgICAgICAgICAgICAgc3RvcD1zdGVwPj00MDAwIGFuZCBwYXNzZWRfY2hlY2tzPj0yCiAgICAgICAgICAgICAgICAjIFJlcGVhdCBtZXRyaWNzIGluIHRoaXMgZGlhZ25vc3RpYyBsaW5lOiBldmVyeSB0cmFpbmluZyBwcmludCBpbmNsdWRlcyBDRkQgZXJyb3JzLgogICAgICAgICAgICAgICAgZGlhZ25vc3RpY19yZWNvcmQ9eydzdGVwJzpzdGVwLCdwaGFzZSc6J3BoeXNpY3NfZGlhZ25vc3RpYycsJ3ZhbGlkYXRpb24nOnJlY29yZFsndmFsaWRhdGlvbiddLCd2YWxpZGF0aW9uX3BoeXNpY3MnOmRpYWdub3N0aWMsJ2Vhcmx5X3N0b3AnOnN0b3B9CiAgICAgICAgICAgICAgICBwcmludChqc29uLmR1bXBzKGRpYWdub3N0aWNfcmVjb3JkKSxmbHVzaD1UcnVlKTtsb2cud3JpdGUoanNvbi5kdW1wcyhkaWFnbm9zdGljX3JlY29yZCkrJ1xuJykKICAgICAgICAgICAgdGVtcG9yYXJ5PW91dHB1dC8ndjJfbW9kZWwudG1wJwogICAgICAgICAgICB0b3JjaC5zYXZlKHsnc3RhdGVfZGljdCc6bW9kZWwuc3RhdGVfZGljdCgpLCdjb25maWcnOm1vZGVsLmNvbmZpZywnb3B0aW1pemVyJzpvcHRpbWl6ZXIuc3RhdGVfZGljdCgpLCdzdGVwJzpzdGVwLAogICAgICAgICAgICAgICAgJ21vZGVsX3R5cGUnOidzdHJlYW1fZGVlcG9uZXRfdjInLCdkYXRhX3dlaWdodCc6ZGF0YV93ZWlnaHQsJ251bXB5X3JuZ19zdGF0ZSc6cm5nLmJpdF9nZW5lcmF0b3Iuc3RhdGUsCiAgICAgICAgICAgICAgICAndG9yY2hfcm5nX3N0YXRlJzp0b3JjaC5nZXRfcm5nX3N0YXRlKCksJ2N1ZGFfcm5nX3N0YXRlcyc6dG9yY2guY3VkYS5nZXRfcm5nX3N0YXRlX2FsbCgpIGlmIGRldmljZT09J2N1ZGEnIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgICdzZWVkJzpzZWVkLCd0b3JjaF92ZXJzaW9uJzpzdHIodG9yY2guX192ZXJzaW9uX18pLCd2YWxpZGF0aW9uJzpyZWNvcmRbJ3ZhbGlkYXRpb24nXX0sdGVtcG9yYXJ5KQogICAgICAgICAgICB0ZW1wb3JhcnkucmVwbGFjZShvdXRwdXQvJ3YyX21vZGVsLnB0JykKICAgICAgICAgICAgaWYgc3RvcDpicmVhawogICAgbG9nLmNsb3NlKCkKICAgIChvdXRwdXQvJ3YyX3RyYWluaW5nX3N1bW1hcnkuanNvbicpLndyaXRlX3RleHQoanNvbi5kdW1wcyh7J2RldmljZSc6ZGV2aWNlLCdncHUnOnRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKCkgaWYgZGV2aWNlPT0nY3VkYScgZWxzZSBOb25lLAogICAgICAgICdwbGFubmVkX3N0ZXBzJzpzdGVwcywnc3RlcHMnOnN0ZXAsJ3NlY29uZHMnOnRpbWUudGltZSgpLXN0YXJ0LCdmaW5hbCc6aGlzdG9yeVstMV0gaWYgaGlzdG9yeSBlbHNlIHN0YXJ0X3JlY29yZH0saW5kZW50PTIpKQogICAgcmV0dXJuIG1vZGVsCg==', 'channel_operator/v2_data.py': 'aW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBudW1weSBhcyBucApmcm9tIC52Ml9wcm9maWxlcyBpbXBvcnQgc2FtcGxlLG1vbml0b3JpbmdfY29uZGl0aW9ucyxpbmxldF92YWx1ZXMKZnJvbSAucmVmZXJlbmNlIGltcG9ydCBzb2x2ZV9jaGFubmVsCgoKZGVmIHByZXBhcmUob3V0cHV0LHRyYWluX2Nhc2VzPTEyOCk6CiAgICBvdXRwdXQ9UGF0aChvdXRwdXQpO291dHB1dC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIG1ldGFkYXRhPXsndHJhaW5fc2VlZCc6OTcwMzEsJ21vbml0b3JfaXNfdHJhaW5pbmcnOkZhbHNlLCdtb25pdG9yX3JvbGUnOid2YWxpZGF0aW9uLCByZXBlYXRlZGx5IGluc3BlY3RlZDsgbm90IGEgYmxpbmQgdGVzdCcsJ3NwbGl0cyc6e319CiAgICBmb3IgbmFtZSxjb25kaXRpb25zIGluIFsoJ3RyYWluJyxzYW1wbGUobnAucmFuZG9tLmRlZmF1bHRfcm5nKDk3MDMxKSx0cmFpbl9jYXNlcykpLCgnbW9uaXRvcicsbW9uaXRvcmluZ19jb25kaXRpb25zKCkpXToKICAgICAgICBpZiAob3V0cHV0L2Yne25hbWV9Lm5weicpLmV4aXN0cygpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnUmVmdXNpbmcgdG8gb3ZlcndyaXRlIGV4aXN0aW5nIHtuYW1lfSBkYXRhJykKICAgICAgICBwcm9maWxlcyxyZXlub2xkcyxjb2VmZj1jb25kaXRpb25zCiAgICAgICAgeCx5PW5wLm1lc2hncmlkKG5wLmxpbnNwYWNlKC4wMSw0Ljk5LDQ5KSxucC5saW5zcGFjZSguMDA1LC45OTUsMjUpLGluZGV4aW5nPSdpaicpCiAgICAgICAgeHk9bnAuc3RhY2soW3gseV0sLTEpLnJlc2hhcGUoLTEsMikKICAgICAgICBmaWVsZHMscmVjb3Jkcz1bXSxbXQogICAgICAgIGZvciBpLChwLHJlLGEpIGluIGVudW1lcmF0ZSh6aXAocHJvZmlsZXMscmV5bm9sZHMsY29lZmYpKToKICAgICAgICAgICAgaW5sZXQ9bGFtYmRhIHksYT1hOmlubGV0X3ZhbHVlcyhhLHkpCiAgICAgICAgICAgIG54LG55PSg0MCwyMCkgaWYgbmFtZT09J3RyYWluJyBlbHNlICg4MCwzMikKICAgICAgICAgICAgZm4saW5mbz1zb2x2ZV9jaGFubmVsKHAscmUsbng9bngsbnk9bnksaW5sZXRfZnVuY3Rpb249aW5sZXQpCiAgICAgICAgICAgIHZhbHVlcz1mbih4eSk7ZmllbGRzLmFwcGVuZCh2YWx1ZXMpCiAgICAgICAgICAgIGlmIG5hbWU9PSdtb25pdG9yJzoKICAgICAgICAgICAgICAgIGNvYXJzZSxfPXNvbHZlX2NoYW5uZWwocCxyZSxueD01NixueT0yNCxpbmxldF9mdW5jdGlvbj1pbmxldCkKICAgICAgICAgICAgICAgIGRpZmY9Y29hcnNlKHh5KS12YWx1ZXMKICAgICAgICAgICAgICAgIGluZm9bJ21lc2hfdmVsb2NpdHlfcmVsYXRpdmVfbDInXT1mbG9hdChucC5saW5hbGcubm9ybShkaWZmWzosOjJdKS9ucC5saW5hbGcubm9ybSh2YWx1ZXNbOiw6Ml0pKQogICAgICAgICAgICAgICAgaW5mb1snbWVzaF9wcmVzc3VyZV9yZWxhdGl2ZV9sMiddPWZsb2F0KG5wLmxpbmFsZy5ub3JtKGRpZmZbOiwyXSkvbnAubGluYWxnLm5vcm0odmFsdWVzWzosMl0pKQogICAgICAgICAgICAgICAgaWYgaW5mb1snbWVzaF92ZWxvY2l0eV9yZWxhdGl2ZV9sMiddPi4wMSBvciBpbmZvWydtZXNoX3ByZXNzdXJlX3JlbGF0aXZlX2wyJ10+LjAyOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmJ01vbml0b3IgcmVmZXJlbmNlIG5vdCBtZXNoIGNvbnZlcmdlZDoge2luZm99JykKICAgICAgICAgICAgcmVjb3Jkcy5hcHBlbmQoaW5mbykKICAgICAgICAgICAgcHJpbnQoZidSRUZFUkVOQ0Uge25hbWV9IHtpKzF9L3tsZW4ocHJvZmlsZXMpfSBSZT17cmU6LjFmfScsZmx1c2g9VHJ1ZSkKICAgICAgICBucC5zYXZlel9jb21wcmVzc2VkKG91dHB1dC9mJ3tuYW1lfS5ucHonLHByb2ZpbGVzPXByb2ZpbGVzLHJleW5vbGRzPXJleW5vbGRzLGNvZWZmaWNpZW50cz1jb2VmZix4eT14eSxmaWVsZHM9bnAuYXJyYXkoZmllbGRzKSkKICAgICAgICBtZXRhZGF0YVsnc3BsaXRzJ11bbmFtZV09cmVjb3JkcwogICAgdHJhaW49bnAubG9hZChvdXRwdXQvJ3RyYWluLm5weicpO21vbml0b3I9bnAubG9hZChvdXRwdXQvJ21vbml0b3IubnB6JykKICAgIGZvciBhIGluIG1vbml0b3JbJ2NvZWZmaWNpZW50cyddOgogICAgICAgIGlmIG5wLmFueShucC5tYXgobnAuYWJzKHRyYWluWydjb2VmZmljaWVudHMnXS1hKSxheGlzPTEpPDFlLTgpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ0EgbW9uaXRvcmVkIGlubGV0IGxlYWtlZCBpbnRvIHRyYWluaW5nJykKICAgIChvdXRwdXQvJ3JlZmVyZW5jZV9tZXRhZGF0YS5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKG1ldGFkYXRhLGluZGVudD0yKSkK', 'channel_operator/v2_profiles.py': 'IiIiQW5hbHl0aWMgc21vb3RoIGZvdXItbW9kZSBpbmxldHM7IG5vIHNwYXRpYWwgaW50ZXJwb2xhdGlvbiBpbnNpZGUgUERFIGRlcml2YXRpdmVzLiIiIgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSAucHJvZmlsZXMgaW1wb3J0IFkKCk1PREVTID0gbnAuYXJhbmdlKDEsIDUpCklOVEVHUkFMUyA9IC02KigxKygtMS4pKipNT0RFUykvKG5wLnBpKk1PREVTKSoqMgpBID0gNipZWzosIE5vbmVdKigxLVlbOiwgTm9uZV0pKm5wLmNvcyhucC5waSpZWzosIE5vbmVdKm5wLmFyYW5nZSg1KSkKUFJPSkVDVCA9IG5wLmxpbmFsZy5waW52KEEpCgoKZGVmIGNvZWZmaWNpZW50c190b19iKGEpOgogICAgYSA9IG5wLmFzYXJyYXkoYSwgZHR5cGU9ZmxvYXQpCiAgICBub3JtID0gMSthQElOVEVHUkFMUwogICAgcmV0dXJuIG5wLmNvbmNhdGVuYXRlKFtucC5vbmVzKCgqYS5zaGFwZVs6LTFdLCAxKSksIGFdLCAtMSkvbm9ybVsuLi4sIE5vbmVdCgoKZGVmIGlubGV0X3ZhbHVlcyhhLCB5PVkpOgogICAgYiA9IGNvZWZmaWNpZW50c190b19iKGEpCiAgICB5ID0gbnAuYXNhcnJheSh5KQogICAgcmV0dXJuIChiQG5wLmNvcyhucC5hcmFuZ2UoNSlbOiwgTm9uZV0qbnAucGkqeS5yYXZlbCgpKSkucmVzaGFwZSgoKmIuc2hhcGVbOi0xXSwgKnkuc2hhcGUpKSo2KnkqKDEteSkKCgpkZWYgc2FtcGxlKHJuZywgbiwgcmFkaXVzPS44KToKICAgICIiIkNvdmVyIG1peHR1cmVzIEFORCBheGVzOyBtb25pdG9yIHByb2ZpbGVzIGF0IGV4YWN0IHxhX2t8PS42IGFyZSBleGNsdWRlZC4iIiIKICAgIGEgPSBybmcubm9ybWFsKHNpemU9KG4sIDQpKQogICAgYSAqPSBybmcudW5pZm9ybSguMDUsIHJhZGl1cywgKG4sIDEpKS9ucC5hYnMoYSkuc3VtKDEsIGtlZXBkaW1zPVRydWUpCiAgICBraW5kID0gcm5nLnJhbmRvbShuKQogICAgZm9yIGkgaW4gcmFuZ2Uobik6CiAgICAgICAgaWYga2luZFtpXSA8IC4zNToKICAgICAgICAgICAgYVtpXSA9IDAKICAgICAgICAgICAgayA9IHJuZy5pbnRlZ2Vycyg0KQogICAgICAgICAgICB2YWx1ZSA9IHJuZy51bmlmb3JtKC1yYWRpdXMsIHJhZGl1cykKICAgICAgICAgICAgaWYgYWJzKGFicyh2YWx1ZSktLjYpPDFlLTQ6CiAgICAgICAgICAgICAgICB2YWx1ZSAqPSAuOTkKICAgICAgICAgICAgYVtpLGtdID0gdmFsdWUKICAgICAgICBlbGlmIGtpbmRbaV0gPCAuNTU6CiAgICAgICAgICAgIGtlZXAgPSBybmcuY2hvaWNlKDQsIDIsIHJlcGxhY2U9RmFsc2UpCiAgICAgICAgICAgIG1hc2sgPSBucC56ZXJvcyg0KTsgbWFza1trZWVwXSA9IDEKICAgICAgICAgICAgYVtpXSAqPSBtYXNrCiAgICAgICAgICAgIGFbaV0gKj0gcm5nLnVuaWZvcm0oLjQsIHJhZGl1cykvbWF4KG5wLmFicyhhW2ldKS5zdW0oKSwxZS0xMikKICAgIHJlID0gbnAuZXhwKHJuZy51bmlmb3JtKG5wLmxvZygxMCksIG5wLmxvZygxMDApLCBuKSkKICAgIGVkZ2UgPSBybmcucmFuZG9tKG4pPC4yNQogICAgcmVbZWRnZV0gPSBybmcuY2hvaWNlKFsxMC4sMTAwLl0sc2l6ZT1lZGdlLnN1bSgpKQogICAgcmV0dXJuIGlubGV0X3ZhbHVlcyhhKSwgcmUsIGEKCgpkZWYgbW9uaXRvcmluZ19jb25kaXRpb25zKCk6CiAgICBhLCByZXMgPSBbXSwgW10KICAgIGZvciBtb2RlIGluIHJhbmdlKDQpOgogICAgICAgIGZvciBzaWduIGluIFstMSwxXToKICAgICAgICAgICAgZm9yIHJlIGluIFsxMC4sMTAwLl06CiAgICAgICAgICAgICAgICBjPW5wLnplcm9zKDQpOyBjW21vZGVdPXNpZ24qLjYKICAgICAgICAgICAgICAgIGEuYXBwZW5kKGMpOyByZXMuYXBwZW5kKHJlKQogICAgYT1ucC5hcnJheShhKQogICAgcmV0dXJuIGlubGV0X3ZhbHVlcyhhKSwgbnAuYXJyYXkocmVzKSwgYQo=', 'channel_operator/v2_jax.py': 'IiIiSklULWNvbXBpbGVkIGltcGxlbWVudGF0aW9uIG9mIHRoZSBTQU1FIHN0cmVhbWZ1bmN0aW9uIG1vZGVsIGFuZCBOUyBlcXVhdGlvbnMuCgpQYXJhbWV0ZXIga2V5cyBkZWxpYmVyYXRlbHkgbWF0Y2ggdjJfbW9kZWwucHkgZm9yIG51bWVyaWNhbCBwYXJpdHkgYW5kIGV4cG9ydC4KIiIiCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgamF4CmltcG9ydCBqYXgubnVtcHkgYXMgam5wCmZyb20gLnYyX3Byb2ZpbGVzIGltcG9ydCBQUk9KRUNULElOVEVHUkFMUwpmcm9tIC5wcm9maWxlcyBpbXBvcnQgWQoKamF4LmNvbmZpZy51cGRhdGUoJ2pheF9kZWZhdWx0X21hdG11bF9wcmVjaXNpb24nLCdoaWdoZXN0JykKCgpkZWYgaW5pdGlhbGl6ZSh3aWR0aD02NCxsYXRlbnQ9MzIsZGVwdGg9MyxzZWVkPTUxOTcscHJlc3N1cmVfbW9kZT0ndmlzY291cycpOgogICAgcm5nPW5wLnJhbmRvbS5kZWZhdWx0X3JuZyhzZWVkKTtwYXJhbXM9e30KICAgIGZvciBwcmVmaXgsaW5wdXRzIGluIFsoJ2JyYW5jaCcsMzQpLCgndHJ1bmsnLDE0KV06CiAgICAgICAgc2hhcGVzPVsoJ3UnLGlucHV0cyx3aWR0aCksKCd2JyxpbnB1dHMsd2lkdGgpXQogICAgICAgIHNoYXBlcyArPSBbKGYnZ2F0ZXMue2t9JyxpbnB1dHMgaWYgaz09MCBlbHNlIHdpZHRoLHdpZHRoKSBmb3IgayBpbiByYW5nZShkZXB0aCldCiAgICAgICAgc2hhcGVzICs9IFsoJ291dHB1dCcsd2lkdGgsMipsYXRlbnQpXQogICAgICAgIGZvciBuYW1lLGRpLGRvIGluIHNoYXBlczoKICAgICAgICAgICAgc3RkPTFlLTMgaWYgcHJlZml4PT0ndHJ1bmsnIGFuZCBuYW1lPT0nb3V0cHV0JyBlbHNlIG5wLnNxcnQoMi8oZGkrZG8pKQogICAgICAgICAgICBwYXJhbXNbZid7cHJlZml4fS57bmFtZX0ud2VpZ2h0J109am5wLmFzYXJyYXkocm5nLm5vcm1hbCgwLHN0ZCwoZG8sZGkpKSxkdHlwZT1qbnAuZmxvYXQzMikKICAgICAgICAgICAgcGFyYW1zW2Yne3ByZWZpeH0ue25hbWV9LmJpYXMnXT1qbnAuemVyb3MoZG8sZHR5cGU9am5wLmZsb2F0MzIpCiAgICByZXR1cm4gcGFyYW1zCgoKZGVmIG1ha2VfZnVuY3Rpb25zKGxhdGVudD0zMixkZXB0aD0zLHByZXNzdXJlX21vZGU9J3Zpc2NvdXMnKToKICAgIHByb2plY3Rpb249am5wLmFzYXJyYXkoUFJPSkVDVCxkdHlwZT1qbnAuZmxvYXQzMikKICAgIGludGVncmFscz1qbnAuYXNhcnJheShJTlRFR1JBTFMsZHR5cGU9am5wLmZsb2F0MzIpCiAgICBiYXNlbGluZT1qbnAuYXNhcnJheSg2KlkqKDEtWSksZHR5cGU9am5wLmZsb2F0MzIpCiAgICBhbHBoYT1qbnAucGkqam5wLmFyYW5nZSgxLDUsZHR5cGU9am5wLmZsb2F0MzIpCiAgICBtb2Rlcz1qbnAuYXJhbmdlKDEsNSxkdHlwZT1qbnAuZmxvYXQzMikKICAgIGRlZiBkZW5zZShwYXJhbXMscHJlZml4LHgpOgogICAgICAgIHJldHVybiB4QHBhcmFtc1twcmVmaXgrJy53ZWlnaHQnXS5UK3BhcmFtc1twcmVmaXgrJy5iaWFzJ10KICAgIGRlZiBtbHAocGFyYW1zLHByZWZpeCx4KToKICAgICAgICB1PWpucC50YW5oKGRlbnNlKHBhcmFtcyxwcmVmaXgrJy51Jyx4KSk7dj1qbnAudGFuaChkZW5zZShwYXJhbXMscHJlZml4KycudicseCkpO2g9eAogICAgICAgIGZvciBpIGluIHJhbmdlKGRlcHRoKToKICAgICAgICAgICAgej1qbnAudGFuaChkZW5zZShwYXJhbXMscHJlZml4K2YnLmdhdGVzLntpfScsaCkpO2g9eip1KygxLXopKnYKICAgICAgICByZXR1cm4gZGVuc2UocGFyYW1zLHByZWZpeCsnLm91dHB1dCcsaCkKICAgIGRlZiBjb250ZXh0KHBhcmFtcyxwcm9maWxlLHJlKToKICAgICAgICBjb25kaXRpb249am5wLmNvbmNhdGVuYXRlKFtwcm9maWxlLWJhc2VsaW5lLGpucC5hcnJheShbam5wLmxvZyhyZS8xMCkvam5wLmxvZygxMC4pXSldKQogICAgICAgIGI9cHJvZmlsZUBwcm9qZWN0aW9uLlQ7Yj1iLyhiWzBdK2pucC5zdW0oYlsxOl0qaW50ZWdyYWxzKSkKICAgICAgICByZXR1cm4gbWxwKHBhcmFtcywnYnJhbmNoJyxjb25kaXRpb24pLnJlc2hhcGUoMixsYXRlbnQpLGIscmUKICAgIGNvbnRleHRzPWpheC52bWFwKGNvbnRleHQsaW5fYXhlcz0oTm9uZSwwLDApKQogICAgZGVmIHBvdGVudGlhbHMocGFyYW1zLGN0eCx4eSk6CiAgICAgICAgYnJhbmNoLGIscmU9Y3R4O3gseT14eQogICAgICAgIHBoYXNlPXkqYWxwaGEKICAgICAgICBmZWF0dXJlcz1qbnAuY29uY2F0ZW5hdGUoW2pucC5hcnJheShbMip4LzUtMSwyKnktMV0pLGpucC5zaW4ocGhhc2UpLGpucC5jb3MocGhhc2UpLAogICAgICAgICAgICAgICAgICAgIGpucC5hcnJheShbam5wLmxvZzFwKDIwKngpL2pucC5sb2coMTAxLiksam5wLmV4cCgteCksam5wLmV4cCgtNCp4KSxqbnAuZXhwKC0xNip4KV0pXSkKICAgICAgICByYXc9am5wLnN1bShicmFuY2gqbWxwKHBhcmFtcywndHJ1bmsnLGZlYXR1cmVzKS5yZXNoYXBlKDIsbGF0ZW50KSwtMSkvam5wLnNxcnQoZmxvYXQobGF0ZW50KSkKICAgICAgICBiYXNlPTMqeSp5LTIqeSoqMwogICAgICAgIGlrPTYqKCh5LXkqeSkqam5wLnNpbihwaGFzZSkvYWxwaGErKDEtMip5KSpqbnAuY29zKHBoYXNlKS9hbHBoYSoqMisyKmpucC5zaW4ocGhhc2UpL2FscGhhKiozLTEvYWxwaGEqKjIpCiAgICAgICAgZWxsPS4xKy4wNSpyZS9tb2RlcyoqMgogICAgICAgIHBzaT1iYXNlK2pucC5zdW0oYlsxOl0qKGlrLWJhc2UqaW50ZWdyYWxzKSpqbnAuZXhwKC0oeC9lbGwpKioyKSkKICAgICAgICBwc2k9cHNpKzE2KnkqeSooMS15KSoqMiooMS1qbnAuZXhwKC14Ly4yKSkqKjIqcmF3WzBdCiAgICAgICAgcHJlc3N1cmU9MTIvcmUqKDUteCkrKDEyL3JlIGlmIHByZXNzdXJlX21vZGU9PSd2aXNjb3VzJyBlbHNlIDEuKSpyYXdbMV0KICAgICAgICByZXR1cm4gcHNpLHByZXNzdXJlCiAgICBkZWYgcG9pbnRfZmllbGRzKHBhcmFtcyxjdHgseHkpOgogICAgICAgIGc9amF4LmdyYWQobGFtYmRhIHo6cG90ZW50aWFscyhwYXJhbXMsY3R4LHopWzBdKSh4eSkKICAgICAgICByZXR1cm4gam5wLmFycmF5KFtnWzFdLC1nWzBdLHBvdGVudGlhbHMocGFyYW1zLGN0eCx4eSlbMV1dKQogICAgamFjPWpheC5qYWNmd2QocG9pbnRfZmllbGRzLGFyZ251bXM9MikKICAgIGhlc3NpYW49amF4LmphY2Z3ZChqYWMsYXJnbnVtcz0yKQogICAgZGVmIHBvaW50X3BoeXNpY3MocGFyYW1zLGN0eCx4eSk6CiAgICAgICAgdSx2LHA9cG9pbnRfZmllbGRzKHBhcmFtcyxjdHgseHkpO2c9amFjKHBhcmFtcyxjdHgseHkpO2g9aGVzc2lhbihwYXJhbXMsY3R4LHh5KQogICAgICAgIG14PXUqZ1swLDBdK3YqZ1swLDFdK2dbMiwwXS0oaFswLDAsMF0raFswLDEsMV0pL2N0eFsyXQogICAgICAgIG15PXUqZ1sxLDBdK3YqZ1sxLDFdK2dbMiwxXS0oaFsxLDAsMF0raFsxLDEsMV0pL2N0eFsyXQogICAgICAgIHJldHVybiBqbnAuYXJyYXkoW214LG15LGdbMCwwXStnWzEsMV1dKQogICAgZGVmIHBvaW50X291dGxldChwYXJhbXMsY3R4LHkpOgogICAgICAgIHh5PWpucC5hcnJheShbNS4seV0pO2Y9cG9pbnRfZmllbGRzKHBhcmFtcyxjdHgseHkpO2c9amFjKHBhcmFtcyxjdHgseHkpCiAgICAgICAgcmV0dXJuIGpucC5hcnJheShbZ1swLDBdLWN0eFsyXSpmWzJdLGdbMSwwXV0pCiAgICBmaWVsZF9tYXA9amF4LnZtYXAoamF4LnZtYXAocG9pbnRfZmllbGRzLGluX2F4ZXM9KE5vbmUsTm9uZSwwKSksaW5fYXhlcz0oTm9uZSwwLDApKQogICAgcGh5c2ljc19tYXA9amF4LnZtYXAoamF4LnZtYXAocG9pbnRfcGh5c2ljcyxpbl9heGVzPShOb25lLE5vbmUsMCkpLGluX2F4ZXM9KE5vbmUsMCwwKSkKICAgIG91dGxldF9tYXA9amF4LnZtYXAoamF4LnZtYXAocG9pbnRfb3V0bGV0LGluX2F4ZXM9KE5vbmUsTm9uZSwwKSksaW5fYXhlcz0oTm9uZSwwLDApKQogICAgZGVmIGZpZWxkcyhwYXJhbXMscHJvZmlsZXMscmUseHkpOnJldHVybiBmaWVsZF9tYXAocGFyYW1zLGNvbnRleHRzKHBhcmFtcyxwcm9maWxlcyxyZSkseHkpCiAgICBkZWYgcGh5c2ljcyhwYXJhbXMscHJvZmlsZXMscmUseHkpOnJldHVybiBwaHlzaWNzX21hcChwYXJhbXMsY29udGV4dHMocGFyYW1zLHByb2ZpbGVzLHJlKSx4eSkKICAgIGRlZiBvdXQocGFyYW1zLHByb2ZpbGVzLHJlLHkpOnJldHVybiBvdXRsZXRfbWFwKHBhcmFtcyxjb250ZXh0cyhwYXJhbXMscHJvZmlsZXMscmUpLHkpCiAgICBkZWYgdGVybXMocGFyYW1zLGJhdGNoKToKICAgICAgICBjdHg9Y29udGV4dHMocGFyYW1zLGJhdGNoWydwcm9maWxlcyddLGJhdGNoWydyZXlub2xkcyddKQogICAgICAgIHJlc2lkdWFsPXBoeXNpY3NfbWFwKHBhcmFtcyxjdHgsYmF0Y2hbJ3h5J10pCiAgICAgICAgbW9tZW50dW09am5wLm1lYW4oam5wLnN1bShyZXNpZHVhbFsuLi4sOjJdKioyLC0xKSkKICAgICAgICBib3VuZGFyeT1qbnAubWVhbihqbnAuc3VtKG91dGxldF9tYXAocGFyYW1zLGN0eCxiYXRjaFsnb3V0X3knXSkqKjIsLTEpKQogICAgICAgIHByZWQ9ZmllbGRzKHBhcmFtcyxiYXRjaFsnZGF0YV9wcm9maWxlcyddLGJhdGNoWydkYXRhX3JleW5vbGRzJ10sYmF0Y2hbJ2RhdGFfeHknXSkKICAgICAgICBpZiAnZGF0YV9zY2FsZScgaW4gYmF0Y2g6CiAgICAgICAgICAgIHNjYWxlPWJhdGNoWydkYXRhX3NjYWxlJ11bOixOb25lLDpdCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc2NhbGU9am5wLnN0YWNrKFtqbnAub25lc19saWtlKGJhdGNoWydkYXRhX3JleW5vbGRzJ10pLGpucC5mdWxsX2xpa2UoYmF0Y2hbJ2RhdGFfcmV5bm9sZHMnXSwxMC4pLGJhdGNoWydkYXRhX3JleW5vbGRzJ10vNjBdLC0xKVs6LE5vbmUsOl0KICAgICAgICBkYXRhPWpucC5tZWFuKCgocHJlZC1iYXRjaFsndGFyZ2V0J10pKnNjYWxlKSoqMikKICAgICAgICByZXR1cm4gbW9tZW50dW0sYm91bmRhcnksZGF0YSxqbnAuc3FydChqbnAubWVhbihyZXNpZHVhbFsuLi4sMl0qKjIpKQogICAgZGVmIGxvc3MocGFyYW1zLGJhdGNoLHdlaWdodCk6CiAgICAgICAgbW9tZW50dW0sYm91bmRhcnksZGF0YSxtYXNzPXRlcm1zKHBhcmFtcyxiYXRjaCkKICAgICAgICByZXR1cm4gbW9tZW50dW0rYm91bmRhcnkrd2VpZ2h0KmRhdGEsam5wLmFycmF5KFttb21lbnR1bSxib3VuZGFyeSxkYXRhLG1hc3NdKQogICAgZGVmIG5vcm0odHJlZSk6cmV0dXJuIGpucC5zcXJ0KHN1bShqbnAuc3VtKHYqdikgZm9yIHYgaW4gamF4LnRyZWVfdXRpbC50cmVlX2xlYXZlcyh0cmVlKSkpCiAgICBkZWYgc3RlcChwYXJhbXMsbSx2LGNvdW50LHdlaWdodCxiYXRjaCk6CiAgICAgICAgKHZhbHVlLGF1eCksZ3JhZD1qYXgudmFsdWVfYW5kX2dyYWQobG9zcyxoYXNfYXV4PVRydWUpKHBhcmFtcyxiYXRjaCx3ZWlnaHQpCiAgICAgICAgZ249bm9ybShncmFkKTtmYWN0b3I9am5wLm1pbmltdW0oMS4sMTAvKGduKzFlLTEyKSkKICAgICAgICBncmFkPWpheC50cmVlX3V0aWwudHJlZV9tYXAobGFtYmRhIGc6ZypmYWN0b3IsZ3JhZCkKICAgICAgICBtPWpheC50cmVlX3V0aWwudHJlZV9tYXAobGFtYmRhIG9sZCxnOi45Km9sZCsuMSpnLG0sZ3JhZCkKICAgICAgICB2PWpheC50cmVlX3V0aWwudHJlZV9tYXAobGFtYmRhIG9sZCxnOi45OTkqb2xkKy4wMDEqZypnLHYsZ3JhZCkKICAgICAgICBscj1iYXRjaC5nZXQoJ2xlYXJuaW5nX3JhdGUnLGpucC5tYXhpbXVtKDFlLTUsMWUtMyouOSoqKGNvdW50LzEwMDApKSkKICAgICAgICBwYXJhbXM9amF4LnRyZWVfdXRpbC50cmVlX21hcChsYW1iZGEgdyxhLGI6dy1sciooYS8oMS0uOSoqY291bnQpKS8oam5wLnNxcnQoYi8oMS0uOTk5Kipjb3VudCkpKzFlLTgpLHBhcmFtcyxtLHYpCiAgICAgICAgcmV0dXJuIHBhcmFtcyxtLHYsdmFsdWUsYXV4LGduCiAgICBkZWYgYmFsYW5jZShwYXJhbXMsYmF0Y2gpOgogICAgICAgIGdwPWpheC5ncmFkKGxhbWJkYSB0aGV0YTpzdW0odGVybXModGhldGEsYmF0Y2gpWzoyXSkpKHBhcmFtcykKICAgICAgICBnZD1qYXguZ3JhZChsYW1iZGEgdGhldGE6dGVybXModGhldGEsYmF0Y2gpWzJdKShwYXJhbXMpCiAgICAgICAgcmV0dXJuIG5vcm0oZ3ApLG5vcm0oZ2QpCiAgICByZXR1cm4geydmaWVsZHMnOmpheC5qaXQoZmllbGRzKSwncGh5c2ljcyc6amF4LmppdChwaHlzaWNzKSwnb3V0bGV0JzpqYXguaml0KG91dCksCiAgICAgICAgICAgICdzdGVwJzpqYXguaml0KHN0ZXApLCdiYWxhbmNlJzpqYXguaml0KGJhbGFuY2UpLCd0ZXJtcyc6amF4LmppdCh0ZXJtcyl9Cg==', 'channel_operator/v2_model.py': 'IiIiRGl2ZXJnZW5jZS1mcmVlIHBoeXNpY3MtaW5mb3JtZWQgRGVlcE9OZXQgd2l0aCBwYXBlcidzIG1vZGlmaWVkIHRhbmggTUxQcy4iIiIKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCmZyb20gLnYyX3Byb2ZpbGVzIGltcG9ydCBQUk9KRUNULCBJTlRFR1JBTFMKZnJvbSAucHJvZmlsZXMgaW1wb3J0IFkKZnJvbSAubW9kZWwgaW1wb3J0IGdyYWRpZW50CgoKY2xhc3MgTW9kaWZpZWRNTFAobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnB1dHMsIHdpZHRoLCBvdXRwdXRzLCBkZXB0aD0zKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLnU9bm4uTGluZWFyKGlucHV0cyx3aWR0aCk7IHNlbGYudj1ubi5MaW5lYXIoaW5wdXRzLHdpZHRoKQogICAgICAgIHNlbGYuZ2F0ZXM9bm4uTW9kdWxlTGlzdChbbm4uTGluZWFyKGlucHV0cyx3aWR0aCldK1tubi5MaW5lYXIod2lkdGgsd2lkdGgpIGZvciBfIGluIHJhbmdlKGRlcHRoLTEpXSkKICAgICAgICBzZWxmLm91dHB1dD1ubi5MaW5lYXIod2lkdGgsb3V0cHV0cykKICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5tb2R1bGVzKCk6CiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UobGF5ZXIsbm4uTGluZWFyKToKICAgICAgICAgICAgICAgIG5uLmluaXQueGF2aWVyX25vcm1hbF8obGF5ZXIud2VpZ2h0KTtubi5pbml0Lnplcm9zXyhsYXllci5iaWFzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYseCk6CiAgICAgICAgdSx2PXRvcmNoLnRhbmgoc2VsZi51KHgpKSx0b3JjaC50YW5oKHNlbGYudih4KSkKICAgICAgICBoPXgKICAgICAgICBmb3IgZ2F0ZSBpbiBzZWxmLmdhdGVzOgogICAgICAgICAgICB6PXRvcmNoLnRhbmgoZ2F0ZShoKSk7aD16KnUrKDEteikqdgogICAgICAgIHJldHVybiBzZWxmLm91dHB1dChoKQoKCmNsYXNzIFN0cmVhbURlZXBPTmV0KG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZix3aWR0aD02NCxsYXRlbnQ9MzIsZGVwdGg9MyxwcmVzc3VyZV9tb2RlPSd2aXNjb3VzJyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi5jb25maWc9ZGljdCh3aWR0aD13aWR0aCxsYXRlbnQ9bGF0ZW50LGRlcHRoPWRlcHRoKQogICAgICAgIHNlbGYucHJlc3N1cmVfbW9kZT1wcmVzc3VyZV9tb2RlCiAgICAgICAgaWYgcHJlc3N1cmVfbW9kZSBub3QgaW4gWyd2aXNjb3VzJywnY29udmVjdGl2ZSddOnJhaXNlIFZhbHVlRXJyb3IoJ1Vua25vd24gcHJlc3N1cmUgbW9kZScpCiAgICAgICAgaWYgcHJlc3N1cmVfbW9kZSE9J3Zpc2NvdXMnOnNlbGYuY29uZmlnWydwcmVzc3VyZV9tb2RlJ109cHJlc3N1cmVfbW9kZQogICAgICAgIHNlbGYubGF0ZW50PWxhdGVudAogICAgICAgIHNlbGYuYnJhbmNoPU1vZGlmaWVkTUxQKDM0LHdpZHRoLDIqbGF0ZW50LGRlcHRoKQogICAgICAgIHNlbGYudHJ1bms9TW9kaWZpZWRNTFAoMTQsd2lkdGgsMipsYXRlbnQsZGVwdGgpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYudHJ1bmsub3V0cHV0LndlaWdodCxzdGQ9MWUtMykKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigncHJvamVjdGlvbicsdG9yY2gudGVuc29yKFBST0pFQ1QsZHR5cGU9dG9yY2guZmxvYXQzMikpCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoJ2ludGVncmFscycsdG9yY2gudGVuc29yKElOVEVHUkFMUyxkdHlwZT10b3JjaC5mbG9hdDMyKSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcignYmFzZScsdG9yY2gudGVuc29yKDYqWSooMS1ZKSxkdHlwZT10b3JjaC5mbG9hdDMyKSkKCiAgICBkZWYgaW5sZXRfcGFyYW1ldGVycyhzZWxmLHByb2ZpbGVzKToKICAgICAgICBiPXByb2ZpbGVzQHNlbGYucHJvamVjdGlvbi5UCiAgICAgICAgZmx1eD1iWzosMF0rKGJbOiwxOl0qc2VsZi5pbnRlZ3JhbHMpLnN1bSgtMSkKICAgICAgICByZXR1cm4gYi9mbHV4WzosTm9uZV0KCiAgICBkZWYgcG90ZW50aWFscyhzZWxmLHByb2ZpbGVzLHJlLHh5KToKICAgICAgICB4LHk9eHlbLi4uLDBdLHh5Wy4uLiwxXQogICAgICAgIGNvbmRpdGlvbj10b3JjaC5jYXQoW3Byb2ZpbGVzLXNlbGYuYmFzZSx0b3JjaC5sb2cocmUvMTApWzosTm9uZV0vdG9yY2gubG9nKHJlLm5ld190ZW5zb3IoMTAuKSldLC0xKQogICAgICAgIGJyYW5jaD1zZWxmLmJyYW5jaChjb25kaXRpb24pLnJlc2hhcGUoLTEsMixzZWxmLmxhdGVudCkKICAgICAgICBtb2Rlcz10b3JjaC5hcmFuZ2UoMSw1LGR0eXBlPXh5LmR0eXBlLGRldmljZT14eS5kZXZpY2UpCiAgICAgICAgYWxwaGE9dG9yY2gucGkqbW9kZXMKICAgICAgICBwaGFzZT15Wy4uLixOb25lXSphbHBoYQogICAgICAgIGZlYXR1cmVzPXRvcmNoLmNhdChbdG9yY2guc3RhY2soWzIqeC81LTEsMip5LTFdLC0xKSx0b3JjaC5zaW4ocGhhc2UpLHRvcmNoLmNvcyhwaGFzZSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3JjaC5zdGFjayhbdG9yY2gubG9nMXAoMjAqeCkvdG9yY2gubG9nKHgubmV3X3RlbnNvcigxMDEuKSksdG9yY2guZXhwKC14KSx0b3JjaC5leHAoLTQqeCksdG9yY2guZXhwKC0xNip4KV0sLTEpXSwtMSkKICAgICAgICB0cnVuaz1zZWxmLnRydW5rKGZlYXR1cmVzKS5yZXNoYXBlKCp4LnNoYXBlLDIsc2VsZi5sYXRlbnQpCiAgICAgICAgcmF3PXRvcmNoLmVpbnN1bSgnYm9sLGJub2wtPmJubycsYnJhbmNoLHRydW5rKS9zZWxmLmxhdGVudCoqLjUKICAgICAgICBiPXNlbGYuaW5sZXRfcGFyYW1ldGVycyhwcm9maWxlcykKICAgICAgICBiYXNlPTMqeSp5LTIqeSoqMwogICAgICAgICMgQW5hbHl0aWMgaW50ZWdyYWxzIG9mIDYgeSgxLXkpIGNvcyhrKnBpKnkpLCB6ZXJvIGF0IHk9MC4KICAgICAgICBpaz02KigoeS15KnkpWy4uLixOb25lXSp0b3JjaC5zaW4ocGhhc2UpL2FscGhhICsgKDEtMip5KVsuLi4sTm9uZV0qdG9yY2guY29zKHBoYXNlKS9hbHBoYSoqMgogICAgICAgICAgICAgICsyKnRvcmNoLnNpbihwaGFzZSkvYWxwaGEqKjMtMS9hbHBoYSoqMikKICAgICAgICBkZXZpYXRpb249aWstYmFzZVsuLi4sTm9uZV0qc2VsZi5pbnRlZ3JhbHMKICAgICAgICAjIE1vZGUtZGVwZW5kZW50IHZpc2NvdXMgZGV2ZWxvcG1lbnQgc2NhbGUsIGNvcnJlY3RlZCBieSB0aGUgbGVhcm5lZCBmaWVsZC4KICAgICAgICBlbGw9LjErLjA1KnJlWzosTm9uZSxOb25lXS9tb2RlcyoqMgogICAgICAgIGRlY2F5PXRvcmNoLmV4cCgtKHhbLi4uLE5vbmVdL2VsbCkqKjIpCiAgICAgICAgcHNpPWJhc2UrKGJbOixOb25lLDE6XSpkZXZpYXRpb24qZGVjYXkpLnN1bSgtMSkKICAgICAgICBtYXNrPTE2KnkqeSooMS15KSoqMiooMS10b3JjaC5leHAoLXgvLjIpKSoqMgogICAgICAgIHBzaT1wc2krbWFzaypyYXdbLi4uLDBdCiAgICAgICAgcHJlc3N1cmVfc2NhbGU9MTIvcmVbOixOb25lXSBpZiBzZWxmLnByZXNzdXJlX21vZGU9PSd2aXNjb3VzJyBlbHNlIDEuCiAgICAgICAgcD0xMi9yZVs6LE5vbmVdKig1LXgpK3ByZXNzdXJlX3NjYWxlKnJhd1suLi4sMV0KICAgICAgICByZXR1cm4gcHNpLHAKCiAgICBkZWYgZm9yd2FyZChzZWxmLHByb2ZpbGVzLHJlLHh5KToKICAgICAgICBhY3RpdmU9dG9yY2guaXNfZ3JhZF9lbmFibGVkKCkKICAgICAgICB3aXRoIHRvcmNoLmVuYWJsZV9ncmFkKCk6CiAgICAgICAgICAgIGlmIG5vdCB4eS5yZXF1aXJlc19ncmFkOgogICAgICAgICAgICAgICAgeHk9eHkuZGV0YWNoKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgICAgICAgICAgcHNpLHA9c2VsZi5wb3RlbnRpYWxzKHByb2ZpbGVzLHJlLHh5KQogICAgICAgICAgICBnPXRvcmNoLmF1dG9ncmFkLmdyYWQocHNpLnN1bSgpLHh5LGNyZWF0ZV9ncmFwaD1hY3RpdmUscmV0YWluX2dyYXBoPVRydWUpWzBdCiAgICAgICAgICAgIGZpZWxkcz10b3JjaC5zdGFjayhbZ1suLi4sMV0sLWdbLi4uLDBdLHBdLC0xKQogICAgICAgIHJldHVybiBmaWVsZHMgaWYgYWN0aXZlIGVsc2UgZmllbGRzLmRldGFjaCgpCgoKZGVmIHBoeXNpY3MobW9kZWwscHJvZmlsZXMscmUseHkpOgogICAgeHk9eHkuZGV0YWNoKCkucmVxdWlyZXNfZ3JhZF8oVHJ1ZSkKICAgIGZpZWxkcz1tb2RlbChwcm9maWxlcyxyZSx4eSkKICAgIHUsdixwPWZpZWxkcy51bmJpbmQoLTEpCiAgICBndSxndixncD1bZ3JhZGllbnQoeix4eSkgZm9yIHogaW4gKHUsdixwKV0KICAgIGx1PWdyYWRpZW50KGd1Wy4uLiwwXSx4eSlbLi4uLDBdK2dyYWRpZW50KGd1Wy4uLiwxXSx4eSlbLi4uLDFdCiAgICBsdj1ncmFkaWVudChndlsuLi4sMF0seHkpWy4uLiwwXStncmFkaWVudChndlsuLi4sMV0seHkpWy4uLiwxXQogICAgbXg9dSpndVsuLi4sMF0rdipndVsuLi4sMV0rZ3BbLi4uLDBdLWx1L3JlWzosTm9uZV0KICAgIG15PXUqZ3ZbLi4uLDBdK3YqZ3ZbLi4uLDFdK2dwWy4uLiwxXS1sdi9yZVs6LE5vbmVdCiAgICByZXR1cm4gbXgsbXksZ3VbLi4uLDBdK2d2Wy4uLiwxXQoKCmRlZiBvdXRsZXQobW9kZWwscHJvZmlsZXMscmUseSk6CiAgICB4eT10b3JjaC5zdGFjayhbdG9yY2guZnVsbF9saWtlKHksNS4pLHldLC0xKS5yZXF1aXJlc19ncmFkXyhUcnVlKQogICAgZmllbGRzPW1vZGVsKHByb2ZpbGVzLHJlLHh5KQogICAgdXg9Z3JhZGllbnQoZmllbGRzWy4uLiwwXSx4eSlbLi4uLDBdCiAgICB2eD1ncmFkaWVudChmaWVsZHNbLi4uLDFdLHh5KVsuLi4sMF0KICAgIHJldHVybiB1eC1yZVs6LE5vbmVdKmZpZWxkc1suLi4sMl0sdngKCgpkZWYgbG9hZChwYXRoLGRldmljZT0nY3B1Jyk6CiAgICBzdGF0ZT10b3JjaC5sb2FkKHBhdGgsbWFwX2xvY2F0aW9uPWRldmljZSx3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICBtb2RlbD1TdHJlYW1EZWVwT05ldCgqKnN0YXRlWydjb25maWcnXSkudG8oZGV2aWNlKQogICAgbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHN0YXRlWydzdGF0ZV9kaWN0J10pO21vZGVsLmV2YWwoKQogICAgcmV0dXJuIG1vZGVsLHN0YXRlCg==', 'channel_operator/v2_evaluate.py': 'IiIiSW5kZXBlbmRlbnQgcG9zdC10cmFpbmluZyBldmFsdWF0aW9uIG9mIHRoZSBleHBvcnRlZCBmaXhlZC13ZWlnaHQgb3BlcmF0b3IuIiIiCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCBtYXRwbG90bGliCm1hdHBsb3RsaWIudXNlKCdBZ2cnKQppbXBvcnQgbWF0cGxvdGxpYi5weXBsb3QgYXMgcGx0CmZyb20gLnYyX21vZGVsIGltcG9ydCBsb2FkLHBoeXNpY3Msb3V0bGV0CmZyb20gLnYyX3Byb2ZpbGVzIGltcG9ydCBpbmxldF92YWx1ZXMKCgpkZWYgZXZhbHVhdGUoY2hlY2twb2ludCxkYXRhX3BhdGgsb3V0cHV0LHJvbGUsdHJhaW5pbmdfZGF0YT1Ob25lKToKICAgIG91dHB1dD1QYXRoKG91dHB1dCk7b3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1UcnVlKTt0b3JjaC5zZXRfbnVtX3RocmVhZHMoNCkKICAgIG1vZGVsLHN0YXRlPWxvYWQoY2hlY2twb2ludCk7bW9kZWwuZG91YmxlKCkKICAgIHQ9bGFtYmRhIHg6dG9yY2gudGVuc29yKHgsZHR5cGU9dG9yY2guZmxvYXQ2NCkKICAgIHdpdGggbnAubG9hZChkYXRhX3BhdGgpIGFzIGQ6ZGF0YT17azpkW2tdLmNvcHkoKSBmb3IgayBpbiBkLmZpbGVzfQogICAgcHJvZmlsZV9kaXNqb2ludD1Ob25lCiAgICBpZiB0cmFpbmluZ19kYXRhOgogICAgICAgIHdpdGggbnAubG9hZCh0cmFpbmluZ19kYXRhKSBhcyBkOnRyYWluX3Byb2ZpbGVzPWRbJ3Byb2ZpbGVzJ10uYXN0eXBlKG5wLmZsb2F0MzIpCiAgICAgICAgaGVsZD1kYXRhWydwcm9maWxlcyddLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIHByb2ZpbGVfZGlzam9pbnQ9bm90IG5wLmFueShucC5hbGwodHJhaW5fcHJvZmlsZXNbOixOb25lLDpdPT1oZWxkW05vbmUsOiw6XSxheGlzPS0xKSkKICAgICAgICBpZiBub3QgcHJvZmlsZV9kaXNqb2ludDpyYWlzZSBWYWx1ZUVycm9yKCdIZWxkLW91dCBwcm9maWxlIG92ZXJsYXBzIHN1cGVydmlzZWQgdHJhaW5pbmcgaW5wdXRzJykKICAgIHByZWRpY3Rpb249W107cmVjb3Jkcz1bXTtybmc9bnAucmFuZG9tLmRlZmF1bHRfcm5nKDczMDUzMSkKICAgIGZvciBpIGluIHJhbmdlKGxlbihkYXRhWydwcm9maWxlcyddKSk6CiAgICAgICAgcCxyPXQoZGF0YVsncHJvZmlsZXMnXVtpOmkrMV0pLHQoZGF0YVsncmV5bm9sZHMnXVtpOmkrMV0pCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6cHJlZD1tb2RlbChwLHIsdChkYXRhWyd4eSddW05vbmVdKSkubnVtcHkoKVswXQogICAgICAgIHRydXRoPWRhdGFbJ2ZpZWxkcyddW2ldO2Vycj1wcmVkLXRydXRoO3ByZWRpY3Rpb24uYXBwZW5kKHByZWQpCiAgICAgICAgeHk9dChybmcudW5pZm9ybShbLjAwNSwuMDA1XSxbNC45OTUsLjk5NV0sKDEsNTEyLDIpKSkKICAgICAgICBteCxteSxtYXNzPXBoeXNpY3MobW9kZWwscCxyLHh5KQogICAgICAgIG9uLG90PW91dGxldChtb2RlbCxwLHIsdChucC5saW5zcGFjZSguMDAxLC45OTksMTI5KVtOb25lXSkpCiAgICAgICAgZ3gsZ3k9bnAubWVzaGdyaWQobnAubGluc3BhY2UoMCw1LDIxKSxucC5saW5zcGFjZSgwLDEsMjU3KSxpbmRleGluZz0naWonKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOmdyaWQ9bW9kZWwocCxyLHQobnAuc3RhY2soW2d4LGd5XSwtMSkucmVzaGFwZSgxLC0xLDIpKSkubnVtcHkoKS5yZXNoYXBlKDIxLDI1NywzKQogICAgICAgIGZsdXg9bnAudHJhcGV6b2lkKGdyaWRbLi4uLDBdLGd5LGF4aXM9LTEpCiAgICAgICAgYnk9bnAubGluc3BhY2UoMCwxLDI1NykKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKTppbmxldD1tb2RlbChwLHIsdChucC5zdGFjayhbbnAuemVyb3NfbGlrZShieSksYnldLC0xKVtOb25lXSkpLm51bXB5KClbMF0KICAgICAgICBpbmxldF9leHBlY3RlZD1pbmxldF92YWx1ZXMoZGF0YVsnY29lZmZpY2llbnRzJ11baV0sYnkpCiAgICAgICAgcmVjb3Jkcy5hcHBlbmQoeydjYXNlJzppLCdjb2VmZmljaWVudHMnOmRhdGFbJ2NvZWZmaWNpZW50cyddW2ldLnRvbGlzdCgpLCdyZXlub2xkcyc6ZmxvYXQoZGF0YVsncmV5bm9sZHMnXVtpXSksCiAgICAgICAgICAgICd2ZWxvY2l0eV9yZWxhdGl2ZV9sMic6ZmxvYXQobnAubGluYWxnLm5vcm0oZXJyWzosOjJdKS9ucC5saW5hbGcubm9ybSh0cnV0aFs6LDoyXSkpLAogICAgICAgICAgICAndV9yZWxhdGl2ZV9sMic6ZmxvYXQobnAubGluYWxnLm5vcm0oZXJyWzosMF0pL25wLmxpbmFsZy5ub3JtKHRydXRoWzosMF0pKSwKICAgICAgICAgICAgJ3ZfcmVsYXRpdmVfbDInOmZsb2F0KG5wLmxpbmFsZy5ub3JtKGVycls6LDFdKS9tYXgobnAubGluYWxnLm5vcm0odHJ1dGhbOiwxXSksMWUtMTIpKSwKICAgICAgICAgICAgJ3Zfcm1zZSc6ZmxvYXQobnAuc3FydChucC5tZWFuKGVycls6LDFdKioyKSkpLAogICAgICAgICAgICAncHJlc3N1cmVfcmVsYXRpdmVfbDInOmZsb2F0KG5wLmxpbmFsZy5ub3JtKGVycls6LDJdKS9ucC5saW5hbGcubm9ybSh0cnV0aFs6LDJdKSksCiAgICAgICAgICAgICdkaXZlcmdlbmNlX3Jtcyc6ZmxvYXQobWFzcy5zcXVhcmUoKS5tZWFuKCkuc3FydCgpLmRldGFjaCgpKSwKICAgICAgICAgICAgJ21vbWVudHVtX3Jtcyc6ZmxvYXQoKG14LnNxdWFyZSgpLm1lYW4oKStteS5zcXVhcmUoKS5tZWFuKCkpLnNxcnQoKS5kZXRhY2goKSksCiAgICAgICAgICAgICdvdXRsZXRfcm1zJzpmbG9hdCgob24uc3F1YXJlKCkubWVhbigpK290LnNxdWFyZSgpLm1lYW4oKSkuc3FydCgpLmRldGFjaCgpKSwKICAgICAgICAgICAgJ2ZsdXhfbWF4X2Vycm9yJzpmbG9hdChucC5hYnMoZmx1eC0xKS5tYXgoKSksJ3dhbGxfbWF4X3ZlbG9jaXR5JzpmbG9hdChucC5hYnMoZ3JpZFs6LFswLC0xXSw6Ml0pLm1heCgpKSwKICAgICAgICAgICAgJ2lubGV0X21heF92ZWxvY2l0eV9lcnJvcic6ZmxvYXQobWF4KG5wLmFicyhpbmxldFs6LDBdLWlubGV0X2V4cGVjdGVkKS5tYXgoKSxucC5hYnMoaW5sZXRbOiwxXSkubWF4KCkpKX0pCiAgICAgICAgcHJpbnQoanNvbi5kdW1wcyhyZWNvcmRzWy0xXSksZmx1c2g9VHJ1ZSkKICAgIG1heGltdW09bGFtYmRhIGtleTptYXgoY1trZXldIGZvciBjIGluIHJlY29yZHMpCiAgICBnYXRlcz17J3ZlbG9jaXR5JzptYXhpbXVtKCd2ZWxvY2l0eV9yZWxhdGl2ZV9sMicpPC4wNSwncHJlc3N1cmUnOm1heGltdW0oJ3ByZXNzdXJlX3JlbGF0aXZlX2wyJyk8LjEsCiAgICAgICAgJ21vbWVudHVtJzptYXhpbXVtKCdtb21lbnR1bV9ybXMnKTwuMDUsJ291dGxldCc6bWF4aW11bSgnb3V0bGV0X3JtcycpPC4xLCdkaXZlcmdlbmNlJzptYXhpbXVtKCdkaXZlcmdlbmNlX3JtcycpPDFlLTQsCiAgICAgICAgJ2ZsdXgnOm1heGltdW0oJ2ZsdXhfbWF4X2Vycm9yJyk8LjAyLCdoYXJkX2JvdW5kYXJpZXMnOm1heChtYXhpbXVtKCd3YWxsX21heF92ZWxvY2l0eScpLG1heGltdW0oJ2lubGV0X21heF92ZWxvY2l0eV9lcnJvcicpKTwxZS01fQogICAgcmVwb3J0PXsnc3RlcCc6c3RhdGVbJ3N0ZXAnXSwncm9sZSc6cm9sZSwncHJvZmlsZV9kaXNqb2ludF9mcm9tX3N1cGVydmlzZWRfdHJhaW5pbmcnOnByb2ZpbGVfZGlzam9pbnQsJ2R0eXBlJzonZmxvYXQ2NCBpbmZlcmVuY2UvZGlhZ25vc3RpY3MgZnJvbSB0cmFpbmVkIGZsb2F0MzIgd2VpZ2h0cycsCiAgICAgICAgICAgICdjb25maWcnOnN0YXRlWydjb25maWcnXSwnY2FzZXMnOnJlY29yZHMsJ2dhdGVzJzpnYXRlcywncGFzc2VkJzphbGwoZ2F0ZXMudmFsdWVzKCkpLAogICAgICAgICAgICAnZmllbGRfYWNjdXJhY3lfcGFzc2VkJzpnYXRlc1sndmVsb2NpdHknXSBhbmQgZ2F0ZXNbJ3ByZXNzdXJlJ119CiAgICAob3V0cHV0LydtZXRyaWNzLmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LGluZGVudD0yKSkKICAgIHByZWQ9bnAuYXJyYXkocHJlZGljdGlvbikKICAgIG5wLnNhdmV6X2NvbXByZXNzZWQob3V0cHV0LydwcmVkaWN0aW9ucy5ucHonLHByZWRpY3Rpb249cHJlZCxyZWZlcmVuY2U9ZGF0YVsnZmllbGRzJ10seHk9ZGF0YVsneHknXSxwcm9maWxlcz1kYXRhWydwcm9maWxlcyddLHJleW5vbGRzPWRhdGFbJ3JleW5vbGRzJ10pCiAgICB3b3JzdD1pbnQobnAuYXJnbWF4KFtjWydwcmVzc3VyZV9yZWxhdGl2ZV9sMiddIGZvciBjIGluIHJlY29yZHNdKSk7bng9bGVuKG5wLnVuaXF1ZShkYXRhWyd4eSddWzosMF0pKTtueT1sZW4obnAudW5pcXVlKGRhdGFbJ3h5J11bOiwxXSkpCiAgICBmaWcsYXhlcz1wbHQuc3VicGxvdHMoMywzLGZpZ3NpemU9KDEzLDgpLGNvbnN0cmFpbmVkX2xheW91dD1UcnVlKQogICAgZm9yIGosbGFiZWwgaW4gZW51bWVyYXRlKFsndScsJ3YnLCdwJ10pOgogICAgICAgIGZvciBrLChuYW1lLGZpZWxkKSBpbiBlbnVtZXJhdGUoWygnRkVNJyxkYXRhWydmaWVsZHMnXSksKCdmaXhlZCBvcGVyYXRvcicscHJlZCksKCdlcnJvcicscHJlZC1kYXRhWydmaWVsZHMnXSldKToKICAgICAgICAgICAgbGltaXRzPXt9CiAgICAgICAgICAgIGlmIGs8MjpsaW1pdHM9ZGljdCh2bWluPW1pbihkYXRhWydmaWVsZHMnXVt3b3JzdCw6LGpdLm1pbigpLHByZWRbd29yc3QsOixqXS5taW4oKSksdm1heD1tYXgoZGF0YVsnZmllbGRzJ11bd29yc3QsOixqXS5tYXgoKSxwcmVkW3dvcnN0LDosal0ubWF4KCkpKQogICAgICAgICAgICBpbT1heGVzW2osa10uaW1zaG93KGZpZWxkW3dvcnN0LDosal0ucmVzaGFwZShueCxueSkuVCxvcmlnaW49J2xvd2VyJyxleHRlbnQ9Wy4wMSw0Ljk5LC4wMDUsLjk5NV0sYXNwZWN0PSdhdXRvJyxjbWFwPSdjb29sd2FybScsKipsaW1pdHMpCiAgICAgICAgICAgIGF4ZXNbaixrXS5zZXRfdGl0bGUoZid7bGFiZWx9OiB7bmFtZX0nKTtmaWcuY29sb3JiYXIoaW0sYXg9YXhlc1tqLGtdKQogICAgZmlnLnN1cHRpdGxlKGYne3JvbGV9OiB3b3JzdCBwcmVzc3VyZSBjYXNlLCBSZT17ZGF0YVsicmV5bm9sZHMiXVt3b3JzdF06Z30sIGNhc2Uge3dvcnN0fScpCiAgICBmaWcuc2F2ZWZpZyhvdXRwdXQvJ2ZpZWxkcy5wbmcnLGRwaT0xNjApO3BsdC5jbG9zZShmaWcpCiAgICByZXR1cm4gcmVwb3J0Cg==', 'channel_operator/v2_predict.py': 'IiIiSW5mZXJlbmNlIHdpdGggZml4ZWQgdjIgd2VpZ2h0cywgbmV2ZXIgcGVyLWlubGV0IHJldHJhaW5pbmcgb3IgQ0ZEIHNvbHZpbmcuIiIiCmltcG9ydCBhcmdwYXJzZSxqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmZyb20gLnYyX21vZGVsIGltcG9ydCBsb2FkLHBoeXNpY3MKZnJvbSAudjJfcHJvZmlsZXMgaW1wb3J0IGlubGV0X3ZhbHVlcyxQUk9KRUNULEEsSU5URUdSQUxTCmZyb20gLnByb2ZpbGVzIGltcG9ydCBsb2FkX2NzdgoKCmRlZiBtYWluKCk6CiAgICBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cC5hZGRfYXJndW1lbnQoJy0tY2hlY2twb2ludCcscmVxdWlyZWQ9VHJ1ZSk7cC5hZGRfYXJndW1lbnQoJy0tcmUnLHJlcXVpcmVkPVRydWUsdHlwZT1mbG9hdCkKICAgIGc9cC5hZGRfbXV0dWFsbHlfZXhjbHVzaXZlX2dyb3VwKHJlcXVpcmVkPVRydWUpO2cuYWRkX2FyZ3VtZW50KCctLWNvZWZmaWNpZW50cycsdHlwZT1mbG9hdCxuYXJncz00KTtnLmFkZF9hcmd1bWVudCgnLS1pbmxldC1jc3YnKQogICAgcC5hZGRfYXJndW1lbnQoJy0tb3V0cHV0JyxkZWZhdWx0PSdwcmVkaWN0aW9uX3YyLm5weicpO3AuYWRkX2FyZ3VtZW50KCctLW54Jyx0eXBlPWludCxkZWZhdWx0PTIwMSk7cC5hZGRfYXJndW1lbnQoJy0tbnknLHR5cGU9aW50LGRlZmF1bHQ9ODEpCiAgICBhPXAucGFyc2VfYXJncygpO3RvcmNoLnNldF9udW1fdGhyZWFkcyg0KQogICAgaWYgbm90IDEwPD1hLnJlPD0xMDA6cC5lcnJvcignUmV5bm9sZHMgbXVzdCBiZSB3aXRoaW4gWzEwLDEwMF0nKQogICAgaWYgYS5jb2VmZmljaWVudHM6CiAgICAgICAgaWYgc3VtKGFicyh2KSBmb3IgdiBpbiBhLmNvZWZmaWNpZW50cyk+LjgwMDAwMTpwLmVycm9yKCdDb2VmZmljaWVudCBMMSByYWRpdXMgZXhjZWVkcyB0cmFpbmVkIGludGVydmFsIDAuOCcpCiAgICAgICAgcHJvZmlsZT1pbmxldF92YWx1ZXMobnAuYXJyYXkoYS5jb2VmZmljaWVudHMpKQogICAgZWxzZToKICAgICAgICBzdXBwbGllZD1sb2FkX2NzdihhLmlubGV0X2Nzdik7Yj1QUk9KRUNUQHN1cHBsaWVkO2IvPWJbMF0rYlsxOl1ASU5URUdSQUxTCiAgICAgICAgcHJvZmlsZT1BQGIKICAgICAgICBtaXNtYXRjaD1ucC5saW5hbGcubm9ybShwcm9maWxlLXN1cHBsaWVkKS9ucC5saW5hbGcubm9ybShzdXBwbGllZCkKICAgICAgICBpZiBtaXNtYXRjaD4uMDAxOnAuZXJyb3IoZidJbnB1dCBpcyBvdXRzaWRlIHRoZSBmb3VyLW1vZGUgaW5sZXQgZmFtaWx5IChmaXQgZXJyb3Ige21pc21hdGNoOi4yJX0pOyBib3VuZGFyeSB3aWxsIG5vdCBiZSBzaWxlbnRseSBjaGFuZ2VkJykKICAgICAgICBpZiBucC5hYnMoYlsxOl0vYlswXSkuc3VtKCk+LjgwMDAwMTpwLmVycm9yKCdDU1YgbGllcyBvdXRzaWRlIHRoZSB0cmFpbmVkIGFtcGxpdHVkZSBkb21haW4nKQogICAgbW9kZWwsc3RhdGU9bG9hZChhLmNoZWNrcG9pbnQpO21vZGVsLmRvdWJsZSgpCiAgICB0PWxhbWJkYSB4OnRvcmNoLnRlbnNvcih4LGR0eXBlPXRvcmNoLmZsb2F0NjQpCiAgICB4LHk9bnAubWVzaGdyaWQobnAubGluc3BhY2UoMCw1LGEubngpLG5wLmxpbnNwYWNlKDAsMSxhLm55KSxpbmRleGluZz0naWonKTt4eT1ucC5zdGFjayhbeCx5XSwtMSkKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOmY9bW9kZWwodChwcm9maWxlW05vbmVdKSx0KFthLnJlXSksdCh4eS5yZXNoYXBlKDEsLTEsMikpKS5udW1weSgpLnJlc2hhcGUoYS5ueCxhLm55LDMpCiAgICBvdXRwdXQ9UGF0aChhLm91dHB1dCk7b3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsZXhpc3Rfb2s9VHJ1ZSkKICAgIG5wLnNhdmV6X2NvbXByZXNzZWQob3V0cHV0LHg9eCx5PXksdT1mWy4uLiwwXSx2PWZbLi4uLDFdLHA9ZlsuLi4sMl0saW5sZXQ9cHJvZmlsZSxyZXlub2xkcz1hLnJlKQogICAgY29vcmRzPW5wLnJhbmRvbS5kZWZhdWx0X3JuZyg3NDY4MSkudW5pZm9ybShbLjAwNSwuMDA1XSxbNC45OTUsLjk5NV0sKDEsNTEyLDIpKQogICAgbXgsbXksbWFzcz1waHlzaWNzKG1vZGVsLHQocHJvZmlsZVtOb25lXSksdChbYS5yZV0pLHQoY29vcmRzKSkKICAgIGZsdXg9bnAudHJhcGV6b2lkKGZbLi4uLDBdLHksYXhpcz0tMSkKICAgIHJlcG9ydD17J3N0ZXAnOnN0YXRlWydzdGVwJ10sJ3JldHJhaW5pbmcnOkZhbHNlLCdyZXlub2xkcyc6YS5yZSwnb3V0cHV0JzpzdHIob3V0cHV0KSwKICAgICAgICAnZGl2ZXJnZW5jZV9ybXMnOmZsb2F0KG1hc3Muc3F1YXJlKCkubWVhbigpLnNxcnQoKS5kZXRhY2goKSksJ21vbWVudHVtX3Jtcyc6ZmxvYXQoKG14LnNxdWFyZSgpLm1lYW4oKStteS5zcXVhcmUoKS5tZWFuKCkpLnNxcnQoKS5kZXRhY2goKSksCiAgICAgICAgJ2ZsdXhfbWF4X2Vycm9yJzpmbG9hdChucC5hYnMoZmx1eC0xKS5tYXgoKSksJ3Njb3BlJzonZm91ciBzbW9vdGggY29zaW5lIG1vZGVzLCBMMTw9MC44LCBSZSBpbiBbMTAsMTAwXTsgZml4ZWQgdHJhaW5lZCBvcGVyYXRvcid9CiAgICB2YWxpZGF0aW9uPVBhdGgoYS5jaGVja3BvaW50KS5wYXJlbnQvJ3ZhbGlkYXRpb25fc3VtbWFyeS5qc29uJwogICAgaWYgdmFsaWRhdGlvbi5leGlzdHMoKToKICAgICAgICBjaGVja3M9anNvbi5sb2Fkcyh2YWxpZGF0aW9uLnJlYWRfdGV4dCgpKQogICAgICAgIHJlcG9ydFsncHVibGlzaGVkX3ZhbGlkYXRpb24nXT1jaGVja3MKICAgICAgICBpZiBub3QgY2hlY2tzLmdldCgnYWxsX2dhdGVzX3Bhc3NlZCcpOgogICAgICAgICAgICBwcmludCgnV0FSTklORzogZmllbGQgdGVzdHMgcGFzc2VkIGJ1dCB0aGUgc3RyaWN0IG1vbWVudHVtLXJlc2lkdWFsIHRhcmdldCByZW1haW5zIHVubWV0OyBzZWUgdGhlIG1vZGVsIHJlcG9ydCcpCiAgICBvdXRwdXQud2l0aF9zdWZmaXgoJy5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlcG9ydCxpbmRlbnQ9MikpO3ByaW50KGpzb24uZHVtcHMocmVwb3J0KSkKCgppZiBfX25hbWVfXz09J19fbWFpbl9fJzptYWluKCkK', 'scripts/evaluate_v2.py': 'aW1wb3J0IGFyZ3BhcnNlLHN5cwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKc3lzLnBhdGguaW5zZXJ0KDAsc3RyKFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdKSkKZnJvbSBjaGFubmVsX29wZXJhdG9yLnYyX2V2YWx1YXRlIGltcG9ydCBldmFsdWF0ZQoKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiAgICBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cC5hZGRfYXJndW1lbnQoJy0tY2hlY2twb2ludCcscmVxdWlyZWQ9VHJ1ZSk7cC5hZGRfYXJndW1lbnQoJy0tZGF0YScscmVxdWlyZWQ9VHJ1ZSk7cC5hZGRfYXJndW1lbnQoJy0tb3V0cHV0JyxyZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoJy0tcm9sZScsY2hvaWNlcz1bJ3JlcGVhdGVkX21vbml0b3JfdmFsaWRhdGlvbicsJ3VudG91Y2hlZF9maW5hbF9ibGluZF90ZXN0J10scmVxdWlyZWQ9VHJ1ZSkKICAgIHAuYWRkX2FyZ3VtZW50KCctLXRyYWluaW5nLWRhdGEnKQogICAgYT1wLnBhcnNlX2FyZ3MoKTtldmFsdWF0ZShhLmNoZWNrcG9pbnQsYS5kYXRhLGEub3V0cHV0LGEucm9sZSxhLnRyYWluaW5nX2RhdGEpCg==', 'run_v2_jax.py': 'IiIiQ29tcGlsZWQgdHJhaW5pbmcgd2l0aCBtb25pdG9yZWQgQ0ZEIGxhYmVscyBleGNsdWRlZCBmcm9tIGV2ZXJ5IGdyYWRpZW50LiIiIgppbXBvcnQgb3MKb3MuZW52aXJvbi5zZXRkZWZhdWx0KCdYTEFfUFlUSE9OX0NMSUVOVF9QUkVBTExPQ0FURScsJ2ZhbHNlJykKaW1wb3J0IGFyZ3BhcnNlLGpzb24sdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBqYXgKaW1wb3J0IGpheC5udW1weSBhcyBqbnAKZnJvbSBjaGFubmVsX29wZXJhdG9yLnYyX2pheCBpbXBvcnQgaW5pdGlhbGl6ZSxtYWtlX2Z1bmN0aW9ucwpmcm9tIGNoYW5uZWxfb3BlcmF0b3IudjJfcHJvZmlsZXMgaW1wb3J0IHNhbXBsZQoKCmRlZiB0cmFpbihhcmdzKToKICAgIG91dHB1dD1QYXRoKGFyZ3Mub3V0cHV0KTtvdXRwdXQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBkZXZpY2VzPWpheC5kZXZpY2VzKCkKICAgIGlmIGFyZ3MucmVxdWlyZV9ncHUgYW5kIG5vdCBhbnkoZC5wbGF0Zm9ybT09J2dwdScgZm9yIGQgaW4gZGV2aWNlcyk6cmFpc2UgUnVudGltZUVycm9yKCdHUFUgcmVxdWlyZWQnKQogICAgd2l0aCBucC5sb2FkKFBhdGgoYXJncy5kYXRhKS8ndHJhaW4ubnB6JykgYXMgZDp0cmFpbj17azpkW2tdLmFzdHlwZShucC5mbG9hdDMyKSBmb3IgayBpbiBkLmZpbGVzfQogICAgd2l0aCBucC5sb2FkKFBhdGgoYXJncy5kYXRhKS8nbW9uaXRvci5ucHonKSBhcyBkOm1vbml0b3I9e2s6ZFtrXS5hc3R5cGUobnAuZmxvYXQzMikgZm9yIGsgaW4gZC5maWxlc30KICAgIGlmIGFyZ3MucmVsYXRpdmVfZGF0YToKICAgICAgICBybXM9bnAuc3FydChucC5tZWFuKHRyYWluWydmaWVsZHMnXSoqMixheGlzPTEpKQogICAgICAgIHRyYWluWydkYXRhX3NjYWxlJ109MS9ucC5tYXhpbXVtKHJtcyxucC5hcnJheShbLjI1LC4wMSwuMDVdLGR0eXBlPW5wLmZsb2F0MzIpKQogICAgZWxzZToKICAgICAgICB0cmFpblsnZGF0YV9zY2FsZSddPW5wLmNvbHVtbl9zdGFjayhbbnAub25lcyhsZW4odHJhaW5bJ3Byb2ZpbGVzJ10pKSxucC5mdWxsKGxlbih0cmFpblsncHJvZmlsZXMnXSksMTAuKSx0cmFpblsncmV5bm9sZHMnXS82MF0pLmFzdHlwZShucC5mbG9hdDMyKQogICAgaWYgbnAubWluKG5wLmFicyh0cmFpblsnY29lZmZpY2llbnRzJ11bOixOb25lXS1tb25pdG9yWydjb2VmZmljaWVudHMnXVtOb25lXSkubWF4KC0xKSk8MWUtODpyYWlzZSBWYWx1ZUVycm9yKCdNb25pdG9yIGxlYWthZ2UnKQogICAgY29uZmlnPWRpY3Qod2lkdGg9YXJncy53aWR0aCxsYXRlbnQ9YXJncy5sYXRlbnQsZGVwdGg9YXJncy5kZXB0aCkKICAgIGlmIGFyZ3MucHJlc3N1cmVfbW9kZSE9J3Zpc2NvdXMnOmNvbmZpZ1sncHJlc3N1cmVfbW9kZSddPWFyZ3MucHJlc3N1cmVfbW9kZQogICAgcGFyYW1zPWluaXRpYWxpemUoKipjb25maWcpO209amF4LnRyZWVfdXRpbC50cmVlX21hcChqbnAuemVyb3NfbGlrZSxwYXJhbXMpO3Y9amF4LnRyZWVfdXRpbC50cmVlX21hcChqbnAuemVyb3NfbGlrZSxwYXJhbXMpCiAgICBybmc9bnAucmFuZG9tLmRlZmF1bHRfcm5nKDUxOTcpO3dlaWdodD0xLjtmaXJzdD0wCiAgICBpZiBhcmdzLnJlc3VtZToKICAgICAgICB3aXRoIG5wLmxvYWQoYXJncy5yZXN1bWUsYWxsb3dfcGlja2xlPUZhbHNlKSBhcyBzOgogICAgICAgICAgICBpZiBqc29uLmxvYWRzKHN0cihzWydjb25maWdfanNvbiddKSkhPWNvbmZpZzpyYWlzZSBWYWx1ZUVycm9yKCdSZXN1bWUgbW9kZWwgY29uZmlndXJhdGlvbiBtaXNtYXRjaCcpCiAgICAgICAgICAgIHBhcmFtcz17azpqbnAuYXNhcnJheShzWydwYXJhbTo6JytrXSkgZm9yIGsgaW4gcGFyYW1zfTttPXtrOmpucC5hc2FycmF5KHNbJ206Oicra10pIGZvciBrIGluIHBhcmFtc307dj17azpqbnAuYXNhcnJheShzWyd2OjonK2tdKSBmb3IgayBpbiBwYXJhbXN9CiAgICAgICAgICAgIGZpcnN0PWludChzWydzdGVwJ10pO3dlaWdodD1mbG9hdChzWydkYXRhX3dlaWdodCddKTtybmcuYml0X2dlbmVyYXRvci5zdGF0ZT1qc29uLmxvYWRzKHN0cihzWydybmdfanNvbiddKSkKICAgIGZucz1tYWtlX2Z1bmN0aW9ucyhhcmdzLmxhdGVudCxhcmdzLmRlcHRoLGFyZ3MucHJlc3N1cmVfbW9kZSkKICAgIGxvZz1vcGVuKG91dHB1dC8nbGVhcm5pbmdfY3VydmUuanNvbmwnLCdhJyxidWZmZXJpbmc9MSk7c3RhcnQ9dGltZS50aW1lKCk7Y2hlY2tzPTAKICAgIG1vbml0b3JfeHk9bnAuYnJvYWRjYXN0X3RvKG1vbml0b3JbJ3h5J10sKGxlbihtb25pdG9yWydwcm9maWxlcyddKSxsZW4obW9uaXRvclsneHknXSksMikpLmNvcHkoKQogICAgZGVmIHZhbGlkYXRpb24oKToKICAgICAgICBwcmVkPW5wLmFzYXJyYXkoZm5zWydmaWVsZHMnXShwYXJhbXMsbW9uaXRvclsncHJvZmlsZXMnXSxtb25pdG9yWydyZXlub2xkcyddLG1vbml0b3JfeHkpKQogICAgICAgIHJlY29yZHM9W10KICAgICAgICBmb3IgaSBpbiByYW5nZShsZW4ocHJlZCkpOgogICAgICAgICAgICB0cnV0aD1tb25pdG9yWydmaWVsZHMnXVtpXTtkPXByZWRbaV0tdHJ1dGgKICAgICAgICAgICAgcmVjb3Jkcy5hcHBlbmQoeydjYXNlJzppLCdSZSc6ZmxvYXQobW9uaXRvclsncmV5bm9sZHMnXVtpXSksJ3VfcmVsX2wyJzpmbG9hdChucC5saW5hbGcubm9ybShkWzosMF0pL25wLmxpbmFsZy5ub3JtKHRydXRoWzosMF0pKSwKICAgICAgICAgICAgICAgICd2X3JlbF9sMic6ZmxvYXQobnAubGluYWxnLm5vcm0oZFs6LDFdKS9tYXgobnAubGluYWxnLm5vcm0odHJ1dGhbOiwxXSksMWUtMTIpKSwndl9ybXNlJzpmbG9hdChucC5zcXJ0KG5wLm1lYW4oZFs6LDFdKioyKSkpLAogICAgICAgICAgICAgICAgJ3BfcmVsX2wyJzpmbG9hdChucC5saW5hbGcubm9ybShkWzosMl0pL25wLmxpbmFsZy5ub3JtKHRydXRoWzosMl0pKSwKICAgICAgICAgICAgICAgICd2ZWxvY2l0eV9yZWxfbDInOmZsb2F0KG5wLmxpbmFsZy5ub3JtKGRbOiw6Ml0pL25wLmxpbmFsZy5ub3JtKHRydXRoWzosOjJdKSl9KQogICAgICAgIHJldHVybiB7J3JvbGUnOid2YWxpZGF0aW9uX29ubHlfbmV2ZXJfYmFja3Byb3BhZ2F0ZWQnLCdjYXNlcyc6cmVjb3JkcywndmVsb2NpdHlfcmVsX2wyX21heCc6bWF4KGNbJ3ZlbG9jaXR5X3JlbF9sMiddIGZvciBjIGluIHJlY29yZHMpLAogICAgICAgICAgICAncF9yZWxfbDJfbWF4JzptYXgoY1sncF9yZWxfbDInXSBmb3IgYyBpbiByZWNvcmRzKSwndmVsb2NpdHlfcmVsX2wyX21lYW4nOmZsb2F0KG5wLm1lYW4oW2NbJ3ZlbG9jaXR5X3JlbF9sMiddIGZvciBjIGluIHJlY29yZHNdKSksCiAgICAgICAgICAgICdwX3JlbF9sMl9tZWFuJzpmbG9hdChucC5tZWFuKFtjWydwX3JlbF9sMiddIGZvciBjIGluIHJlY29yZHNdKSl9CiAgICBkZWYgcGh5c2ljYWxfdmFsaWRhdGlvbigpOgogICAgICAgIHBybmc9bnAucmFuZG9tLmRlZmF1bHRfcm5nKDgxMzcyKTt4eT1wcm5nLnVuaWZvcm0oWy4wMDUsLjAwNV0sWzQuOTk1LC45OTVdLChsZW4obW9uaXRvclsncHJvZmlsZXMnXSksMjU2LDIpKS5hc3R5cGUobnAuZmxvYXQzMikKICAgICAgICByZXNpZHVhbD1ucC5hc2FycmF5KGZuc1sncGh5c2ljcyddKHBhcmFtcyxtb25pdG9yWydwcm9maWxlcyddLG1vbml0b3JbJ3JleW5vbGRzJ10seHkpKQogICAgICAgIHk9bnAuYnJvYWRjYXN0X3RvKG5wLmxpbnNwYWNlKC4wMDEsLjk5OSw2NSxkdHlwZT1ucC5mbG9hdDMyKSwobGVuKG1vbml0b3JbJ3Byb2ZpbGVzJ10pLDY1KSkuY29weSgpCiAgICAgICAgb3V0PW5wLmFzYXJyYXkoZm5zWydvdXRsZXQnXShwYXJhbXMsbW9uaXRvclsncHJvZmlsZXMnXSxtb25pdG9yWydyZXlub2xkcyddLHkpKQogICAgICAgIG1vbT1ucC5zcXJ0KG5wLm1lYW4obnAuc3VtKHJlc2lkdWFsWy4uLiw6Ml0qKjIsLTEpLC0xKSk7bWFzcz1ucC5zcXJ0KG5wLm1lYW4ocmVzaWR1YWxbLi4uLDJdKioyLC0xKSk7YmM9bnAuc3FydChucC5tZWFuKG5wLnN1bShvdXQqb3V0LC0xKSwtMSkpCiAgICAgICAgcmV0dXJuIHsnbW9tZW50dW1fcm1zX21heCc6ZmxvYXQobW9tLm1heCgpKSwnZGl2ZXJnZW5jZV9ybXNfbWF4JzpmbG9hdChtYXNzLm1heCgpKSwnb3V0bGV0X3Jtc19tYXgnOmZsb2F0KGJjLm1heCgpKSwKICAgICAgICAgICAgJ2Nhc2VzJzpbeydjYXNlJzppLCdtb21lbnR1bV9ybXMnOmZsb2F0KG1vbVtpXSksJ2RpdmVyZ2VuY2Vfcm1zJzpmbG9hdChtYXNzW2ldKSwnb3V0bGV0X3Jtcyc6ZmxvYXQoYmNbaV0pfSBmb3IgaSBpbiByYW5nZShsZW4obW9tKSldfQogICAgZGVmIGVtaXQocmVjb3JkKToKICAgICAgICBwcmludChqc29uLmR1bXBzKHJlY29yZCksZmx1c2g9VHJ1ZSk7bG9nLndyaXRlKGpzb24uZHVtcHMocmVjb3JkKSsnXG4nKQogICAgZW1pdCh7J3N0ZXAnOmZpcnN0LCdwaGFzZSc6J3N0YXJ0Jywnc2Vjb25kcyc6dGltZS50aW1lKCktc3RhcnQsJ2JhY2tlbmQnOidqYXgnLCdkZXZpY2VzJzpbc3RyKHgpIGZvciB4IGluIGRldmljZXNdLCd2YWxpZGF0aW9uJzp2YWxpZGF0aW9uKCl9KQogICAgZGVmIHNhdmUoc3RlcCxyZWNvcmQpOgogICAgICAgIGFycmF5cz17fQogICAgICAgIGZvciBwcmVmaXgsdHJlZSBpbiBbKCdwYXJhbScscGFyYW1zKSwoJ20nLG0pLCgndicsdildOgogICAgICAgICAgICBhcnJheXMudXBkYXRlKHtwcmVmaXgrJzo6JytrOm5wLmFzYXJyYXkodmFsdWUpIGZvciBrLHZhbHVlIGluIHRyZWUuaXRlbXMoKX0pCiAgICAgICAgYXJyYXlzLnVwZGF0ZShzdGVwPW5wLmFycmF5KHN0ZXApLGRhdGFfd2VpZ2h0PW5wLmFycmF5KHdlaWdodCkscm5nX2pzb249bnAuYXJyYXkoanNvbi5kdW1wcyhybmcuYml0X2dlbmVyYXRvci5zdGF0ZSkpLGNvbmZpZ19qc29uPW5wLmFycmF5KGpzb24uZHVtcHMoY29uZmlnKSkpCiAgICAgICAgbnAuc2F2ZXpfY29tcHJlc3NlZChvdXRwdXQvJ2pheF9jaGVja3BvaW50LnRtcC5ucHonLCoqYXJyYXlzKQogICAgICAgIChvdXRwdXQvJ2pheF9jaGVja3BvaW50LnRtcC5ucHonKS5yZXBsYWNlKG91dHB1dC8namF4X2NoZWNrcG9pbnQubnB6JykKICAgICAgICAob3V0cHV0LydsYXRlc3RfdmFsaWRhdGlvbi5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlY29yZCxpbmRlbnQ9MikpCiAgICBncmFkaWVudF9iYWxhbmNlPXt9O2xhc3Q9Tm9uZTtjb25kaXRpb25fcHJvYmFiaWxpdHk9bnAuZnVsbChsZW4odHJhaW5bJ3Byb2ZpbGVzJ10pLDEvbGVuKHRyYWluWydwcm9maWxlcyddKSkKICAgIHByb2JlX3JuZz1ucC5yYW5kb20uZGVmYXVsdF9ybmcoNDQxMTkzKQogICAgY29uZGl0aW9uX3Byb2JlPXByb2JlX3JuZy51bmlmb3JtKHNpemU9KGxlbih0cmFpblsncHJvZmlsZXMnXSksMjU2LDIpKS5hc3R5cGUobnAuZmxvYXQzMikKICAgIGNvbmRpdGlvbl9wcm9iZVsuLi4sMF09NSpjb25kaXRpb25fcHJvYmVbLi4uLDBdKioyCiAgICBmb3Igc3RlcCBpbiByYW5nZShmaXJzdCsxLGFyZ3Muc3RlcHMrMSk6CiAgICAgICAgaWYgYXJncy5hZGFwdGl2ZV9waHlzaWNzIGFuZCAoc3RlcD09Zmlyc3QrMSBvciBzdGVwJTUwMD09MCk6CiAgICAgICAgICAgIHNjb3Jlcz1bXQogICAgICAgICAgICBmb3IgbGVmdCBpbiByYW5nZSgwLGxlbih0cmFpblsncHJvZmlsZXMnXSksMTYpOgogICAgICAgICAgICAgICAgcmlnaHQ9bWluKGxlZnQrMTYsbGVuKHRyYWluWydwcm9maWxlcyddKSkKICAgICAgICAgICAgICAgIHJyPW5wLmFzYXJyYXkoZm5zWydwaHlzaWNzJ10ocGFyYW1zLHRyYWluWydwcm9maWxlcyddW2xlZnQ6cmlnaHRdLHRyYWluWydyZXlub2xkcyddW2xlZnQ6cmlnaHRdLGNvbmRpdGlvbl9wcm9iZVtsZWZ0OnJpZ2h0XSkpCiAgICAgICAgICAgICAgICBzY29yZXMuZXh0ZW5kKG5wLnNxcnQobnAubWVhbihucC5zdW0ocnJbLi4uLDoyXSoqMiwtMSksLTEpKzFlLTEyKSkKICAgICAgICAgICAgc2NvcmVzPW5wLmFycmF5KHNjb3JlcyxkdHlwZT1ucC5mbG9hdDY0KQogICAgICAgICAgICBjb25kaXRpb25fcHJvYmFiaWxpdHk9LjMvbGVuKHNjb3JlcykrLjcqc2NvcmVzL3Njb3Jlcy5zdW0oKQogICAgICAgIGNpPXJuZy5pbnRlZ2VycyhsZW4odHJhaW5bJ3Byb2ZpbGVzJ10pLHNpemU9YXJncy5iYXRjaCk7cGk9cm5nLmludGVnZXJzKGxlbih0cmFpblsneHknXSksc2l6ZT0xOTIpCiAgICAgICAgZnJlc2gscmUsXz1zYW1wbGUocm5nLGFyZ3MuYmF0Y2gvLzIpCiAgICAgICAgbj1hcmdzLmJhdGNoLWFyZ3MuYmF0Y2gvLzIKICAgICAgICBwaHlzaWNzX2NpPXJuZy5jaG9pY2UobGVuKHRyYWluWydwcm9maWxlcyddKSxzaXplPW4scD1jb25kaXRpb25fcHJvYmFiaWxpdHkpIGlmIGFyZ3MuYWRhcHRpdmVfcGh5c2ljcyBlbHNlIGNpWzpuXQogICAgICAgIHBwPW5wLmNvbmNhdGVuYXRlKFt0cmFpblsncHJvZmlsZXMnXVtwaHlzaWNzX2NpXSxmcmVzaF0pLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIHJyPW5wLmNvbmNhdGVuYXRlKFt0cmFpblsncmV5bm9sZHMnXVtwaHlzaWNzX2NpXSxyZV0pLmFzdHlwZShucC5mbG9hdDMyKQogICAgICAgIGNvdW50PTI1NiBpZiBhcmdzLmFkYXB0aXZlX3BoeXNpY3MgZWxzZSBhcmdzLnBvaW50cwogICAgICAgIHh5PXJuZy51bmlmb3JtKHNpemU9KGFyZ3MuYmF0Y2gsY291bnQsMikpLmFzdHlwZShucC5mbG9hdDMyKTtjaG9vc2U9cm5nLnVuaWZvcm0oc2l6ZT14eS5zaGFwZVs6LTFdKTwuNQogICAgICAgIHh5Wy4uLiwwXT01Km5wLndoZXJlKGNob29zZSx4eVsuLi4sMF0qKjIseHlbLi4uLDBdKQogICAgICAgIGlmIGFyZ3MuYWRhcHRpdmVfcGh5c2ljczoKICAgICAgICAgICAgY2FuZGlkYXRlX3Jlc2lkdWFsPW5wLmFzYXJyYXkoZm5zWydwaHlzaWNzJ10ocGFyYW1zLHBwLHJyLHh5KSkKICAgICAgICAgICAgcmlzaz1ucC5zcXJ0KG5wLnN1bShjYW5kaWRhdGVfcmVzaWR1YWxbLi4uLDoyXSoqMiwtMSkrMWUtMTIpCiAgICAgICAgICAgIHByb2JhYmlsaXRpZXM9LjUvY291bnQrLjUqcmlzay9yaXNrLnN1bSgtMSxrZWVwZGltcz1UcnVlKQogICAgICAgICAgICBwcm9iYWJpbGl0aWVzPXByb2JhYmlsaXRpZXMuYXN0eXBlKG5wLmZsb2F0NjQpO3Byb2JhYmlsaXRpZXMvPXByb2JhYmlsaXRpZXMuc3VtKC0xLGtlZXBkaW1zPVRydWUpCiAgICAgICAgICAgIHNlbGVjdD1ucC5zdGFjayhbcm5nLmNob2ljZShjb3VudCxzaXplPWFyZ3MucG9pbnRzLHA9cHJvYikgZm9yIHByb2IgaW4gcHJvYmFiaWxpdGllc10pCiAgICAgICAgICAgIHh5PXh5W25wLmFyYW5nZShhcmdzLmJhdGNoKVs6LE5vbmVdLHNlbGVjdF0KICAgICAgICBiYXRjaD17J3Byb2ZpbGVzJzpwcCwncmV5bm9sZHMnOnJyLCd4eSc6eHksJ291dF95JzpybmcudW5pZm9ybShzaXplPShhcmdzLmJhdGNoLDMyKSkuYXN0eXBlKG5wLmZsb2F0MzIpLAogICAgICAgICAgICAnZGF0YV9wcm9maWxlcyc6dHJhaW5bJ3Byb2ZpbGVzJ11bY2ldLCdkYXRhX3JleW5vbGRzJzp0cmFpblsncmV5bm9sZHMnXVtjaV0sCiAgICAgICAgICAgICdkYXRhX3h5JzpucC5icm9hZGNhc3RfdG8odHJhaW5bJ3h5J11bcGldLChhcmdzLmJhdGNoLGxlbihwaSksMikpLmNvcHkoKSwndGFyZ2V0Jzp0cmFpblsnZmllbGRzJ11bY2lbOixOb25lXSxwaVtOb25lLDpdXSwnZGF0YV9zY2FsZSc6dHJhaW5bJ2RhdGFfc2NhbGUnXVtjaV19CiAgICAgICAgaWYgYXJncy5yZXN0YXJ0X2xyIGlzIG5vdCBOb25lOgogICAgICAgICAgICBiYXRjaFsnbGVhcm5pbmdfcmF0ZSddPW5wLmFycmF5KG1heCgxZS01LGFyZ3MucmVzdGFydF9sciouOSoqKChzdGVwLWZpcnN0KS8xMDAwKSksZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBpZiBzdGVwPT1maXJzdCsxIG9yIHN0ZXAlMjUwPT0wOgogICAgICAgICAgICBncCxnZD1mbnNbJ2JhbGFuY2UnXShwYXJhbXMsYmF0Y2gpO2dwLGdkPWZsb2F0KGdwKSxmbG9hdChnZCkKICAgICAgICAgICAgd2VpZ2h0PS44KndlaWdodCsuMipucC5jbGlwKGdwLyhnZCsxZS0xMiksMWUtMywxMCkKICAgICAgICAgICAgZ3JhZGllbnRfYmFsYW5jZT17J3BoeXNpY2FsX2dyYWRpZW50X2wyJzpncCwncmF3X2RhdGFfZ3JhZGllbnRfbDInOmdkLCd3ZWlnaHRlZF9kYXRhX2dyYWRpZW50X2wyJzp3ZWlnaHQqZ2R9CiAgICAgICAgcGFyYW1zLG0sdix2YWx1ZSxhdXgsZ249Zm5zWydzdGVwJ10ocGFyYW1zLG0sdixqbnAuYXJyYXkoc3RlcCksam5wLmFycmF5KHdlaWdodCxkdHlwZT1qbnAuZmxvYXQzMiksYmF0Y2gpCiAgICAgICAgaWYgc3RlcCVhcmdzLmxvZ19ldmVyeT09MCBvciBzdGVwPT1hcmdzLnN0ZXBzOgogICAgICAgICAgICB2YWx1ZT1mbG9hdCh2YWx1ZSk7YXV4PW5wLmFzYXJyYXkoYXV4KQogICAgICAgICAgICBpZiBub3QgbnAuaXNmaW5pdGUodmFsdWUpOnJhaXNlIFJ1bnRpbWVFcnJvcignTm9uZmluaXRlIGxvc3MnKQogICAgICAgICAgICByZWNvcmQ9eydzdGVwJzpzdGVwLCdzZWNvbmRzJzp0aW1lLnRpbWUoKS1zdGFydCwnbG9zcyc6dmFsdWUsJ21vbWVudHVtX21zZSc6ZmxvYXQoYXV4WzBdKSwnb3V0bGV0X21zZSc6ZmxvYXQoYXV4WzFdKSwKICAgICAgICAgICAgICAgICdkYXRhX21zZSc6ZmxvYXQoYXV4WzJdKSwnZGl2ZXJnZW5jZV9ybXMnOmZsb2F0KGF1eFszXSksJ2RhdGFfd2VpZ2h0JzpmbG9hdCh3ZWlnaHQpLCdncmFkaWVudF9ub3JtX2JlZm9yZV9jbGlwJzpmbG9hdChnbiksCiAgICAgICAgICAgICAgICAnZ3JhZGllbnRfYmFsYW5jZSc6Z3JhZGllbnRfYmFsYW5jZSwnYWRhcHRpdmVfcGh5c2ljcyc6YXJncy5hZGFwdGl2ZV9waHlzaWNzLCd2YWxpZGF0aW9uJzp2YWxpZGF0aW9uKCl9CiAgICAgICAgICAgIHN0b3A9RmFsc2UKICAgICAgICAgICAgaWYgc3RlcCUxMDAwPT0wIG9yIHN0ZXA9PWFyZ3Muc3RlcHM6CiAgICAgICAgICAgICAgICBwaHk9cGh5c2ljYWxfdmFsaWRhdGlvbigpO3JlY29yZFsndmFsaWRhdGlvbl9waHlzaWNzJ109cGh5CiAgICAgICAgICAgICAgICBwYXNzZWQ9KHJlY29yZFsndmFsaWRhdGlvbiddWyd2ZWxvY2l0eV9yZWxfbDJfbWF4J108LjA1IGFuZCByZWNvcmRbJ3ZhbGlkYXRpb24nXVsncF9yZWxfbDJfbWF4J108LjEgYW5kIHBoeVsnbW9tZW50dW1fcm1zX21heCddPC4wNSBhbmQgcGh5WydvdXRsZXRfcm1zX21heCddPC4xKQogICAgICAgICAgICAgICAgY2hlY2tzPWNoZWNrcysxIGlmIHBhc3NlZCBlbHNlIDA7c3RvcD1zdGVwPj00MDAwIGFuZCBjaGVja3M+PTIKICAgICAgICAgICAgICAgIHJlY29yZFsnZWFybHlfc3RvcCddPXN0b3AKICAgICAgICAgICAgZW1pdChyZWNvcmQpO3NhdmUoc3RlcCxyZWNvcmQpO2xhc3Q9cmVjb3JkCiAgICAgICAgICAgIGlmIHN0b3A6YnJlYWsKICAgIGxvZy5jbG9zZSgpCiAgICAjIEV4cG9ydCB0byB0aGUgbnVtZXJpY2FsbHkgcGFyaXR5LXRlc3RlZCBQeVRvcmNoIGluZmVyZW5jZSBjbGFzczsgb3B0aW1pemVyCiAgICAjIHJlbWFpbnMgaW4gdGhlIEpBWCBjaGVja3BvaW50LCBub3QgaW4gdGhpcyBwb3J0YWJsZSBpbmZlcmVuY2Utb25seSBmaWxlLgogICAgaW1wb3J0IHRvcmNoCiAgICBmcm9tIGNoYW5uZWxfb3BlcmF0b3IudjJfbW9kZWwgaW1wb3J0IFN0cmVhbURlZXBPTmV0CiAgICBtb2RlbD1TdHJlYW1EZWVwT05ldCgqKmNvbmZpZyk7c3RhdGU9bW9kZWwuc3RhdGVfZGljdCgpCiAgICBzdGF0ZS51cGRhdGUoe2s6dG9yY2guZnJvbV9udW1weShucC5hcnJheSh2YWx1ZSkpIGZvciBrLHZhbHVlIGluIHBhcmFtcy5pdGVtcygpfSkKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZSkKICAgIG1vZGU9J3JlbGF0aXZlX3JtcycgaWYgYXJncy5yZWxhdGl2ZV9kYXRhIGVsc2UgJ3NjYWxlZF9tc2UnCiAgICB0b3JjaC5zYXZlKHsnc3RhdGVfZGljdCc6bW9kZWwuc3RhdGVfZGljdCgpLCdjb25maWcnOmNvbmZpZywnc3RlcCc6c3RlcCwnbW9kZWxfdHlwZSc6J3N0cmVhbV9kZWVwb25ldF92Ml9qYXgnLCdkYXRhX3dlaWdodCc6ZmxvYXQod2VpZ2h0KSwndHJhaW5pbmdfY2FzZXMnOmxlbih0cmFpblsncHJvZmlsZXMnXSksJ2RhdGFfbG9zc19tb2RlJzptb2RlfSxvdXRwdXQvJ3YyX21vZGVsLnB0JykKICAgIHN1bW1hcnk9eydiYWNrZW5kJzonamF4JywnamF4X3ZlcnNpb24nOmpheC5fX3ZlcnNpb25fXywnZGV2aWNlcyc6W3N0cih4KSBmb3IgeCBpbiBkZXZpY2VzXSwnc3RlcHMnOnN0ZXAsJ3BsYW5uZWRfc3RlcHMnOmFyZ3Muc3RlcHMsJ3NlY29uZHMnOnRpbWUudGltZSgpLXN0YXJ0LCdjb25maWcnOmNvbmZpZywndHJhaW5pbmdfY2FzZXMnOmxlbih0cmFpblsncHJvZmlsZXMnXSksJ2RhdGFfbG9zc19tb2RlJzptb2RlLCdhZGFwdGl2ZV9waHlzaWNzJzphcmdzLmFkYXB0aXZlX3BoeXNpY3MsJ3Jlc3RhcnRfbHInOmFyZ3MucmVzdGFydF9sciwnZmluYWwnOmxhc3R9CiAgICAob3V0cHV0Lyd2Ml90cmFpbmluZ19zdW1tYXJ5Lmpzb24nKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3VtbWFyeSxpbmRlbnQ9MikpCgoKaWYgX19uYW1lX189PSdfX21haW5fXyc6CiAgICBwPWFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCk7cC5hZGRfYXJndW1lbnQoJy0tZGF0YScscmVxdWlyZWQ9VHJ1ZSk7cC5hZGRfYXJndW1lbnQoJy0tb3V0cHV0JyxyZXF1aXJlZD1UcnVlKQogICAgcC5hZGRfYXJndW1lbnQoJy0tc3RlcHMnLHR5cGU9aW50LGRlZmF1bHQ9NTAwMDApO3AuYWRkX2FyZ3VtZW50KCctLWxvZy1ldmVyeScsdHlwZT1pbnQsZGVmYXVsdD0yNTApCiAgICBwLmFkZF9hcmd1bWVudCgnLS1iYXRjaCcsdHlwZT1pbnQsZGVmYXVsdD0xNik7cC5hZGRfYXJndW1lbnQoJy0tcG9pbnRzJyx0eXBlPWludCxkZWZhdWx0PTk2KQogICAgcC5hZGRfYXJndW1lbnQoJy0td2lkdGgnLHR5cGU9aW50LGRlZmF1bHQ9NjQpO3AuYWRkX2FyZ3VtZW50KCctLWxhdGVudCcsdHlwZT1pbnQsZGVmYXVsdD0zMik7cC5hZGRfYXJndW1lbnQoJy0tZGVwdGgnLHR5cGU9aW50LGRlZmF1bHQ9MykKICAgIHAuYWRkX2FyZ3VtZW50KCctLXJlcXVpcmUtZ3B1JyxhY3Rpb249J3N0b3JlX3RydWUnKTtwLmFkZF9hcmd1bWVudCgnLS1yZXN1bWUnKQogICAgcC5hZGRfYXJndW1lbnQoJy0tcHJlc3N1cmUtbW9kZScsY2hvaWNlcz1bJ3Zpc2NvdXMnLCdjb252ZWN0aXZlJ10sZGVmYXVsdD0ndmlzY291cycpCiAgICBwLmFkZF9hcmd1bWVudCgnLS1yZWxhdGl2ZS1kYXRhJyxhY3Rpb249J3N0b3JlX3RydWUnKQogICAgcC5hZGRfYXJndW1lbnQoJy0tYWRhcHRpdmUtcGh5c2ljcycsYWN0aW9uPSdzdG9yZV90cnVlJyk7cC5hZGRfYXJndW1lbnQoJy0tcmVzdGFydC1scicsdHlwZT1mbG9hdCkKICAgIGFyZ3M9cC5wYXJzZV9hcmdzKCkKICAgIGlmIGFyZ3MuYmF0Y2g8MiBvciBtaW4oYXJncy5zdGVwcyxhcmdzLmxvZ19ldmVyeSxhcmdzLnBvaW50cyk8MTpwLmVycm9yKCdJbnZhbGlkIGNvdW50cycpCiAgICB0cmFpbihhcmdzKQo=', 'models/v2/stream_operator.pt': '...', 'models/v2/jax_checkpoint.npz': '...', 'models/v2/validation_summary.json': 'ewogICJzdGVwIjogNzAwMDAsCiAgImZpZWxkX2FjY3VyYWN5X3Bhc3NlZCI6IHRydWUsCiAgImFsbF9nYXRlc19wYXNzZWQiOiBmYWxzZSwKICAibW9uaXRvcl9jYXNlcyI6IDE2LAogICJibGluZF9jYXNlcyI6IDgsCiAgImZhaWxlZF9nYXRlcyI6IFsKICAgICJtb21lbnR1bSIKICBdLAogICJzY29wZSI6ICJmb3VyIGNvc2luZSBpbmxldCBtb2RlcywgcG9zaXRpdmUgbm8tc2xpcCBpbmxldHMsIEwxPD0wLjgsIFJlPVsxMCwxMDBdLCBjaGFubmVsIDV4MSIKfQ==', 'models/v2/metadata.json': 'ewogICJzdGVwIjogNzAwMDAsCiAgImJhY2tlbmQiOiAiSkFYIDAuMTEuMSBHUFU7IHBhcml0eS10ZXN0ZWQgcG9ydGFibGUgUHlUb3JjaCBpbmZlcmVuY2UgZXhwb3J0IiwKICAicHJlc3N1cmVfbW9kZSI6ICJjb252ZWN0aXZlIiwKICAidHJhaW5pbmdfY2FzZXMiOiAyODgsCiAgInRyYWluaW5nX2FuY2hvcl9wb2ludHNfcGVyX2Nhc2UiOiAyNTYsCiAgInBoeXNpY3NfcG9pbnRzX3NhbXBsZWRfb25saW5lIjogdHJ1ZSwKICAibW9kZSI6ICJoeWJyaWQgRkVNICsgcGh5c2ljcy1pbmZvcm1lZCBEZWVwT05ldCIsCiAgInByaW1hcnlfdXBkYXRlcyI6IDUwMDAwLAogICJhZGFwdGl2ZV9jb250aW51YXRpb25fdXBkYXRlcyI6IDIwMDAwLAogICJvcHRpbWl6ZXJfaGVsZG91dF9wcm9maWxlc19hdWRpdGVkIjogMjQsCiAgImNoZWNrcG9pbnRfc2hhMjU2IjogIjdkYjZiMDczYTgyOTUyOWI3NDA0ZmRmN2FjOWM3MjA5YzcyOGFmMzJmMGU0ZDg4YmU1NGUxMmExNmJhZTI4MmEiLAogICJzb3VyY2VzIjogewogICAgImNoYW5uZWxfb3BlcmF0b3IvdjJfdHJhaW4ucHkiOiAiYTkyN2IyMjZiYzkyZGJlYWJlMjE3NjAyNTA2ZDA5NjU5NzNjYjFiZDlkNTNjMGEzYWYyOTc2MjQ4ZTZlMmEzOCIsCiAgICAiY2hhbm5lbF9vcGVyYXRvci92Ml9kYXRhLnB5IjogIjg0NGYxYTYxMjkzYjMxNzgxY2VmMDU1YmMxMGEyYWFlMGU5MDBhMWE5NWUzNGZjMTY3OTQxZGQxMDFiYzk1ZDYiLAogICAgImNoYW5uZWxfb3BlcmF0b3IvdjJfcHJvZmlsZXMucHkiOiAiZWE1MmJjMjZkZTdjM2MzOTZjOTk4OGEzNmE3OTUyZjRlYmE3YTM5MGU1OTcyNmMzMTViMThmNjA3OGMzOTFhNSIsCiAgICAiY2hhbm5lbF9vcGVyYXRvci92Ml9qYXgucHkiOiAiYTNjOWQ0ZTAzMzg4ZjQxMDA4MmVlNzI4OTcyNDNmZDlmYmIyMGNiMjA5YjI2NmE4Y2FkOWIxYzI1ZmNjZThkZSIsCiAgICAiY2hhbm5lbF9vcGVyYXRvci92Ml9tb2RlbC5weSI6ICI4OWE0YzM2MjNlYmMxZGUwMDYzMWRiMzZkMGNkNDVkMWY0MmM5MjAyZjUxNGJhZWU2ZWZiMmFkNTMwZWRmZDc2IiwKICAgICJjaGFubmVsX29wZXJhdG9yL3YyX2V2YWx1YXRlLnB5IjogImE5NGE5ZTE3NDZmOTVkMjA3ZWY1NmIwOTg0Y2UxMTJiMGUyMDQ1MzUyOTJmNzkyZDQ0NmJhZGI4NTMyMDc5OTMiLAogICAgImNoYW5uZWxfb3BlcmF0b3IvdjJfcHJlZGljdC5weSI6ICI5MTYwMTRmM2ZkZDUzYjRlNjJmMDcyMmFkZDI1ZmM5MDFiYzQ2NzE4MzFhMDkwNWMwOTI0ZDRhM2JlNTgwMmE4IgogIH0sCiAgInZhbGlkYXRpb24iOiB7CiAgICAic3RlcCI6IDcwMDAwLAogICAgImZpZWxkX2FjY3VyYWN5X3Bhc3NlZCI6IHRydWUsCiAgICAiYWxsX2dhdGVzX3Bhc3NlZCI6IGZhbHNlLAogICAgIm1vbml0b3JfY2FzZXMiOiAxNiwKICAgICJibGluZF9jYXNlcyI6IDgsCiAgICAiZmFpbGVkX2dhdGVzIjogWwogICAgICAibW9tZW50dW0iCiAgICBdLAogICAgInNjb3BlIjogImZvdXIgY29zaW5lIGlubGV0IG1vZGVzLCBwb3NpdGl2ZSBuby1zbGlwIGlubGV0cywgTDE8PTAuOCwgUmU9WzEwLDEwMF0sIGNoYW5uZWwgNXgxIgogIH0KfQ==', 'data/v2/train.npz': '...', 'data/v2/monitor.npz': '...', 'data/v2/blind.npz': '...', 'reports/v2/learning_curves.png': '...', 'reports/v2/iteration_errors.csv': '...'}
for name, payload in assets.items():
    p=Path(name)
    p.parent.mkdir(parents=True,exist_ok=True)
    p.write_bytes(base64.b64decode(payload))
print("Restored trained weights, optimizer checkpoint, source and CFD data. No API key required.")


In [ ]:
subprocess.run([sys.executable,"-m","channel_operator.v2_predict","--checkpoint","models/v2/stream_operator.pt","--re","55","--coefficients","0.25","-0.1","0.05","0","--output","prediction.npz"],check=True)


In [ ]:
for name,role in [("monitor","repeated_monitor_validation"),("blind","untouched_final_blind_test")]:
    subprocess.run([sys.executable,"scripts/evaluate_v2.py","--checkpoint","models/v2/stream_operator.pt","--data",f"data/v2/{name}.npz","--training-data","data/v2/train.npz","--output",f"results/{name}","--role",role],check=True)


In [ ]:
import json
from IPython.display import display, Image
for name in ["monitor","blind"]:
    report=json.loads(Path(f"results/{name}/metrics.json").read_text())
    print(name, "velocity max:", max(c["velocity_relative_l2"] for c in report["cases"]), "pressure max:", max(c["pressure_relative_l2"] for c in report["cases"]), "gates:", report["gates"])
    display(Image(filename=f"results/{name}/fields.png"))
display(Image(filename="reports/v2/learning_curves.png"))


## Optional new training
Enable a GPU in Kaggle Settings before setting RUN_TRAINING=True. This cell stays disabled by default. The imported fixed model can be evaluated on CPU.


In [ ]:
RUN_TRAINING = False  # Set True only when you want a new GPU training run.
if RUN_TRAINING:
    subprocess.run([sys.executable,"run_v2_jax.py","--data","data/v2","--output","retrained","--steps","50000","--pressure-mode","convective","--relative-data","--require-gpu"],check=True)
